# PlantVision 2.5 — houseplant disease pipeline

End-to-end deterministic training notebook that produces every artefact
`server/` reads at runtime:

```
models/
├── classifier.pth              ← frozen-DINOv2 ViT-S/14 + MLP head (full state_dict)
├── classifier.onnx             ← FP32 export (backbone+head, single graph)
├── classifier_int8.onnx        ← dynamic INT8 (production)   [absent if INT8 drift > 0.05]
├── classifier_fp32.onnx        ← FP32 ONNX (only present if INT8 drift gate failed)
├── classes.json                ← {architecture, class_names, training.img_size, inference.tta, ood, ...}
├── temperature.json            ← {temperature: T, tta}
├── conformal.json              ← RAPS calibration {q_hat, alpha, ...}
├── detector.{pt,onnx}          ← Stage 1 (re-trained in §3 from PlantDoc bboxes by default)
├── training_metrics.json
└── model_card.json
```

## Class set — 9 trainable classes (v2.x)

```
gray_mold_botrytis, healthy, leaf_scorch_underwatering,
leaf_spot, not_a_plant, nutrient_deficiency,
pests, powdery_mildew, viral_disease
```

`pests` is a single combined head for mealybugs + scale (the two most
visually-separable houseplant pests). It is a first-class trainable class
— §2 fetches its iNat supply unconditionally.

`leaf_scorch_underwatering` and `nutrient_deficiency` are sourced from
free-text iNat queries (`leaf scorch`, `chlorosis`, …). They land at a
60-row per-class floor (`THRESH_TRAIN_PER_CLASS`); logit adjustment +
label smoothing absorbs the noisier free-text labels. Flip
`INCLUDE_PHYSIOLOGICAL_CLASSES = False` in §1 to roll back to the
7-class head if those sources go dark.

The two remaining v2.0 design classes — `edema`, `sunburn_or_scorch` —
stay deferred. Their bilingual content cards live in
`server/diseases_data.py` already, so flipping them on is a §1 +
data-fetch change, not a content port.

## Why these decisions

* **Frozen DINOv2 ViT-S/14 + trained head — not distillation.** The §3
  frozen DINOv2-S linear probe already reaches ~0.91 top-1 on
  `val_realworld`, proving the features separate the 8 classes. With no
  model-size constraint and ~10× single-core CPU latency headroom, the
  highest-ceiling, lowest-risk move is to deploy the frozen ViT-S/14
  (21M params, 384-d CLS embedding) plus a trained MLP head, rather than
  distilling into a weaker, INT8-hostile MobileNetV3 student (vanilla KD
  degrades exactly when the teacher is much stronger — DIST/DKD
  territory). The EfficientNetV2-S teacher and the KD student were
  removed (§4/§5).
* **Loss-side balancing only.** Logit Adjustment (Menon ICLR 2021,
  τ=0.5); the `WeightedRandomSampler` is **off** by default to avoid
  double-correction on rare classes. A source-weighted sampler
  down-weights field-crop PlantDoc rows so the empirical training prior
  shifts toward indoor houseplants.
* **Three independent val slices, one ship gate.** `val_realworld` =
  primary disease macro-F1 gate (≥ 0.70 — must not regress below the
  live model, no class with
  recall=0, healthy-bias < 1.5). `val_indomain` drives early stopping.
  `val_studio` (PlantVillage) is a transfer/background-bias monitor —
  never blocking.
* **Two dedicated calibration splits**, both held out from the train
  set: `temperature_cal` (~5 %) for temperature scaling, `conformal_cal`
  (~10 %) for RAPS quantile fitting. Both are restricted to the
  deployment distribution (`plantdoc + inat`), never PlantVillage, so
  conformal exchangeability holds at deploy.
* **Dynamic INT8 ships only if `val_realworld` disease F1 drops by
  < 0.05** vs FP32. Transformers/ViTs quantise best with *dynamic* INT8
  (ORT official guidance) — activation ranges are computed at run time,
  no calibration set, and the weight-heavy MatMuls get quantised for
  ~2–3× CPU speedup at ~0 accuracy loss. On drift > 0.05 the build does
  NOT rename FP32 as INT8 — it writes `classifier_fp32.onnx` and leaves
  `classifier_int8.onnx` absent.
* **Trustworthy comparison.** The int8-vs-fp32 ordering is reported with
  a **paired bootstrap** CI (report Stage 0) in `model_card.json`; treat
  the ordering as real only when `significant=true`.
* **Test-time augmentation (off by default).** `USE_TTA_HFLIP` enables
  2× horizontal-flip TTA. When on, the ship-gate F1, temperature,
  conformal and MaxLogit are all fit through the same flip-averaging and
  `inference.tta='hflip'` is written so the server reproduces it.
* **Fail-loud everywhere** and **ship-gate-then-write.** All canonical
  declarative artefacts in `models/` are written AFTER the ship gate
  passes; a failed run leaves the prior build untouched.

## Model families (Aqua / Gamma / Frontier)

This notebook produces ONE family at a time, selected by `MODEL_FAMILY`
in §1. All three tiers share this exact recipe (data, losses,
calibration, ONNX export, ship gate); only the backbone + input size
change:

| Family | Backbone | Input | Role |
|---|---|---|---|
| `aqua` | MobileNetV3-Small (distilled from ViT-S/14) | 224 | light, on-device (fast, lower ceiling) |
| **`gamma`** (default) | **DINOv2 ViT-S/14** | **224** | **current production tier — medium accuracy/latency** |
| `frontier` | DINOv2 ViT-B/14 (~86M) | 224 | future high-capacity tier (GPU inference) |

v2.5 continues maximising **gamma**: the backbone is unchanged (the
frozen-probe ~0.91 ceiling is validated) and LoRA capacity stays at
v2.4's rank 16 / last 8 blocks (the §3 probe showed the *features*, not
the head, cap weak-class recall). v2.5 adds a BROADLEAF nutrient source
(melon) so the precision-/supply-bound `nutrient_deficiency` class trains
on dicot leaf morphology, and ships **per-class precision CIs** beside the
recall CIs because the weak physiological classes are precision-bound. `model_family` is written into
`classes.json` + `model_card.json` and surfaced by the server's
`/api/version`.

Total training time on a single A100-40GB is ≈ 20–35 min including data
fetch (the frozen backbone trains only the head + LoRA adapters).


## §1 — Setup: deps, schema, taxonomy, eval helpers

This cell is idempotent — re-running it does not redo anything besides
imports and folder creation. It also:

* installs versioned deps,
* sets every seed,
* defines the `CropAnnotation` sidecar schema,
* declares the canonical `CLASSES` list (the alphabetical order is
  what `ImageFolder` will produce at training time and what every
  downstream artefact expects).

**Colab A100 quickstart.** Pick a GPU (A100) runtime, add a Colab secret
`KAGGLE_API_TOKEN` with *Notebook access* ON (so §2 pulls the real in-domain
weak-class supply + the eval-eligible DoctorP slice), then *Run all*.
`MODEL_FAMILY='aqua'` (§1) trains the gamma teacher, distills the aqua student
and exports both bundles in one ~30–40 min run; the §6 ship gate blocks the
write unless `val_realworld` disease macro-F1 ≥ 0.70. If the §5 aqua/TFLite
step errors on your Colab, the §4 teacher is already in memory — just run §6
to ship the gamma server bundle (or set `MODEL_FAMILY='gamma'`).


In [ ]:
# §1 — Setup
import os, sys, subprocess, random, json, hashlib, pathlib, re, shutil, time, zipfile, math
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Optional

# numpy is intentionally unpinned: recent torch / torchvision wheels link
# against numpy 2.x; pinning <2 breaks `from torchvision import …`.
PIP_PKGS = [
    # torch / torchvision are left as low floors on purpose: Colab ships a
    # CUDA-matched build, and a higher floor would force a slow, possibly
    # CUDA-mismatched reinstall. The rest are bumped to current stable
    # floors (Colab installs the latest satisfying them).
    'torch>=2.2', 'torchvision',
    'ultralytics>=8.3', 'onnx>=1.18', 'onnxruntime>=1.22',
    # torch>=2.10 imports onnxscript at torch.onnx.export module load.
    'onnxscript>=0.2',
    'onnxslim>=0.1.82',  # YOLO ONNX export uses it; pre-install avoids a mid-run AutoUpdate
    'imagehash', 'Pillow', 'scikit-learn>=1.6', 'pandas',
    'peft>=0.14',  # LoRA fine-tune of the DINOv2 backbone (§4)
    'tqdm', 'matplotlib',
    'requests>=2.34', 'datasets>=3.2', 'PyYAML',
    'sympy',  # onnxruntime.quantization.shape_inference (quant_pre_process) needs it
    'kagglehub',  # Kaggle dataset_download for the optional §2 houseplant supplement
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PIP_PKGS])

# torchao compatibility shim. Recent peft hard-rejects an OLD torchao via
# is_torchao_available(), which raises ImportError inside get_peft_model()
# (§4) -- Colab preinstalls torchao 0.10.0 while peft wants > 0.16.0. Standard
# LoRA never touches torchao and we quantise via onnxruntime, not torchao, so
# drop the incompatible build and let peft's optional-integration probe return
# False cleanly. Best-effort; checked via find_spec so we don't import (and
# thus cache) torchao before removing it.
import importlib.util as _ilu
if _ilu.find_spec('torchao') is not None:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'],
                   check=False)
    print('Removed incompatible torchao (unused; unblocks peft LoRA in §4).')

import numpy as np, requests, torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix
from tqdm import tqdm

# ── Reproducibility ─────────────────────────────────────────────
SEED = 0
# Single source of truth for the version tag we write into
# ``classes.json`` and ``model_card.json``. Production servers pin
# behaviour to this string via ``classes.json -> model_version``.
MODEL_VERSION = '2.5.0'

# ── Model families (Aqua / Gamma / Frontier) ────────────────────
# Three capacity tiers. Gamma + Frontier share one frozen-DINOv2 + MLP-head
# recipe (only the backbone + input size differ); Aqua is a DISTILLED mobile
# student. Set MODEL_FAMILY and run the notebook end-to-end to produce that
# tier's artefacts. Everything downstream reads MODEL_FAMILY / FAMILY_CONFIG.
#   aqua     — light ON-DEVICE tier: a MobileNetV3-Small student knowledge-
#              distilled (§5) from the gamma DINOv2 teacher trained in §4, then
#              exported to TFLite INT8 for fully offline phone inference.
#   gamma    — CURRENT production tier: frozen DINOv2 ViT-S/14 + MLP head
#              (validated ~0.91 frozen-probe ceiling). The DEFAULT.
#   frontier — high-capacity tier: frozen DINOv2 ViT-B/14 (~86M) + MLP head.
#              GPU inference; server-side inference is gated OFF by default.
# `distill`/`student_arch`/`export` drive §5/§6; gamma+frontier ignore them.
MODEL_FAMILY = 'aqua'    # v2.5: the gamma+aqua COMBO. An 'aqua' run trains the
                        # DINOv2 ViT-S/14 teacher (== gamma), distills the
                        # MobileNetV3-Small student (§5), and exports BOTH the
                        # gamma server bundle (models/) AND the aqua on-device
                        # TFLite (models/aqua/) from ONE A100 run. Use 'gamma'
                        # for the server model alone (skips §5's TF toolchain);
                        # 'frontier' is the ViT-B/14 A/B tier.
MODEL_FAMILIES = {
    # Aqua's teacher backbone matches gamma (ViT-S/14 @224) so the student
    # distills from the strongest small teacher; the student has its own
    # input size (STUDENT_IMG, set in §5).
    'aqua':     {'hub_id': 'dinov2_vits14', 'img_size': 224, 'on_device': True,
                 'distill': True, 'student_arch': 'mobilenetv3_small',
                 'export': 'tflite_int8',
                 'desc': 'light on-device tier (TFLite student, distilled from gamma)'},
    'gamma':    {'hub_id': 'dinov2_vits14', 'img_size': 224, 'on_device': False,
                 'distill': False, 'export': 'onnx_int8',
                 'desc': 'current medium production tier'},
    'frontier': {'hub_id': 'dinov2_vitb14', 'img_size': 224, 'on_device': False,
                 'distill': False, 'export': 'onnx_int8',
                 'desc': 'high-capacity GPU tier (ViT-B/14, ~86M params)'},
}
if MODEL_FAMILY not in MODEL_FAMILIES:
    raise ValueError(f'MODEL_FAMILY={MODEL_FAMILY!r} not in {list(MODEL_FAMILIES)}')
FAMILY_CONFIG = MODEL_FAMILIES[MODEL_FAMILY]
print(f'model family = {MODEL_FAMILY}  ({FAMILY_CONFIG["desc"]})  '
      f'backbone={FAMILY_CONFIG["hub_id"]}  img_size={FAMILY_CONFIG["img_size"]}')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True       # faster training
torch.backends.cudnn.deterministic = False  # honest about the trade-off

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device={device}  torch={torch.__version__}')

# ── Sidecar metadata schema ─────────────────────────────────────
@dataclass
class CropAnnotation:
    plant_id: str
    species: Optional[str]
    diagnosis: str
    severity: str               # low | medium | high | unknown
    leaf_or_whole_plant: str    # leaf | whole_plant
    capture_session: str
    source: str                 # inat | coco | plantvillage | plantdoc
    label_confidence: float     # in [0, 1]
    bbox: List[int]             # [x, y, w, h]; [0,0,0,0] for whole-image

def write_sidecar(p: Path, ann: CropAnnotation):
    p.with_suffix('.json').write_text(json.dumps(asdict(ann), ensure_ascii=False, indent=2))
def read_sidecar(p: Path):
    s = p.with_suffix('.json'); return json.loads(s.read_text()) if s.exists() else None

# ── Folder layout ───────────────────────────────────────────────
ROOT          = pathlib.Path('data_v2')
CLS_ROOT      = ROOT / 'classification'
TRAIN_ROOT    = CLS_ROOT / 'train'
VAL_ROOT      = CLS_ROOT / 'val'
VAL_INDOMAIN  = VAL_ROOT / 'indomain'
VAL_PRIMARY   = VAL_ROOT / 'realworld'              # ship gate
VAL_STUDIO    = VAL_ROOT / 'plantvillage_studio'    # transfer monitor
TEMP_CAL      = VAL_ROOT / 'temperature_cal'        # INT8 + temperature
CONFORMAL_CAL = VAL_ROOT / 'conformal_cal'          # RAPS

INCLUDE_PHYSIOLOGICAL_CLASSES = True
# 'pests' is a SINGLE combined head for the two most visually-separable
# houseplant pests (mealybug cottony masses + scale waxy bumps). One
# well-supplied class trains far more reliably than two thin ones and the
# user-facing care advice is nearly identical, so they share one label. It
# is a FIRST-CLASS trainable class — no opt-in flag; §2 fetches its supply
# unconditionally. Spider mites stay excluded — their signal is a damage
# texture that overlaps early leaf scorch / nutrient stipple.
BASE_CLASSES = [
    'gray_mold_botrytis', 'healthy', 'leaf_spot',
    'not_a_plant', 'pests', 'powdery_mildew', 'viral_disease',
]
PHYSIOLOGICAL_CLASSES = ['leaf_scorch_underwatering', 'nutrient_deficiency']
CLASSES = sorted(
    BASE_CLASSES
    + (PHYSIOLOGICAL_CLASSES if INCLUDE_PHYSIOLOGICAL_CLASSES else [])
)
# `_drop_low_green_content` floor for gray_mold_botrytis — necrotic
# Botrytis foliage is mostly brown, so the gate is set low to keep
# gray-on-brown leaf shots while still rejecting agar plates / fruit.
BOTRYTIS_GREEN_MIN_RATIO = 0.04
SPLIT_ROOTS = (VAL_INDOMAIN, VAL_PRIMARY, VAL_STUDIO, TEMP_CAL, CONFORMAL_CAL)

def ensure_layout():
    for c in CLASSES:
        (TRAIN_ROOT / c).mkdir(parents=True, exist_ok=True)
    for d in SPLIT_ROOTS:
        d.mkdir(parents=True, exist_ok=True)
ensure_layout()

# ── Eval helpers (used by every later phase) ────────────────────
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# Eval/calibration resize ratio — SINGLE SOURCE OF TRUTH. The eval/cal
# transform is Resize(int(img_size * EVAL_RESIZE_RATIO)) -> CenterCrop(img_size)
# (a mild centre zoom). temperature.json + the conformal q_hat are fit on THIS
# distribution and the server reproduces it from classes.json -> training.
# eval_resize_ratio, so every train-side use must read this one constant
# (its value is also written into classes.json in §6). Hardcoding 1.05 in more
# than one place is exactly how train/serve crop parity silently drifts.
EVAL_RESIZE_RATIO = 1.05
# Scale to half the box's vCPUs, capped at 12 — heavy CPU augs
# (JpegRecompress, WhiteBalanceShift, DefocusOrMotion) bottleneck the
# A100 if we cap too low; more workers than vCPU/2 adds context-switch
# overhead without further throughput.
EVAL_WORKERS = min(12, max(2, (os.cpu_count() or 2) // 2))
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def iter_images(root: Path):
    for p in Path(root).rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield p

def count_images(root):
    return sum(1 for _ in iter_images(root)) if Path(root).exists() else 0

def _seed_worker(worker_id):
    s = (SEED + worker_id) % (2**32)
    np.random.seed(s); random.seed(s)

def _loader_kwargs(workers, **extra):
    """DataLoader kwargs. Caller `extra` overrides defaults so e.g.
    `persistent_workers=False` from a one-off cell takes effect — important
    after a cell interrupt leaves stale workers attached to a closed dataset.
    """
    kw = dict(num_workers=workers,
              pin_memory=(device == 'cuda'),
              worker_init_fn=_seed_worker)
    if workers > 0:
        kw['persistent_workers'] = True
        kw['prefetch_factor'] = 2
    kw.update(extra)
    return kw

import inspect as _inspect
_IMAGEFOLDER_ALLOW_EMPTY = (
    'allow_empty' in _inspect.signature(ImageFolder.__init__).parameters
)

def make_eval_loader(root, img_size, batch=128, workers=EVAL_WORKERS, class_to_idx=None):
    """Build a deterministic eval loader whose label space matches training.

    `ImageFolder` indexes classes by sorted directory listing — an eval
    slice missing one trained class would silently shift the label IDs.
    `class_to_idx` is the train-time map; we remap `target` so y_true is
    comparable to model `argmax` on every slice.
    """
    tx = T.Compose([T.Resize(int(img_size * EVAL_RESIZE_RATIO)),
                    T.CenterCrop(img_size), T.ToTensor(), NORM])
    if_kwargs = {'allow_empty': True} if _IMAGEFOLDER_ALLOW_EMPTY else {}
    target_transform = None
    if class_to_idx is not None:
        tmp = ImageFolder(str(root), **if_kwargs)
        unknown = [c for c in tmp.classes if c not in class_to_idx]
        if unknown:
            raise RuntimeError(
                f'Eval folder {root} has classes outside training taxonomy: {unknown}.'
            )
        slice_to_train = {tmp.class_to_idx[c]: class_to_idx[c] for c in tmp.classes}
        target_transform = slice_to_train.__getitem__
    ds = ImageFolder(str(root), transform=tx, target_transform=target_transform,
                     **if_kwargs)
    if len(ds) == 0:
        raise RuntimeError(f'Eval folder is empty: {root}.  Run §2 first.')
    dl = DataLoader(ds, batch_size=batch, shuffle=False, **_loader_kwargs(workers))
    return ds, dl

@torch.inference_mode()
def evaluate(model, dl, class_names, device, name=''):
    """Disease macro-F1 + per-class recall + confusion matrix + healthy-bias."""
    model.eval()
    y_true, y_pred = [], []
    for x, y in dl:
        logits = model(x.to(device, non_blocking=True))
        y_pred.extend(logits.argmax(1).detach().cpu().tolist())
        y_true.extend(y.tolist())
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    disease_idx = [i for i, n in enumerate(class_names) if n not in ('healthy', 'not_a_plant')]
    f1 = float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))
    recall = recall_score(y_true, y_pred, labels=list(range(len(class_names))),
                          average=None, zero_division=0)
    # Per-class PRECISION too: the physiological classes are precision-bound
    # (they over-trigger), so recall alone hides their real failure mode. Used
    # by the §4 precision-aware checkpoint selection and shipped (with CIs) in
    # the model card. zero_division=0 => a never-predicted class scores 0.
    precision = precision_score(y_true, y_pred, labels=list(range(len(class_names))),
                               average=None, zero_division=0)
    h = class_names.index('healthy') if 'healthy' in class_names else None
    bias = float((y_pred == h).mean() / max((y_true == h).mean(), 1e-9)) if h is not None else 0.0
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names)))).tolist()
    print(f'[{name}] disease_macro_f1={f1:.3f}  healthy_bias={bias:.2f}  n={len(y_true)}')
    if 'val_realworld' in name or 'final' in name:
        for c, r in zip(class_names, recall):
            print(f'    recall[{c:25s}] = {r:.3f}')
    return {'name': name, 'disease_macro_f1': f1, 'healthy_bias': bias,
            'per_class_recall': {c: float(r) for c, r in zip(class_names, recall)},
            'per_class_precision': {c: float(p) for c, p in zip(class_names, precision)},
            'confusion_matrix': cm}

# Persisted run log — every epoch appends a row, useful for post-mortem.
RUN_LOG = Path('models/run_log.jsonl')
RUN_LOG.parent.mkdir(parents=True, exist_ok=True)
def log_run(row: dict):
    row = {'ts': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), **row}
    with RUN_LOG.open('a', encoding='utf-8') as f:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

def _kaggle_sync_colab_secrets():
    """Copy Kaggle secrets from Colab userdata into os.environ.

    Colab secrets are NOT auto-exported as environment variables -- they must
    be pulled explicitly with google.colab.userdata -- which is why setting a
    KAGGLE_API_TOKEN secret alone left the downloader seeing no credentials.
    This bridges the secret into the env so both our checks and kagglehub find
    it. No-op outside Colab, when a var is already set, or when the secret is
    unset / lacks notebook access."""
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return
    for name in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        if os.environ.get(name):
            continue
        try:
            val = userdata.get(name)
        except Exception:
            val = None  # SecretNotFound / access not granted
        if val:
            os.environ[name] = str(val).strip()


def kaggle_upload_credentials():
    """Upload a Kaggle credentials file in Colab and install it where kagglehub
    looks. Accepts the NEW-format API token (a one-line token -> ~/.kaggle/
    access_token, or JSON carrying an access_token/token field) and, for
    convenience, a legacy kaggle.json {username, key}. Sets the matching env
    vars for this kernel and chmods the file to 600 (kaggle refuses a
    world-readable creds file). Returns True on success; prints guidance and
    returns False outside Colab.

    Call this in a cell BEFORE running §2 (or just set a Colab secret named
    KAGGLE_API_TOKEN, which needs no upload at all)."""
    import stat
    cfg_dir = Path.home() / '.kaggle'
    cfg_dir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files  # type: ignore
    except Exception:
        print('Not running in Colab. Provide Kaggle creds another way:\n'
              '  - Colab secret / env KAGGLE_API_TOKEN=<token>  (new format), or\n'
              '  - write the token to ~/.kaggle/access_token, or\n'
              '  - legacy ~/.kaggle/kaggle.json {"username","key"}.')
        return False
    print('Select your Kaggle credentials file — the NEW-format API token '
          '(saved as access_token) or a legacy kaggle.json:')
    up = files.upload()
    if not up:
        print('No file uploaded.')
        return False
    name, body = next(iter(up.items()))
    text = body.decode('utf-8', 'ignore').strip()
    try:
        data = json.loads(text)
    except Exception:
        data = None
    dest = kind = None
    if isinstance(data, dict) and data.get('username') and data.get('key'):
        dest = cfg_dir / 'kaggle.json'
        dest.write_text(json.dumps({'username': data['username'], 'key': data['key']}))
        os.environ['KAGGLE_USERNAME'] = str(data['username'])
        os.environ['KAGGLE_KEY'] = str(data['key'])
        kind = 'legacy kaggle.json'
    else:
        token = None
        if isinstance(data, dict):
            token = data.get('access_token') or data.get('token') or data.get('key')
        elif text and '\n' not in text and 1 < len(text) < 8192:
            token = text  # a bare one-line token (the new format)
        if token:
            dest = cfg_dir / 'access_token'
            dest.write_text(str(token).strip())
            os.environ['KAGGLE_API_TOKEN'] = str(token).strip()
            kind = 'new-format access_token'
    if dest is None:
        print(f'Unrecognised credentials file {name!r}. Expected a new-format '
              'one-line API token or a legacy kaggle.json {"username","key"}.')
        return False
    try:
        dest.chmod(stat.S_IRUSR | stat.S_IWUSR)  # 600
    except Exception:
        pass
    print(f'Installed {kind} -> {dest}')
    return True


### Kaggle credentials (optional)

Enables the §2 real-houseplant-disease supplement. Easiest: add a Colab secret named `KAGGLE_API_TOKEN` (new format) — no upload needed. Otherwise set `KAGGLE_UPLOAD_CREDS = True` in the next cell and run it to upload your token file (or a legacy `kaggle.json`). Fully optional — skip it and the run proceeds without the supplement.

In [ ]:
# Kaggle credentials for the optional §2 houseplant supplement.
#   Easiest: add a Colab secret named KAGGLE_API_TOKEN (toggle 'Notebook
#   access' ON for it). Colab secrets are NOT auto-exported as env vars, so
#   the sync below pulls it in for you -- no upload needed.
#   Alternative: set KAGGLE_UPLOAD_CREDS = True to upload a token file / kaggle.json.
KAGGLE_UPLOAD_CREDS = False
_kaggle_sync_colab_secrets()
_have_kaggle = bool(os.environ.get('KAGGLE_API_TOKEN')
                    or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')))
if not _have_kaggle and KAGGLE_UPLOAD_CREDS:
    _have_kaggle = bool(kaggle_upload_credentials())
if _have_kaggle:
    print('Kaggle credentials detected — the §2 supplement will run (incl. the\n          eval-eligible DoctorP real-photo slice that seeds the weak-class ship gate).')
else:
    print('Kaggle: no credentials yet. Add a Colab secret KAGGLE_API_TOKEN '
          '(Notebook access ON) and re-run THIS cell, or set '
          'KAGGLE_UPLOAD_CREDS = True to upload a token file. The §2 '
          'supplement is optional and will be skipped without it.')


## §2 — Data acquisition + group-aware splits

Public sources only. No manual upload. Every saved image gets a
sidecar JSON so the splitter can be group-aware (`(source, plant_id)`
goes together, never split across train and a held-out set).

| Source | Role |
|---|---|
| **PlantDoc** (HF) | real-world train + `val_realworld` seed |
| **PlantVillage** (HF) | capped train + `val_studio`. PlantVillage **never** enters `val_realworld`, `temp_cal` or `conformal_cal` (would break exchangeability) |
| **iNaturalist** | healthy houseplant leaves (Ficus / Monstera / …), Botrytis supplement (**`Green Leaves` annotation** off for the fungus class), Animalia for NAP, and free-text physiological queries (`leaf scorch`, `chlorosis`, …) for the v2.2 deferred classes |
| **COCO val2017** | `not_a_plant` images, plant-like categories filtered out |
| **Open Images V7** | `not_a_plant` indoor-scene supplement (Couch / Lamp / Vase / Bookcase / …) — closes the indoor-clutter gap behind production NAP failures |
| **CoLeaf-DB** (Mendeley, CC BY 4.0) | curated coffee-leaf **nutrient-deficiency** supplement for the weak `nutrient_deficiency` class. **Train-only** — crop domain, so it never enters any eval/cal split (scoped like PlantVillage studio) and is down-weighted in §4's sampler |
| **Kaggle ornamental** (`pappu54/indoor-plant-disease-dataset-23-classes`) | v2.4: rose / hibiscus / chrysanthemum / **money-plant (Epipremnum, a true houseplant)** leaves with REAL disease morphology on NATURAL backgrounds — the closest public supply to the deployment domain for the weak classes (`powdery_mildew`, `viral_disease`, `leaf_spot`, `nutrient_deficiency`, `leaf_scorch`). **Train-only**, moderately down-weighted (partly augmented). Drops rust / toxicity / wilt / downy / aphid / blight / necrosis. |
| **DoctorP** (`alexanderuzhinskiy/the-doctorp-project-dataset`) | v2.4: ~4k REAL plant-doctor **app photos** (gardener's framing, no augmentation). The only public source that supplies the starved classes in their deployment framing — **Grey mold + Botrytis cinerea → gray_mold_botrytis**, **Mealybug + Scale → pests** (cottony masses / waxy bumps on the plant, not iNat entomology macros) — plus real powdery_mildew / viral / leaf_spot / nutrient_deficiency / scorch / healthy. **EVAL-ELIGIBLE** (the one Kaggle source allowed into the held-out splits): full sampler weight, seeds the real-photo eval slice. |

**v2.4 data-quality fixes** (from a hands-on audit of every source):
the `Cactus_Dactylopius` -> `pests` mapping was removed (outdoor
prickly-pear field shots, not indoor mealybug/scale); the Kaggle
houseplant set's **black-background mask-segmented Aloe** images (a
background shortcut + heavy augmentation) are now rejected at fetch by a
near-black-pixel test; the noisy `'yellowing leaves plant'` iNat query
(~9.7k generic-yellowing obs) was dropped from `nutrient_deficiency`;
and `kaggle_houseplant` is further down-weighted. v2.4 also adds **DoctorP** (real plant-doctor app photos) as the first
EVAL-ELIGIBLE supplement: it gives `gray_mold_botrytis` and `pests` a
real (if small, ~76-173/class) in-domain supply in their deployment
framing, and — because `source='doctorp'` is in every `make_*_split`
`allowed_sources` — it **seeds `val_realworld` / `val_indomain` /
`temp_cal` / `conformal_cal` with real gardener-view photos across the
whole taxonomy**, so the ship gate no longer measures only PlantDoc
field crops. Residual limitation: `gray_mold_botrytis` is still the
thinnest class (~96 real + crop proxies), and DoctorP spans
houseplants + garden/orchard plants (real phone photos, but not
exclusively indoor).

Set `REBUILD_GENERATED_SPLITS = False` if you only want to top-up an
existing run.


In [ ]:
# §2 — Data acquisition + group-aware splits
import functools, threading
from io import BytesIO
from PIL import Image as PILImage

UA = {'User-Agent': 'PlantVision/2.2 (research; contact: local-notebook)'}
REBUILD_GENERATED_SPLITS = True

if REBUILD_GENERATED_SPLITS:
    for d in (TRAIN_ROOT, VAL_ROOT):
        if d.exists():
            shutil.rmtree(d)
    ensure_layout()

# Module-level Session: TCP keep-alive across the per-image fetches in
# this cell. Per-call ``headers=`` still wins over the session default.
_SESSION = requests.Session()
_SESSION.headers.update(UA)

_RETRY_STATUS = (429, 500, 502, 503, 504)

# Cross-call dedup of remote fetches (iNat photos can appear under
# multiple taxa). Guarded by a lock because we update it from worker
# threads in the ThreadPoolExecutor pools below.
_FETCH_SEEN_SHA: set = set()
_FETCH_SEEN_LOCK = threading.Lock()

# ── HTTP helpers (exponential backoff on 429/5xx) ────────────────
def http_get_json(url, params=None, timeout=30, retries=4):
    """GET + .json(). Logs every backoff so a starved class makes the
    upstream failure visible in the run log."""
    last, delay = None, 0.8
    for attempt in range(retries):
        try:
            r = _SESSION.get(url, params=params, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                      f'-> {r.status_code}; sleeping {delay:.1f}s')
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.json()
        except Exception as e:
            last = e
            print(f'  http_get_json {url} attempt {attempt+1}/{retries} '
                  f'-> {type(e).__name__}: {e}; sleeping {delay:.1f}s')
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_get_json exhausted retries: {url}')

def http_download(url, dest, timeout=120, retries=4):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.tmp')
    last, delay = None, 1.0
    for _ in range(retries):
        try:
            with _SESSION.get(url, stream=True, timeout=timeout) as r:
                if r.status_code in _RETRY_STATUS:
                    last = requests.HTTPError(f'{r.status_code} {r.reason} on {url}')
                    time.sleep(delay); delay *= 2; continue
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk: f.write(chunk)
            tmp.replace(dest); return dest
        except Exception as e:
            last = e
            if tmp.exists():
                try: tmp.unlink()
                except Exception: pass
            time.sleep(delay); delay *= 2
    raise last if last is not None else RuntimeError(f'http_download exhausted retries: {url}')

def http_get_bytes(url, timeout=20, retries=4):
    """Returns body bytes or None — callers treat None as a soft skip
    so a single bad image doesn't abort the surrounding pool."""
    delay = 0.5
    for _ in range(retries):
        try:
            r = _SESSION.get(url, timeout=timeout)
            if r.status_code in _RETRY_STATUS:
                time.sleep(delay); delay *= 2; continue
            r.raise_for_status(); return r.content
        except Exception:
            time.sleep(delay); delay *= 2
    return None

def _stable_frac(text: str) -> float:
    return int(hashlib.sha1(text.encode('utf-8')).hexdigest()[:12], 16) / float(16**12 - 1)

def _save_rgb(img, out: Path, quality=92):
    out.parent.mkdir(parents=True, exist_ok=True)
    img.convert('RGB').save(out, quality=quality)

def _valid_image_bytes(body, min_side=160):
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        if min(im.size) < min_side: return None
        return im
    except Exception:
        return None

def _claim_sha(body: bytes) -> bool:
    """Try to claim a SHA-256. Returns True if this is the first time we've
    seen these bytes (and the caller may proceed), False on duplicate."""
    sha = hashlib.sha256(body).hexdigest()
    with _FETCH_SEEN_LOCK:
        if sha in _FETCH_SEEN_SHA:
            return False
        _FETCH_SEEN_SHA.add(sha)
    return True

def _fetch_save_remote(url, fp, ann_factory, *, timeout=20):
    """Fetch → validate as image → save RGB → write sidecar. Returns 1 on
    save, 0 on any skip (existing, failed fetch, bad image, dup)."""
    if fp.exists(): return 0
    body = http_get_bytes(url, timeout=timeout)
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, ann_factory())
    return 1

# ── iNaturalist (Leaves annotation, research-grade only) ────────
INAT_BASE = 'https://api.inaturalist.org/v1'
INAT_LEAF_TERM_ID, INAT_LEAF_TERM_VALUE_ID = 36, 38   # iNat 'Leaves' annotation, value='Green Leaves'

@functools.lru_cache(maxsize=64)
def inat_resolve_taxon(name, rank='genus'):
    res = http_get_json(f'{INAT_BASE}/taxa', {'q': name, 'rank': rank}).get('results') or []
    return res[0]['id'] if res else None

def inat_observations(taxon_id, max_n, leaf_only=True):
    rows, page, per = [], 1, 100
    while len(rows) < max_n:
        params = {'taxon_id': taxon_id, 'quality_grade': 'research', 'photos': 'true',
                  'per_page': per, 'page': page, 'order': 'desc', 'order_by': 'created_at'}
        if leaf_only:
            params['term_id'] = INAT_LEAF_TERM_ID
            params['term_value_id'] = INAT_LEAF_TERM_VALUE_ID
        batch = (http_get_json(f'{INAT_BASE}/observations', params).get('results') or [])
        if not batch: break
        rows.extend(batch)
        if len(batch) < per: break
        page += 1; time.sleep(0.25)
    return rows[:max_n]

def _photo_url(photo, size='medium'):
    raw = photo.get('url') or ''
    return re.sub(r'/square\.', f'/{size}.', raw) if raw.startswith('http') else None

def _save_one_inat(item, dest, klass, tx, label_confidence):
    obs, ph = item
    url = _photo_url(ph, 'medium')
    if not url: return 0
    fp = dest / f'inat_{tx}_{obs["id"]}_{ph["id"]}.jpg'
    if fp.exists(): return 0
    body = http_get_bytes(url)
    if not body: return 0
    if not _claim_sha(body): return 0
    img = _valid_image_bytes(body)
    if img is None: return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=str(obs.get('id')),
        species=(obs.get('taxon') or {}).get('name'),
        diagnosis=klass, severity='unknown', leaf_or_whole_plant='leaf',
        capture_session=f'inat_taxon_{tx}', source='inat',
        label_confidence=label_confidence, bbox=[0, 0, 0, 0]))
    return 1

def download_inat_class(taxon_ids, dest, klass, max_per_taxon=160, leaf_only=True,
                        workers=8, label_confidence=0.7, photos_per_obs=1):
    dest.mkdir(parents=True, exist_ok=True)
    # Dedup resolved taxon ids. iNat treats some genera as synonyms of one
    # another (e.g. Botryotinia -> Botrytis, both taxon 324264), so two names
    # in a *_TAXA list can resolve to the SAME id. Without this, the second id
    # re-fetches the identical observations + photos that SHA-dedup then drops
    # — wasted API calls/bandwidth, and on the thinnest classes (gray mold)
    # that is the slow tail of §2. dict.fromkeys preserves first-seen order.
    taxon_ids = list(dict.fromkeys(taxon_ids))
    n = 0
    for tx in taxon_ids:
        try:
            obs_list = inat_observations(tx, max_per_taxon, leaf_only=leaf_only)
        except Exception as e:
            print(f'  iNat taxon {tx} failed: {e}'); continue
        items = [(o, p) for o in obs_list for p in (o.get('photos') or [])[:photos_per_obs]]
        if not items: continue
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futs = [pool.submit(_save_one_inat, it, dest, klass, tx, label_confidence) for it in items]
            for f in as_completed(futs):
                n += f.result()
    return n

INAT_HEALTHY_TAXA = [
    ('Ficus','genus'),('Monstera','genus'),('Epipremnum','genus'),('Calathea','genus'),
    ('Spathiphyllum','genus'),('Dracaena','genus'),('Sansevieria','genus'),('Philodendron','genus'),
    # Variegated / vividly patterned ornamentals as HEALTHY negatives.
    # Their natural mottling + bold colour is the dominant viral_disease
    # false positive (the classifier reads it as mosaic — see MODEL_CARD).
    # iNat research-grade 'Calathea' skews to plain-green specimens, so the
    # strongly-patterned cultivars were under-represented; these add them.
    # (Goeppertia is the current genus for many 'Calathea'; both kept for
    # yield. No green-content filter runs on the healthy class, so vivid
    # red/pink leaves are retained rather than dropped.)
    ('Goeppertia','genus'),('Maranta','genus'),('Stromanthe','genus'),('Ctenanthe','genus'),
    ('Caladium','genus'),('Aglaonema','genus'),('Codiaeum','genus'),('Coleus','genus'),
    ('Fittonia','genus'),('Hypoestes','genus'),('Begonia','genus'),('Tradescantia','genus'),
]
# A genus taxon_id query on iNat already returns every descendant
# species, so the genus rows alone cover all Botrytis spp. The
# species-level rows we used to list added ~0 deduped yield (measured
# ~138 foliage images either way) while re-downloading photos that SHA
# dedup then dropped, so they were cut to save the redundant fetches.
INAT_BOTRYTIS_TAXA = [
    # 'Botryotinia' is an iNat SYNONYM of 'Botrytis' (both resolve to taxon
    # 324264, verified live 2026-06), so it contributes 0 distinct observations.
    # download_inat_class() dedups the resolved ids, so the synonym never
    # double-fetches; it is kept only as a resolver fallback in case iNat ever
    # re-splits the genera.
    ('Botrytis','genus'),('Botryotinia','genus'),
]

def _green_content_ratio(img) -> float:
    """Fraction of foliage-green pixels in HSV. Cheap heuristic for filtering
    fungal observations that aren't actually on a leaf (agar plates, dead
    tissue, fruit). Indoor leaf shots score 0.30–0.70; non-foliage close-ups
    score < 0.05."""
    # Pillow HSV (H,S,V each 0-255), no OpenCV dependency. The green-hue
    # band [50,120]/255 ≈ 70-170° reproduces the old OpenCV [35,85]/179.
    arr = np.asarray(img.convert('HSV'))
    h, s, v = arr[..., 0], arr[..., 1], arr[..., 2]
    mask = (h >= 50) & (h <= 120) & (s >= 40) & (v >= 50)
    return float(mask.mean())

def _green_ratio_for_path(p: Path):
    try:
        return p, _green_content_ratio(PILImage.open(p))
    except Exception:
        return p, None

def _drop_low_green_content(class_dir: Path, min_ratio: float = 0.10) -> int:
    """Remove images whose green-content ratio falls below the floor.
    Sidecars are removed alongside. PIL+cv2 work is GIL-released so we
    parallelise across EVAL_WORKERS threads."""
    paths = list(iter_images(class_dir))
    if not paths:
        return 0
    dropped = 0
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        for p, ratio in pool.map(_green_ratio_for_path, paths):
            if ratio is None or ratio >= min_ratio:
                continue
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
                dropped += 1
            except OSError:
                pass
    return dropped

def fetch_inat_supplement(healthy_per_taxon=180, botrytis_per_taxon=2400):
    healthy_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_HEALTHY_TAXA) if t]
    n = download_inat_class(healthy_ids, TRAIN_ROOT / 'healthy', 'healthy',
                            max_per_taxon=healthy_per_taxon, leaf_only=True,
                            label_confidence=0.75)
    print(f'iNat healthy += {n}')
    # Botrytis is a fungus, not a plant — iNat's `Green Leaves` annotation
    # returns ~0 hits, so we fetch leaf_only=False and accept some non-leaf
    # hosts. The green-content gate filters obvious non-foliage. v2.2 raises
    # botrytis_per_taxon to compensate for Openverse being dropped.
    botrytis_ids = [t for t in (inat_resolve_taxon(n, r) for n, r in INAT_BOTRYTIS_TAXA) if t]
    # v2.4: pull up to 3 photos per RESEARCH-GRADE observation (the API
    # returns ~2.5 photos/obs and the code previously kept only the first),
    # ~1.6x-ing the high-confidence botrytis foliage supply WITHOUT dropping to
    # the noisier needs_id grade. The green-content gate below still filters
    # non-foliage (agar plates / herbarium / fruit macros).
    n = download_inat_class(botrytis_ids, TRAIN_ROOT / 'gray_mold_botrytis', 'gray_mold_botrytis',
                            max_per_taxon=botrytis_per_taxon, leaf_only=False,
                            label_confidence=0.55, photos_per_obs=3)
    ft = _fetch_inat_freetext_class('gray_mold_botrytis', GRAY_MOLD_FREETEXT_QUERIES,
                                    source='inat', per_query=140, label_confidence=0.5)
    print(f'iNat gray_mold_botrytis free-text += {ft}')
    dropped = _drop_low_green_content(TRAIN_ROOT / 'gray_mold_botrytis', min_ratio=BOTRYTIS_GREEN_MIN_RATIO)
    print(f'iNat gray_mold_botrytis supplement += {n}, dropped {dropped} non-foliage')

# ── COCO val2017 → not_a_plant ──────────────────────────────────
COCO_CACHE = ROOT / 'cache_coco_val2017'
# Mirror used when images.cocodataset.org serves a CloudFront certificate
# whose SAN doesn't list the host (intermittent, currently broken).
# ``pcuenq/coco-2017-mirror`` exposes the canonical val2017.zip (816 MB)
# and annotations_trainval2017.zip (253 MB) as direct downloadable files
# under their canonical filenames — same SHA, same internal layout.
_COCO_HF_MIRROR = 'pcuenq/coco-2017-mirror'

def _coco_download(canonical_url: str, hf_filename: str, dest: Path) -> Path:
    """Fetch a COCO asset, HF-MIRROR-FIRST. images.cocodataset.org has been
    serving a CloudFront cert whose SAN omits the host, so the canonical-first
    order burned ~15 s of SSL retries + an alarming log line every run before
    falling back. Prefer pcuenq/coco-2017-mirror (identical SHA + layout) and
    only hit the canonical host if the mirror is unreachable."""
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    try:
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(_COCO_HF_MIRROR, hf_filename, repo_type='dataset')
        shutil.copy(src, dest)
        return dest
    except Exception as e:  # noqa: BLE001 — mirror down: try the canonical host
        print(f'  COCO HF mirror {_COCO_HF_MIRROR} failed ({type(e).__name__}: {e}); '
              f'trying canonical {canonical_url}')
        return http_download(canonical_url, dest)

def fetch_coco_for_nap(target_n=1500):
    COCO_CACHE.mkdir(parents=True, exist_ok=True)
    img_zip, ann_zip = COCO_CACHE/'val2017.zip', COCO_CACHE/'annotations.zip'
    if not (COCO_CACHE/'val2017').exists():
        print('Downloading COCO val2017 (~1 GB)…')
        _coco_download('https://images.cocodataset.org/zips/val2017.zip',
                       'val2017.zip', img_zip)
        with zipfile.ZipFile(img_zip) as z: z.extractall(COCO_CACHE)
    if not (COCO_CACHE/'annotations').exists():
        _coco_download(
            'https://images.cocodataset.org/annotations/annotations_trainval2017.zip',
            'annotations_trainval2017.zip', ann_zip,
        )
        with zipfile.ZipFile(ann_zip) as z: z.extractall(COCO_CACHE)
    ann = json.loads((COCO_CACHE/'annotations'/'instances_val2017.json').read_text())
    # COCO is not exhaustively labelled, so dropping food / dining categories
    # cuts the worst plant-leak offenders without losing the indoor-clutter
    # frames we want for NAP.
    plant_like_names = {
        'potted plant', 'vase',
        'broccoli', 'carrot', 'banana', 'apple', 'orange',
        'sandwich', 'cake', 'donut', 'pizza', 'hot dog',
        'dining table', 'bowl',
    }
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in plant_like_names}
    plant_imgs = {a['image_id'] for a in ann['annotations'] if a['category_id'] in plant_cat}
    eligible = [im for im in ann['images'] if im['id'] not in plant_imgs]
    rng = random.Random(42); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for im in eligible[:target_n]:
        src, dst = COCO_CACHE/'val2017'/im['file_name'], dest/f'coco_{im["id"]}.jpg'
        if not src.exists() or dst.exists(): continue
        shutil.copy(src, dst)
        write_sidecar(dst, CropAnnotation(plant_id=f'coco_{im["id"]}', species=None,
            diagnosis='not_a_plant', severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_val2017', source='coco', label_confidence=0.99, bbox=[0,0,0,0]))
        n += 1
    print(f'COCO: not_a_plant += {n}')

# ── iNat Animalia → not_a_plant (indoor-clutter approximation) ─
# COCO is too studio-clean to teach the OOD gate about cats / dogs / birds
# in a leaf photo. iNat research-grade Animalia gets tagged
# source='inat_animalia' so the source-weighted sampler can distinguish
# pet-in-frame NAP from real-leaf iNat rows.
INAT_ANIMALIA_KINGDOM = 'Animalia'

def fetch_inat_animalia_for_nap(target_n=500):
    kingdom_id = inat_resolve_taxon(INAT_ANIMALIA_KINGDOM, rank='kingdom')
    if not kingdom_id:
        print('iNat Animalia kingdom not resolvable; skipping NAP supplement')
        return
    obs = inat_observations(kingdom_id, target_n, leaf_only=False)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    jobs = []
    for ob in obs:
        for ph in (ob.get('photos') or []):
            if len(jobs) >= target_n * 2: break  # oversample for skip slack
            url = _photo_url(ph, 'medium')
            if not url: continue
            fp = dest / f'inat_animalia_{ob["id"]}_{ph["id"]}.jpg'
            obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
            ann = lambda obs_id=obs_id, taxon_name=taxon_name: CropAnnotation(
                plant_id=f'inat_animalia_{obs_id}', species=taxon_name,
                diagnosis='not_a_plant', severity='low',
                leaf_or_whole_plant='whole_plant',
                capture_session=f'inat_animalia_{kingdom_id}',
                source='inat_animalia', label_confidence=0.99,
                bbox=[0, 0, 0, 0])
            jobs.append((url, fp, ann))
        if len(jobs) >= target_n * 2: break
    n = 0
    pool = ThreadPoolExecutor(max_workers=8)
    try:
        futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
        for fut in as_completed(futs):
            n += fut.result()
            if n >= target_n:
                break
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    print(f'iNat Animalia: not_a_plant += {n}')

# ── Open Images V7 → not_a_plant (indoor-clutter NAP) ─────────────
# v5 bbox CSV is the canonical validation-bbox file referenced by V6/V7
# (validation annotations didn't change between revisions).
OI_CACHE = ROOT / 'cache_openimages_v7'
OI_CDN_BASE = 'https://s3.amazonaws.com/open-images-dataset/validation/'
OI_VALIDATION_BBOX_URL = 'https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv'
OI_CLASS_CSV_URL = 'https://storage.googleapis.com/openimages/2018_04/class-descriptions-boxable.csv'

# MIDs resolved at runtime from class-descriptions-boxable.csv.
OI_INDOOR_LABELS = {
    'Couch', 'Vase', 'Lamp', 'Bookcase', 'Bed', 'Refrigerator',
    'Coffee table', 'Television', 'Pillow', 'Curtain', 'Toilet',
    'Sink', 'Furniture', 'Microwave oven', 'Oven', 'Sofa bed',
    'Chair', 'Desk', 'Cabinetry', 'Wardrobe', 'Nightstand',
}
OI_PLANT_LABELS = {
    'Houseplant', 'Plant', 'Flower', 'Tree', 'Leaf', 'Fruit',
    'Flowerpot',
}

def _oi_load_label_map():
    """LabelName MID → human-readable class name."""
    csv_local = OI_CACHE / 'class-descriptions-boxable.csv'
    if not csv_local.exists():
        OI_CACHE.mkdir(parents=True, exist_ok=True)
        http_download(OI_CLASS_CSV_URL, csv_local)
    mid_to_name = {}
    with csv_local.open() as fh:
        for line in fh:
            parts = line.rstrip('\n').split(',', 1)
            if len(parts) == 2:
                mid_to_name[parts[0]] = parts[1]
    return mid_to_name

def fetch_openimages_indoor(target_n=600):
    """NAP supplement: indoor-clutter images from Open Images V7 validation."""
    OI_CACHE.mkdir(parents=True, exist_ok=True)
    bbox_csv = OI_CACHE / 'validation-annotations-bbox.csv'
    if not bbox_csv.exists():
        print('Downloading Open Images V5 validation bbox CSV (~25 MB)…')
        http_download(OI_VALIDATION_BBOX_URL, bbox_csv)
    mid_to_name = _oi_load_label_map()
    indoor_mids = {m for m, n in mid_to_name.items() if n in OI_INDOOR_LABELS}
    plant_mids  = {m for m, n in mid_to_name.items() if n in OI_PLANT_LABELS}
    if not indoor_mids:
        print('Open Images: no indoor labels resolved; skipping NAP supplement')
        return
    per_image = {}  # image_id → set(label_mid)
    with bbox_csv.open() as fh:
        fh.readline()  # header
        for line in fh:
            parts = line.split(',', 3)
            if len(parts) < 3: continue
            per_image.setdefault(parts[0], set()).add(parts[2])
    eligible = [img for img, labs in per_image.items()
                if labs & indoor_mids and not (labs & plant_mids)]
    rng = random.Random(SEED + 11); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    # Oversample 2× so the pool has slack for misses (404s, decode failures,
    # files already present) and still hits target_n.
    jobs = []
    for img_id in eligible[:target_n * 2]:
        fp = dest / f'oi_{img_id}.jpg'
        ann = lambda img_id=img_id: CropAnnotation(
            plant_id=f'oi_{img_id}', species=None,
            diagnosis='not_a_plant', severity='low',
            leaf_or_whole_plant='whole_plant',
            capture_session='openimages_v7_validation',
            source='openimages_indoor', label_confidence=0.99,
            bbox=[0, 0, 0, 0])
        jobs.append((OI_CDN_BASE + img_id + '.jpg', fp, ann))
    n = 0
    pool = ThreadPoolExecutor(max_workers=8)
    try:
        futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
        for fut in as_completed(futs):
            n += fut.result()
            if n >= target_n:
                break
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    print(f'Open Images indoor: not_a_plant += {n}')

# ── iNat → physiological classes ───────────────────────────────
# Free-text iNat queries for damage-related observations. Noisy (no curated
# annotation for "chlorosis" — iNat's Leaf annotations top out at
# `Coloured Leaves`). per_class is set high (400) because free-text recall
# is low and the round-robin claim below splits the deduped pool evenly
# across the two physiological classes. label_confidence is set low so
# label-smoothing + the curated CoLeaf supplement (high-confidence;
# fetch_coleaf_nutrient_deficiency below) dilute the residual query noise.
# The queries themselves are constrained to iconic_taxa='Plantae' and an
# operator review blocklist (below) at fetch time, so non-plant matches
# and hand-rejected observations never enter the pool in the first place.
# Stage 0 (report): broaden the in-domain free-text supply for the weak
# classes so BOTH their train and their (ratio-fixed) eval slices grow —
# the eval CIs were +/-0.2 at n~=19-28, too wide to certify a change. More
# DISTINCT query terms is the lever that adds distinct observations (raising
# per_class alone did not: the v2.3.x runs were supply-, not cap-, bound).
INAT_PHYSIOLOGICAL_QUERIES = {
    'leaf_scorch_underwatering': [
        # Pruned 6 free-text queries that returned 0 research-grade+photo
        # observations on two live checks (2026-06): 'underwatered houseplant',
        # 'leaf margin necrosis drought', 'brown leaf tips houseplant',
        # 'wilting houseplant', 'crispy leaf edges indoor', 'leaf scorch indoor
        # plant'. They only spent API calls; removing them loses no images.
        'leaf scorch', 'wilting leaves drought',
        'leaf curl drought',
        'leaf tip burn', 'crispy brown leaf edges',
        'dehydrated plant leaves', 'water stress leaf',
        'sunburn leaf',
        'leaf edge browning',
        'scorched leaves heat', 'drought stressed plant',
        'desiccated leaves',
        'heat stress foliage',
        'parched plant leaves', 'leaf browning underwatering',
        'shrivelled leaves drought',
    ],
    'nutrient_deficiency': [
        # v2.4: 'yellowing leaves plant' removed — it returned ~9.7k obs that
        # are a generic-yellowing catch-all (senescence / overwatering / light),
        # the dominant label-noise source for nutrient_deficiency.
        # v2.5: pruned 10 free-text queries that returned 0 research-grade+photo
        # observations on two live checks (2026-06): 'potassium/manganese/
        # phosphorus/calcium/zinc deficiency leaf', 'micronutrient deficiency
        # plant', 'nutrient deficiency houseplant', 'stunted pale leaves
        # nutrient', 'leaf bronzing deficiency', 'veinal chlorosis leaf'. iNat
        # free-text doesn't carry these phrasings — the deficiency signal comes
        # from MPLD + the Kaggle nutrient sets. v2.5: 'marginal leaf yellowing'
        # REMOVED - a live check measured it at ~91% of this free-text yield and it
        # is generic-yellowing noise (senescence / overwatering / light). The
        # deficiency SIGNAL now comes from MPLD + the Kaggle nutrient/ornamental
        # sets + DoctorP; the residual iNat physiological free-text is now
        # TRAIN-ONLY (inat_physiological is in no eval/cal split), so DoctorP's
        # real higher-confidence rows anchor the held-out physiological slices.
        'chlorosis leaf', 'iron chlorosis',
        'nitrogen deficiency leaf', 'magnesium deficiency leaf',
        'interveinal chlorosis',
        'leaf yellowing nitrogen',
        'pale young leaves nutrient',
        'sulphur deficiency leaf',
        'purple leaves phosphorus',
        'general leaf yellowing deficiency',
    ],
}

def _inat_freetext_observations(query, per_query, iconic_taxa=None):
    """Wrap the existing iNat layer for q= free-text search.

    ``iconic_taxa`` (e.g. ``'Plantae'``) restricts results to a top-level
    iNat group. It is passed for the physiological queries: a free-text
    search for ``'leaf scorch'`` / ``'chlorosis'`` otherwise returns
    insects, fungi and birds whose common names share those words, which
    is the dominant label-noise source for those classes. It is NOT passed
    for the gray-mold queries, whose observations are legitimately filed
    under the fungus taxon (iconic_taxon=Fungi)."""
    base = INAT_BASE + '/observations'
    params = {
        'q': query, 'quality_grade': 'research',
        'per_page': 100, 'photos': 'true', 'order': 'desc',
        'order_by': 'created_at',
    }
    if iconic_taxa:
        params['iconic_taxa'] = iconic_taxa
    page, out = 1, []
    while len(out) < per_query and page <= 5:
        params['page'] = page
        try:
            data = http_get_json(base, params=params)
        except Exception as e:
            print(f'  inat_freetext({query!r}) failed: {e}')
            break
        results = (data.get('results') or [])
        if not results: break
        out.extend(results)
        page += 1
    return out[:per_query]

# Free-text gray-mold supplement. Live checks (2026-06) show this path is
# nearly empty — only 3 of the 12 original queries return anything and the
# whole list yields ~4 observations — so 9 zero-yield queries were pruned.
# gray_mold_botrytis is carried by the Kaggle strawberry set (~621 'Gray Mold'
# instances, capped 250) + DoctorP (~96) + the iNat genus taxon (~138 foliage),
# NOT this free-text path; the 3 survivors are kept only as a marginal top-up.
GRAY_MOLD_FREETEXT_QUERIES = [
    'gray mold botrytis', 'botrytis leaf rot', 'botrytis grey mould',
]
def _fetch_inat_freetext_class(cls, queries, source, *, per_query=120,
                               label_confidence=0.5, severity='unknown',
                               leaf_or_whole='leaf'):
    """Free-text iNat search -> save photos under ``cls`` with the given
    ``source`` tag. Tops up supply-starved classes (e.g. gray mold, which
    has no PlantDoc / PlantVillage label and is iNat-taxon-capped) beyond
    what the taxon-id query alone returns."""
    dest = TRAIN_ROOT / cls; dest.mkdir(parents=True, exist_ok=True)
    saved = 0
    for q in queries:
        obs = _inat_freetext_observations(q, per_query)
        jobs = []
        for ob in obs:
            for ph in (ob.get('photos') or [])[:1]:
                url = _photo_url(ph, 'medium')
                if not url:
                    continue
                fp = dest / f'inat_ft_{cls}_{ob["id"]}_{ph["id"]}.jpg'
                obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                ann = lambda obs_id=obs_id, taxon_name=taxon_name, q=q: CropAnnotation(
                    plant_id=f'inat_ft_{obs_id}', species=taxon_name,
                    diagnosis=cls, severity=severity, leaf_or_whole_plant=leaf_or_whole,
                    capture_session=f'inat_ft_{q.replace(" ", "_")}',
                    source=source, label_confidence=label_confidence, bbox=[0, 0, 0, 0])
                jobs.append((url, fp, ann))
        pool = ThreadPoolExecutor(max_workers=8)
        try:
            futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
            for fut in as_completed(futs):
                saved += fut.result()
        finally:
            pool.shutdown(wait=False, cancel_futures=True)
    return saved

def _save_inat_phys_observation(cls, q, ob, seen_obs) -> bool:
    """Try to save one iNat free-text observation under ``cls``. Returns
    True if a photo landed (and the obs is now claimed in ``seen_obs``),
    False otherwise. Failed photo fetches do NOT add to ``seen_obs`` so
    the sibling class can still try the same obs."""
    obs_id = ob['id']
    for ph in (ob.get('photos') or []):
        url = _photo_url(ph, 'medium')
        if not url:
            continue
        fp = TRAIN_ROOT / cls / f'inat_phys_{cls}_{obs_id}_{ph["id"]}.jpg'
        if fp.exists():
            seen_obs.add(obs_id)
            return True
        body = http_get_bytes(url)
        if not body or not _claim_sha(body):
            continue
        img = _valid_image_bytes(body)
        if img is None:
            continue
        _save_rgb(img, fp)
        write_sidecar(fp, CropAnnotation(
            plant_id=f'inat_phys_{obs_id}',
            species=(ob.get('taxon') or {}).get('name'),
            diagnosis=cls, severity='medium',
            leaf_or_whole_plant='leaf',
            capture_session=f'inat_phys_{q.replace(" ", "_")}',
            source='inat_physiological',
            label_confidence=0.55, bbox=[0, 0, 0, 0]))
        seen_obs.add(obs_id)
        return True
    return False

def fetch_inat_physiological(per_class=400):
    """Free-text iNat queries → physiological classes.

    Cross-class observation dedup is fairness-critical: a
    chlorosis-and-scorch photo can't land in both classes (the label
    signal would be contradictory). The two classes share several visual
    modes (yellowing leaves, generalised wilting), so first-come-first-
    served on the dict-insertion order silently privileges
    ``leaf_scorch_underwatering`` and starves ``nutrient_deficiency``
    of ~20% of its supply.

    Fix: fetch the candidate observation pools for every class up front,
    then claim observations in **round-robin** order. Each cycle every
    still-hungry class gets exactly one new observation (skipping any
    already claimed by a sibling), so both classes converge toward
    their target at the same rate. The green-content gate is
    intentionally NOT applied here — its HSV mask rejects pale-yellow
    chlorosis and brown leaf scorch as readily as it rejects agar
    plates.
    """
    targets: dict[str, int] = {}
    for cls in INAT_PHYSIOLOGICAL_QUERIES:
        dest = TRAIN_ROOT / cls
        dest.mkdir(parents=True, exist_ok=True)
        n_have = sum(1 for _ in iter_images(dest))
        if n_have >= per_class:
            print(f'iNat physiological ({cls}): already have {n_have}, skipping')
            continue
        targets[cls] = per_class - n_have
    if not targets:
        return

    pools: dict[str, list] = {}
    for cls, target in targets.items():
        queries = INAT_PHYSIOLOGICAL_QUERIES[cls]
        per_query = max(1, target // len(queries))
        pool = []
        for q in queries:
            pool.extend(
                (q, ob)
                for ob in _inat_freetext_observations(q, per_query, iconic_taxa='Plantae')
            )
        pools[cls] = pool

    seen_obs: set = set()
    cursors = {cls: 0 for cls in targets}
    saved = {cls: 0 for cls in targets}
    done: set = set()
    while len(done) < len(targets):
        for cls in list(targets):
            if cls in done:
                continue
            if saved[cls] >= targets[cls]:
                done.add(cls); continue
            advanced = False
            while cursors[cls] < len(pools[cls]):
                q, ob = pools[cls][cursors[cls]]
                cursors[cls] += 1
                if ob['id'] in seen_obs:
                    continue
                if _save_inat_phys_observation(cls, q, ob, seen_obs):
                    saved[cls] += 1
                    advanced = True
                    break
            if not advanced:
                done.add(cls)

    for cls, n in saved.items():
        print(f'iNat physiological ({cls}): += {n}')

# ── Curated abiotic-stress supplement: CoLeaf (report Stage 2) ───
# The weak physiological classes are otherwise fed by noisy iNat free-text
# alone. CoLeaf-DB (Tuesta-Monteza et al., Data in Brief 2023; Mendeley
# brfgw46wzb; CC BY 4.0) is an EXPERT-LABELLED coffee-leaf nutrient-
# deficiency set, one zip per nutrient, leaf-level RGB. It teaches the
# nutrient-deficiency visual signature (interveinal chlorosis, marginal
# necrosis) at a HIGH label_confidence the iNat free-text supply can't.
#
# Domain scoping (so the eval distribution is never corrupted): CoLeaf is
# a CROP species, not a houseplant, so per the report it supplies the
# deficiency SIGNAL only. Its images land in TRAIN_ROOT exclusively and
# are excluded from every eval / calibration split (no make_*_split
# allowed_sources set lists 'coleaf'), exactly how PlantVillage studio
# imagery is scoped. §4 down-weights 'coleaf' in the source-weighted
# sampler so the crop domain can't pull the prior off the houseplant
# target. Only single-nutrient zips are imported (not 'more-deficiencies'
# or 'healthy'), so every imported row is an unambiguous deficiency.
INCLUDE_ABIOTIC_SUPPLEMENTS = True
_COLEAF_BASE = 'https://data.mendeley.com/public-files/datasets/brfgw46wzb/files'
COLEAF_DEFICIENCY_ZIPS = {
    # nutrient -> Mendeley public-file uuid (URLs verified live, valid zip).
    # A diverse, bandwidth-bounded subset (~125 leaf images per zip).
    'iron-Fe':      'a2684886-e7c7-4acc-adc5-56da5ceb0937',
    'nitrogen-N':   '3d5a9c99-0444-40e4-8c17-8340bf365512',
    'magnesium-Mg': '578efbbf-1e03-40ce-b18c-80ca9655ac01',
    'boron-B':      'c8787222-c731-4608-95e1-c1c481ba4cde',
}
ABIOTIC_CACHE = ROOT / '_abiotic_cache'   # under data_v2/ (gitignored);
# survives REBUILD_GENERATED_SPLITS (it only wipes train/ + val/), so the
# zips are not re-downloaded on every split rebuild.

def _save_archive_member(target_cls, source, name_prefix, name_in_zip, body, *,
                         label_confidence, species, severity='medium', min_side=160):
    """Validate one in-archive image's bytes -> save under target_cls with
    a sidecar, sharing the cross-source SHA dedup. ``name_prefix`` makes the
    on-disk stem unique across archives (two zips can hold a '1.jpg');
    ``source`` is the sampler/weight tag. Returns 1 on save else 0."""
    stem = re.sub(r'[^A-Za-z0-9_.-]', '_', f'{name_prefix}_{Path(name_in_zip).stem}')
    fp = TRAIN_ROOT / target_cls / f'{stem}.jpg'
    if fp.exists():
        return 0
    if not _claim_sha(body):
        return 0
    img = _valid_image_bytes(body, min_side=min_side)
    if img is None:
        return 0
    _save_rgb(img, fp)
    write_sidecar(fp, CropAnnotation(
        plant_id=f'{name_prefix}_{Path(name_in_zip).stem}', species=species,
        diagnosis=target_cls, severity=severity, leaf_or_whole_plant='leaf',
        capture_session=source, source=source,
        label_confidence=label_confidence, bbox=[0, 0, 0, 0]))
    return 1

def fetch_coleaf_nutrient_deficiency(per_zip_cap=180, label_confidence=0.90):
    """CoLeaf nutrient zips -> nutrient_deficiency. Streams members straight
    from the downloaded archive (no full extract), caps per zip, SHA-dedups
    against every other source. Fails soft (via _try_fetch) if Mendeley is
    unreachable -- iNat physiological still meets the per-class floor, so a
    CoLeaf outage never blocks the run nor corrupts any class."""
    ABIOTIC_CACHE.mkdir(parents=True, exist_ok=True)
    dest = TRAIN_ROOT / 'nutrient_deficiency'
    dest.mkdir(parents=True, exist_ok=True)
    total = 0
    for nutrient, uuid in COLEAF_DEFICIENCY_ZIPS.items():
        url = f'{_COLEAF_BASE}/{uuid}/file_downloaded'
        archive = ABIOTIC_CACHE / f'coleaf_{nutrient}.zip'
        try:
            http_download(url, archive)
        except Exception as e:
            msg = str(e)
            print(f'  CoLeaf {nutrient}: download failed ({type(e).__name__}: {e}); skipping')
            if '403' in msg or 'Forbidden' in msg:
                print('    NOTE: Mendeley/Cloudflare blocks datacenter IPs (incl. Colab),\n'
                      '    so this 403 is the runtime environment, not a bug or a dead URL\n'
                      '    (it downloads fine from a browser/residential IP). To restore the\n'
                      '    coffee Fe/N/Mg/B nutrient signal from Colab, mirror CoLeaf-DB\n'
                      '    (DOI 10.17632/brfgw46wzb.1) to a Kaggle dataset and add its slug to\n'
                      '    the Kaggle supplement; nutrient_deficiency still trains on iNat + MPLD.')
            continue
        saved = 0
        try:
            with zipfile.ZipFile(archive) as z:
                members = [m for m in z.namelist()
                           if not m.endswith('/') and Path(m).suffix.lower() in IMG_EXTS]
                random.Random(SEED).shuffle(members)
                for m in members:
                    if saved >= per_zip_cap:
                        break
                    try:
                        body = z.read(m)
                    except Exception:
                        continue
                    saved += _save_archive_member(
                        'nutrient_deficiency', 'coleaf', f'coleaf_{nutrient}', m, body,
                        label_confidence=label_confidence,
                        species=f'Coffea (CoLeaf {nutrient} deficiency)')
        except zipfile.BadZipFile:
            print(f'  CoLeaf {nutrient}: corrupt archive; clearing cache + skipping')
            archive.unlink(missing_ok=True)
            continue
        print(f'CoLeaf {nutrient}: += {saved}')
        total += saved
    print(f'CoLeaf nutrient_deficiency total: += {total}')
    return total

# ── Curated abiotic-stress supplement #2: MPLD (report Stage 2) ──
# A SECOND, species-diverse nutrient-deficiency source so the deficiency
# signal is not tied to coffee morphology alone. MPLD (Maize Phosphorus
# Leaf Deficiency Database, 224×224; Zenodo 10041514; CC BY 4.0) is an
# expert-controlled maize phosphorus set shot with five cameras under
# three regimes. Only the deficiency folders are imported:
#   '-P'   = full phosphorus starvation        -> nutrient_deficiency
#   '-P50' = half-dose (partial) deficiency     -> nutrient_deficiency
#   '_C'   = control (fully fed = HEALTHY)       -> EXCLUDED
# Dropping '_C' is the same rule CoLeaf applies (no 'healthy'/'mixed'
# members), so a fully-fed leaf is never mislabelled deficient. Same
# domain scoping as CoLeaf: maize is a CROP, so rows are TRAIN-only (no
# make_*_split allowed_sources lists 'mpld') and §4 down-weights the
# 'mpld' source (0.5) so the crop domain can't pull the LA prior off the
# houseplant target. P-deficiency adds purpling + interveinal chlorosis,
# a signature complementary to CoLeaf's Fe/N/Mg/B chlorosis/necrosis.
_MPLD_URL = 'https://zenodo.org/api/records/10041514/files/MPLD_databse.zip/content'
MPLD_DEFICIENCY_DIRS = ('-P', '-P50')   # '_C' (control / healthy) is excluded

def fetch_mpld_phosphorus_deficiency(cap=700, label_confidence=0.90):
    """MPLD maize phosphorus-deficiency zip -> nutrient_deficiency. Streams
    members straight from the archive (no full extract), importing ONLY the
    '-P' / '-P50' deficiency folders and skipping the '_C' control so a
    fully-fed leaf is never labelled deficient. Shuffles across the five
    cameras + two deficiency levels and caps the total for a balanced,
    bandwidth-bounded subset; SHA-dedups against every other source. Fails
    soft (via _try_fetch) if Zenodo is unreachable -- iNat physiological +
    CoLeaf still meet the per-class floor, so an outage neither blocks the
    run nor corrupts the class."""
    ABIOTIC_CACHE.mkdir(parents=True, exist_ok=True)
    dest = TRAIN_ROOT / 'nutrient_deficiency'
    dest.mkdir(parents=True, exist_ok=True)
    archive = ABIOTIC_CACHE / 'mpld.zip'
    try:
        http_download(_MPLD_URL, archive)
    except Exception as e:
        print(f'  MPLD: download failed ({type(e).__name__}: {e}); skipping')
        return 0
    saved = 0
    try:
        with zipfile.ZipFile(archive) as z:
            members = []
            for m in z.namelist():
                if m.endswith('/') or Path(m).suffix.lower() not in IMG_EXTS:
                    continue
                # path layout: MPLD_databse/<camera>/<level>/<file>
                parts = m.split('/')
                if len(parts) >= 4 and parts[-2] in MPLD_DEFICIENCY_DIRS:
                    members.append(m)
            random.Random(SEED).shuffle(members)
            for m in members:
                if saved >= cap:
                    break
                try:
                    body = z.read(m)
                except Exception:
                    continue
                parts = m.split('/')
                camera, level = parts[-3], parts[-2]
                saved += _save_archive_member(
                    'nutrient_deficiency', 'mpld', f'mpld_{camera}_{level}', m, body,
                    label_confidence=label_confidence,
                    species=f'Zea mays (MPLD {level} phosphorus deficiency)')
    except zipfile.BadZipFile:
        print('  MPLD: corrupt archive; clearing cache + skipping')
        archive.unlink(missing_ok=True)
        return 0
    print(f'MPLD nutrient_deficiency total: += {saved}')
    return saved

# ── Kaggle houseplant-disease supplement (operator opt-in) ───────
# Real INDOOR houseplant/succulent disease photos -- the in-domain supply
# the keyless HF/iNat sources under-provide. Requires Kaggle credentials in
# the environment (KAGGLE_USERNAME + KAGGLE_KEY, or ~/.kaggle/kaggle.json);
# FAIL-SOFT when absent so a keyless run is unaffected. TRAIN-ONLY: the
# 'kaggle_houseplant' source is in NO make_*_split allowed_sources, so it
# never enters an eval/calibration slice. It is moderately down-weighted in
# the sampler (the set may be partly studio-framed; unverified until a run
# prints its structure) and its LA prior is decoupled lower, exactly like
# the CoLeaf/MPLD crop supplements.
INCLUDE_KAGGLE_SUPPLEMENTS = True
KAGGLE_HOUSEPLANT_SLUG = 'abdulahad0296/indoor-plant-disease-detection-dataset'
# Best-effort folder-name -> our-class map (keys are _pv_normalise'd). UNMAPPED
# folders are DROPPED (never guessed into a class); every discovered folder +
# count + resolved mapping is printed so the operator can refine this map
# against the first run before trusting the import.
# Folder names here are <Species>_<Condition> (e.g. 'Aloe_LeafSpot',
# 'Cactus_Dactylopius_Opuntia'), so we map on the CONDITION keyword inside the
# normalised folder name, not the whole string. A DROP list is checked FIRST so
# a condition outside our taxonomy is never force-mapped by a looser keyword:
#   * rust           -> different morphology (we drop rust everywhere, cf. PlantDoc)
#   * *toxicity*     -> a nutrient TOXICITY is the OPPOSITE signal to a deficiency
#   * bacterial wilt -> a vascular/systemic disease, not a leaf spot
# Anything still unmatched is dropped (never guessed); every folder + count +
# resolved class is printed so the mapping stays auditable per run.
KAGGLE_DROP_KEYWORDS = ('rust', 'toxicity', 'bacterial_wilt', 'wilt_disease',
                        'root_rot', 'edema', 'oedema')
# (keyword, our_class), checked in order; first substring hit wins. sunburn /
# heat scorch / leaf-tip necrosis map to leaf_scorch_underwatering (consistent
# with how that class is already fed by iNat 'sunburn leaf' / 'scorched leaves
# heat' / 'leaf tip burn'). 'withering' is deliberately NOT mapped: on snake
# plant etc. it is usually over-watering / root rot, not under-watering scorch,
# and the scorch class is the weakest -- purity beats the extra (redundant)
# volume. anthracnose is a fungal leaf-spot morphology (cf. the PlantDoc map);
# Dactylopius (cochineal) is a scale-like pest.
KAGGLE_CONDITION_KEYWORDS = [
    ('healthy', 'healthy'),
    ('anthracnose', 'leaf_spot'), ('leaf_spot', 'leaf_spot'), ('leafspot', 'leaf_spot'),
    ('bacterial_spot', 'leaf_spot'), ('fungal', 'leaf_spot'),
    # v2.4: 'dactylopius'/'cochineal' removed — the only folder they matched
    # (Cactus_Dactylopius_Opuntia) is OUTDOOR prickly-pear field shots, not the
    # indoor mealybug/scale close-up the 'pests' class targets.
    ('mealybug', 'pests'), ('mealy', 'pests'), ('scale', 'pests'),
    # v2.4: 'sunburn' removed — the only match (Aloe_Sunburn) is black-bg
    # segmented + augmented and ~indistinguishable from Aloe_Healthy.
    ('scorch', 'leaf_scorch_underwatering'),
    ('tip_necrosis', 'leaf_scorch_underwatering'),
    ('underwater', 'leaf_scorch_underwatering'), ('under_watering', 'leaf_scorch_underwatering'),
    ('chlorosis', 'nutrient_deficiency'), ('deficien', 'nutrient_deficiency'),
    ('powdery', 'powdery_mildew'),
    ('botrytis', 'gray_mold_botrytis'), ('gray_mold', 'gray_mold_botrytis'),
    ('grey_mold', 'gray_mold_botrytis'), ('gray_mould', 'gray_mold_botrytis'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
]

def _kaggle_map_folder(folder_norm):
    """Resolve a normalised <species>_<condition> folder name to one of CLASSES,
    or None. DROP keywords win first; then the first matching condition keyword;
    anything unmatched (or mapped to a class not in the taxonomy) returns None."""
    if any(k in folder_norm for k in KAGGLE_DROP_KEYWORDS):
        return None
    for kw, cls in KAGGLE_CONDITION_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

def _kaggle_download(slug):
    """Download a Kaggle dataset via kagglehub; return the local root Path or
    None. Fail-soft: missing package / credentials / network each return None
    with a printed reason so a keyless run is wholly unaffected."""
    import os
    if not INCLUDE_KAGGLE_SUPPLEMENTS:
        return None
    try:
        import kagglehub
    except Exception:
        try:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'],
                           check=True)
            import kagglehub
        except Exception as e:
            print(f'  Kaggle: kagglehub unavailable ({type(e).__name__}); skipping')
            return None
    # Pull a Colab secret (KAGGLE_API_TOKEN) into the env first -- Colab
    # secrets are not auto-exported, so without this a set secret looks
    # like 'no credentials'. Accept BOTH the new-format API token (env
    # KAGGLE_API_TOKEN or ~/.kaggle/access_token) and legacy username+key.
    _kaggle_sync_colab_secrets()
    cfg = Path.home() / '.kaggle'
    has_creds = bool(
        os.environ.get('KAGGLE_API_TOKEN')                                     # new (env)
        or (cfg / 'access_token').exists()                                     # new (file)
        or (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))  # legacy env
        or (cfg / 'kaggle.json').exists()                                      # legacy file
    )
    if not has_creds:
        print('  Kaggle: no credentials. Add a Colab secret KAGGLE_API_TOKEN '
              '(with Notebook access ON) or env KAGGLE_API_TOKEN, or run '
              'kaggle_upload_credentials(); legacy KAGGLE_USERNAME+KAGGLE_KEY / '
              'kaggle.json also work. Skipping the Kaggle supplement')
        return None
    try:
        return Path(kagglehub.dataset_download(slug))
    except Exception as e:
        print(f'  Kaggle download failed for {slug} ({type(e).__name__}: {e}); skipping')
        return None

def _stratified_folder_order(items, seed=SEED):
    """Round-robin index order across source folders so one big folder can't
    dominate a thin class. ``items`` is a list of ``(path, folder_norm)``; we
    bucket indices by ``folder_norm``, shuffle each bucket deterministically,
    then interleave one index per folder per cycle. The per-class save loops
    stop at their cap, so this makes the documented 'diverse cross-folder
    sample' actually hold — plain random.sample drew folders PROPORTIONALLY,
    letting e.g. a 3k-image Rose_Black_Spot folder swamp a 500-image sibling
    in the same class pool (or Aloe swamp the other species in healthy).
    Single-folder pools are returned in a plain deterministic shuffle."""
    from collections import defaultdict, deque
    buckets = defaultdict(list)
    for i, (_p, norm) in enumerate(items):
        buckets[norm].append(i)
    rng = random.Random(seed)
    queues = []
    for norm in sorted(buckets):            # deterministic folder order
        idxs = buckets[norm]; rng.shuffle(idxs)
        queues.append(deque(idxs))
    order = []
    while queues:
        nxt = []
        for q in queues:
            order.append(q.popleft())
            if q:
                nxt.append(q)
        queues = nxt
    return order


def _is_black_bg_segmented(body, thr=18, frac=0.5):
    """True if > ``frac`` of pixels are near-black (max channel <= ``thr``) — the
    signature of a leaf MASK-SEGMENTED onto a black background. Such images teach
    a background shortcut and are usually augmented duplicates, so we reject them
    from the Kaggle houseplant set. Measured: real-photo folders score ~0.00-0.03
    black-frac, segmented Aloe ~0.78-0.83, so 0.5 separates them cleanly."""
    try:
        im = PILImage.open(BytesIO(body)).convert('RGB')
        im.thumbnail((96, 96))
        a = np.asarray(im).astype(np.int16)
        return float((a.max(axis=2) <= thr).mean()) > frac
    except Exception:
        return False

def fetch_kaggle_houseplant_disease(slug=KAGGLE_HOUSEPLANT_SLUG, per_class_cap=250,
                                    label_confidence=0.70):
    """Real indoor houseplant/succulent photos from Kaggle -> TRAIN. Maps each
    <Species>_<Condition> folder to our taxonomy by CONDITION keyword (dropping
    unmapped/ambiguous ones), then for each class draws a DIVERSE cross-folder
    sample up to per_class_cap (so one Aloe-heavy folder can't dominate a thin
    class), SHA-dedups against every other source, and prints the per-folder
    mapping + the per-class totals. label_confidence is moderate because the set
    is partly augmented. TRAIN-ONLY (source 'kaggle_houseplant'); fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    pools = {}   # our_class -> list[(path, folder_norm)]
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    for folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(folder)
        mapped = _kaggle_map_folder(norm)
        print(f'    {folder!r:42s} n={len(files):5d} -> {mapped or "DROP"}')
        if mapped is not None:
            pools.setdefault(mapped, []).extend((f, norm) for f in files)
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            if _is_black_bg_segmented(body):   # v2.4: drop mask-segmented
                continue                        # (black-bg) leaves
            got += _save_archive_member(
                cls, 'kaggle_houseplant', f'kaggle_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'houseplant (Kaggle {slug.split("/")[-1]})')
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  Kaggle houseplant per-class: {saved_per_class}')
    print(f'  Kaggle houseplant total: += {total}')
    return total

# ── Kaggle strawberry gray-mold supplement (lifts the binding thin class) ──
# gray_mold_botrytis is supply-bound at the iNat ceiling (~78 train). The Kaggle
# Strawberry Disease Detection set carries a real Botrytis 'Gray Mold' class, so
# we use it as a CROP-DOMAIN proxy for the gray-mold texture -- the same role
# CoLeaf/MPLD play for nutrient_deficiency. It is a COCO instance-segmentation
# set (images + JSON, not class folders), so we parse the annotations to select
# gray-mold images. TRAIN-ONLY, down-weighted, prior-decoupled; the houseplant/
# iNat gray-mold stays the in-domain anchor + the eval slice.
KAGGLE_STRAWBERRY_SLUG = 'usmanafzaal/strawberry-disease-detection-dataset'

def _is_graymold_name(name):
    n = re.sub(r'[^a-z]', '', str(name).lower())   # 'Gray Mold' -> 'graymold'
    return any(t in n for t in ('graymold', 'greymold', 'graymould', 'greymould', 'botrytis'))

def fetch_kaggle_strawberry_graymold(slug=KAGGLE_STRAWBERRY_SLUG, cap=250,
                                     label_confidence=0.70):
    """Botrytis GRAY MOLD images from the Kaggle strawberry-disease set ->
    gray_mold_botrytis (TRAIN-ONLY). Handles BOTH annotation schemas: LabelMe
    per-image JSON (shapes[].label -- this dataset) and a single COCO file
    (categories/images/annotations), plus a class-folder fallback. Selects
    images whose annotation carries a gray-mold/Botrytis label, resolves the
    sibling image, SHA-dedups, caps, prints the labels it saw. Fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    def _resolve(jp, fname):
        c = jp.with_suffix('.jpg')
        if c.exists():
            return c
        if fname:
            c2 = jp.parent / Path(fname).name
            if c2.exists():
                return c2
        return None
    gm_paths, seen_labels = set(), set()
    for jp in root.rglob('*.json'):
        try:
            data = json.loads(jp.read_text(encoding='utf-8', errors='ignore'))
        except Exception:
            continue
        if not isinstance(data, dict):
            continue
        shapes = data.get('shapes')
        if isinstance(shapes, list):                 # LabelMe per-image JSON
            labs = [str(sh.get('label', '')) for sh in shapes]
            seen_labels.update(labs)
            if any(_is_graymold_name(l) for l in labs):
                img = _resolve(jp, data.get('imagePath'))
                if img:
                    gm_paths.add(img)
            continue
        if {'categories', 'images', 'annotations'} <= set(data):   # COCO single file
            catname = {c['id']: c.get('name', '') for c in data['categories']}
            seen_labels.update(str(v) for v in catname.values())
            gm_ids = {cid for cid, nm in catname.items() if _is_graymold_name(nm)}
            if gm_ids:
                imgname = {im['id']: im.get('file_name', '') for im in data['images']}
                for a in data['annotations']:
                    if a.get('category_id') in gm_ids:
                        c = _resolve(jp, imgname.get(a.get('image_id'), ''))
                        if c:
                            gm_paths.add(c)
    if seen_labels:
        print(f'  Kaggle {slug}: annotation labels = {sorted(seen_labels)}')
    if not gm_paths:                                 # class-folder fallback
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() in IMG_EXTS and _is_graymold_name(p.parent.name):
                gm_paths.add(p)
    paths = sorted(gm_paths)
    print(f'  Kaggle {slug}: {len(paths)} gray-mold images found')
    if not paths:
        print(f'  Kaggle {slug}: no gray-mold images; skipping')
        return 0
    (TRAIN_ROOT / 'gray_mold_botrytis').mkdir(parents=True, exist_ok=True)
    got = 0
    for idx in random.Random(SEED).sample(range(len(paths)), len(paths)):
        if got >= cap:
            break
        try:
            body = paths[idx].read_bytes()
        except Exception:
            continue
        got += _save_archive_member(
            'gray_mold_botrytis', 'kaggle_strawberry', 'kaggle_straw_gm',
            paths[idx].name, body, label_confidence=label_confidence,
            species='Fragaria (strawberry Botrytis gray mold)')
    print(f'  Kaggle strawberry gray_mold: += {got}')
    return got

# ── Kaggle nutrient-deficiency supplement (Colab-reachable CoLeaf replacement) ──
# The Mendeley CoLeaf source is intermittently IP-blocked from Colab. This rice
# N/P/K set (guy007/nutrientdeficiencysymptomsinrice) is reachable via the Kaggle
# secret and adds the N/K deficiency diversity MPLD's maize-P lacks. Same role as
# CoLeaf/MPLD: a crop-domain nutrient proxy -- TRAIN-ONLY, down-weighted,
# prior-decoupled; the iNat houseplant chlorosis stays the in-domain anchor.
KAGGLE_NUTRIENT_SLUG = 'guy007/nutrientdeficiencysymptomsinrice'
_NUTRIENT_KEYWORDS = ('nitrogen', 'phosphor', 'potassium', 'iron', 'magnesium',
                      'manganese', 'boron', 'calcium', 'sulphur', 'sulfur',
                      'zinc', 'chlorosis', 'deficien')
# Some sets name folders by element SYMBOL ('N','P','K','Fe',...) instead of
# the full word; match those by EXACT normalised name so we catch both
# conventions without a single letter false-matching a longer folder name.
_NUTRIENT_SYMBOLS = {'n', 'p', 'k', 'fe', 'mg', 'ca', 'mn', 'zn', 'b', 's', 'cu', 'mo'}

def fetch_kaggle_nutrient_deficiency(slug=KAGGLE_NUTRIENT_SLUG, cap=200,
                                     label_confidence=0.75):
    """Crop nutrient-deficiency images from a Kaggle dataset -> nutrient_deficiency
    (TRAIN-ONLY). Maps any folder whose name contains a nutrient keyword (dropping
    healthy/unmapped), draws a DIVERSE cross-folder sample to cap, SHA-dedups,
    prints the folder->class mapping. Fail-soft via _kaggle_download."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    pool = []
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    for folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(folder)
        mapped = ('nutrient_deficiency'
                  if ((any(k in norm for k in _NUTRIENT_KEYWORDS) or norm in _NUTRIENT_SYMBOLS)
                      and 'healthy' not in norm)
                  else None)
        print(f'    {folder!r:42s} n={len(files):5d} -> {mapped or "DROP"}')
        if mapped is not None:
            pool.extend((f, norm) for f in files)
    if not pool:
        print(f'  Kaggle {slug}: no nutrient-deficiency folders matched; skipping')
        return 0
    (TRAIN_ROOT / 'nutrient_deficiency').mkdir(parents=True, exist_ok=True)
    got = 0
    for idx in _stratified_folder_order(pool):
        if got >= cap:
            break
        path, norm = pool[idx]
        try:
            body = path.read_bytes()
        except Exception:
            continue
        got += _save_archive_member(
            'nutrient_deficiency', 'kaggle_nutrient', f'kaggle_nut_{norm}',
            path.name, body, label_confidence=label_confidence,
            species=f'crop nutrient deficiency (Kaggle {slug.split("/")[-1]})')
    print(f'  Kaggle nutrient_deficiency: += {got}')
    return got

# ── Kaggle melon broadleaf nutrient-deficiency supplement (v2.5.1, BROADLEAF) ──
# binnassor89/melon-macronutrient-deficiency-dataset: melon (Cucurbitaceae)
# leaves with single-nutrient deficiency (nitrogen / calcium / potassium) on a
# clean NEUTRAL (non-segmented) background. Verified live (2026-06): ~50 imgs per
# nutrient, min-side ~500px, near-black-bg fraction ~0.00 (NOT mask-segmented,
# unlike the rejected banana set). Role: a BROADLEAF nutrient proxy whose lobed
# dicot leaf morphology (interveinal chlorosis + marginal necrosis on a wide
# lamina) transfers to houseplants far better than the maize-P (MPLD) /
# rice-N/P/K GRASS-BLADE proxies. Same scoping as every other crop proxy:
# TRAIN-ONLY (source 'kaggle_melon' is in NO make_*_split allowed_sources),
# down-weighted + prior-decoupled in §4, SHA-deduped, fail-soft. The 'health'
# folder is dropped (deficiency signal only).
KAGGLE_MELON_SLUG = 'binnassor89/melon-macronutrient-deficiency-dataset'
def fetch_kaggle_melon_nutrient(slug=KAGGLE_MELON_SLUG, cap=150, label_confidence=0.85):
    """Melon broadleaf single-nutrient-deficiency images -> nutrient_deficiency
    (TRAIN-ONLY). Maps every folder whose normalised name carries a nutrient
    keyword (dropping 'health'/'healthy'), draws a DIVERSE cross-folder sample to
    cap, SHA-dedups, prints the folder->class mapping. Fail-soft via
    _kaggle_download. Folder 'potasium' (a dataset typo for 'potassium') is
    matched explicitly so those deficiency leaves are not silently dropped."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    pool = []
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    for folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(folder)
        is_def = (
            ('health' not in norm)
            and (any(k in norm for k in _NUTRIENT_KEYWORDS)
                 or norm in _NUTRIENT_SYMBOLS
                 or 'potasium' in norm)            # dataset typo for 'potassium'
        )
        mapped = 'nutrient_deficiency' if is_def else None
        print(f'    {folder!r:24s} n={len(files):5d} -> {mapped or "DROP"}')
        if mapped is not None:
            pool.extend((f, norm) for f in files)
    if not pool:
        print(f'  Kaggle {slug}: no nutrient-deficiency folders matched; skipping')
        return 0
    (TRAIN_ROOT / 'nutrient_deficiency').mkdir(parents=True, exist_ok=True)
    got = 0
    for idx in _stratified_folder_order(pool):
        if got >= cap:
            break
        path, norm = pool[idx]
        try:
            body = path.read_bytes()
        except Exception:
            continue
        got += _save_archive_member(
            'nutrient_deficiency', 'kaggle_melon', f'kaggle_melon_{norm}',
            path.name, body, label_confidence=label_confidence,
            species=f'Cucumis melo (melon {norm} deficiency)')
    print(f'  Kaggle melon nutrient_deficiency: += {got}')
    return got

# ── Kaggle real-houseplant HEALTHY supplement (curated, not scraped) ──────
# A large, manually-curated set of REAL indoor houseplant photos across ~47
# species (kacpergregorowicz/house-plant-species). All healthy specimens, so
# every folder maps to `healthy`. Value is DOMAIN + SPECIES DIVERSITY (real
# houseplants incl. variegated species) which reduces the healthy<->disease
# false positives, not volume -- so it's capped low and down-weighted. A
# diverse cross-species sample, TRAIN-ONLY (the labels are species, not
# disease-verified, so they don't enter the eval/cal slices).
KAGGLE_SPECIES_SLUG = 'kacpergregorowicz/house-plant-species'

def fetch_kaggle_houseplant_species(slug=KAGGLE_SPECIES_SLUG, cap=400,
                                    label_confidence=0.70):
    """Curated real-houseplant species photos -> healthy (TRAIN-ONLY). Pools
    images across all species folders, draws a DIVERSE cap-sized sample so no
    one species dominates, SHA-dedups, prints the species folders it saw.
    Fail-soft via _kaggle_download."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    print(f'  Kaggle {slug}: {len(by_folder)} species folders -> healthy')
    pool = [(f, _pv_normalise(folder)) for folder, files in by_folder.items() for f in files]
    (TRAIN_ROOT / 'healthy').mkdir(parents=True, exist_ok=True)
    got = 0
    for idx in _stratified_folder_order(pool):
        if got >= cap:
            break
        path, norm = pool[idx]
        try:
            body = path.read_bytes()
        except Exception:
            continue
        got += _save_archive_member(
            'healthy', 'kaggle_species', f'kaggle_sp_{norm}', path.name, body,
            label_confidence=label_confidence, species=f'houseplant ({norm})')
    print(f'  Kaggle healthy (species): += {got}')
    return got

# ── Kaggle ornamental/houseplant disease supplement (v2.4, in-domain-ish) ──
# pappu54/indoor-plant-disease-dataset-23-classes: rose / hibiscus / turmeric /
# chrysanthemum / MONEY-PLANT (Epipremnum = pothos, a TRUE houseplant) leaves
# with REAL disease morphology on NATURAL backgrounds — the closest public
# supply to the deployment domain for the weak disease classes (powdery_mildew,
# viral_disease, leaf_spot, nutrient_deficiency, leaf_scorch). Empirically
# verified clean morphology; partly augmented, so down-weighted (above the crop
# proxies, below true in-domain). TRAIN-ONLY (source 'kaggle_ornamental'); the
# slug is in NO make_*_split allowed_sources, so it never enters an eval/cal
# slice. Fail-soft via _kaggle_download. DROP list (checked first): rust
# (morphology we drop everywhere), *toxicity* (opposite of a deficiency),
# *wilt* (vascular/systemic), downy (downy != powdery mildew), aphid
# (!= mealybug/scale 'pests'), blight/necrosis/blotch (ambiguous -> unmapped).
KAGGLE_ORNAMENTAL_SLUG = 'pappu54/indoor-plant-disease-dataset-23-classes'
KAGGLE_ORN_DROP = ('rust', 'toxicity', 'wilt', 'downy', 'aphid', 'blight',
                   'necrosis', 'blotch')
KAGGLE_ORN_KEYWORDS = [
    ('healthy', 'healthy'),
    ('powdery', 'powdery_mildew'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
    ('black_spot', 'leaf_spot'), ('leaf_spot', 'leaf_spot'),
    ('leafspot', 'leaf_spot'), ('septoria', 'leaf_spot'),
    ('bacterial_leaf_spot', 'leaf_spot'), ('anthracnose', 'leaf_spot'),
    ('chlorosis', 'nutrient_deficiency'), ('deficien', 'nutrient_deficiency'),
    ('scorch', 'leaf_scorch_underwatering'),
]
def _orn_map_folder(folder_norm):
    """Resolve a normalised <plant>_<condition> folder to one of CLASSES or None.
    DROP keywords win first; then the first matching condition keyword."""
    if any(k in folder_norm for k in KAGGLE_ORN_DROP):
        return None
    for kw, cls in KAGGLE_ORN_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

def fetch_kaggle_ornamental_disease(slug=KAGGLE_ORNAMENTAL_SLUG, per_class_cap=300,
                                    label_confidence=0.75):
    """Ornamental/houseplant disease set -> TRAIN. Maps each <plant>_<condition>
    folder to our taxonomy by CONDITION keyword (dropping the ambiguous /
    out-of-taxonomy ones), then for each class draws a DIVERSE cross-folder
    sample to per_class_cap (so one big folder can't dominate a thin class),
    SHA-dedups against every other source, and prints the per-folder mapping +
    per-class totals. TRAIN-ONLY (source 'kaggle_ornamental'); fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    pools = {}
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    for folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(folder)
        mapped = _orn_map_folder(norm)
        print(f'    {folder!r:46s} n={len(files):5d} -> {mapped or "DROP"}')
        if mapped is not None:
            pools.setdefault(mapped, []).extend((f, norm) for f in files)
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            got += _save_archive_member(
                cls, 'kaggle_ornamental', f'orn_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'ornamental/houseplant (Kaggle {slug.split("/")[-1]})')
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  Kaggle ornamental per-class: {saved_per_class}')
    print(f'  Kaggle ornamental total: += {total}')
    return total

# ── DoctorP plant-doctor app dataset (v2.4, REAL deployment-domain) ──────────
# alexanderuzhinskiy/the-doctorp-project-dataset: ~4k REAL user phone photos
# from a plant-doctor app, labelled by condition (no augmentation, gardener's
# framing). It is the ONLY public source that supplies the chronically-starved
# classes in their DEPLOYMENT framing: Grey mold + Botrytis cinerea ->
# gray_mold_botrytis, Mealybug + Scale -> pests (cottony masses / waxy bumps ON
# a plant, not iNat entomology macros), plus real powdery_mildew / viral /
# leaf_spot / nutrient_deficiency / scorch / healthy. Small per class (76-351)
# and low-res (~128 px), but real.
#
# UNLIKE the other Kaggle sources it is EVAL-ELIGIBLE: source 'doctorp' is in
# every make_*_split allowed_sources, so a held-out fraction seeds val_realworld
# / val_indomain / temp_cal / conformal_cal. This is the v2.4 fix for the
# field-crop-seeded ship gate — the gate now measures real gardener-view photos
# across the whole taxonomy, not just PlantDoc crop leaves. Full sampler weight
# (it IS the deployment distribution). Fail-soft via _kaggle_download.
DOCTORP_SLUG = 'alexanderuzhinskiy/the-doctorp-project-dataset'
DOCTORP_DROP = ('spider_mite', 'mite', 'rust', 'scab', 'esca', 'blight', 'rot',
                'shot_hole', 'hole', 'shute', 'gall', 'aphid', 'lichen', 'thrip',
                'edema', 'oedema', 'whitefly', 'miner', 'mechanical', 'turgor',
                'deformation', 'downy', 'wilt', 'foliage', 'sooty', 'frost',
                'canker', 'beetle', 'ants', 'slug', 'caterpillar', 'wireworm',
                'monilia', 'eyespot', 'chaff', 'polypore', 'anthocyanosis',
                'gryllotalpa', 'coccomyces', 'cyclamen', 'yellow', 'black_spot',
                # 'tubercular' (DoctorP 'Tubercular necrosis', ~40 imgs): a
                # pathogen lesion, NOT abiotic under-watering scorch — the bare
                # 'necrosis' keyword below would otherwise mislabel it into
                # leaf_scorch_underwatering. 'Marginal leaf necrosis' / 'Leaves
                # scorch' / 'Burn' still map to leaf_scorch correctly.
                'blossom', 'pear', 'tubercular')
DOCTORP_KEYWORDS = [
    ('grey_mold', 'gray_mold_botrytis'), ('gray_mold', 'gray_mold_botrytis'),
    ('botrytis', 'gray_mold_botrytis'),
    ('mealybug', 'pests'), ('mealy', 'pests'), ('scale', 'pests'),
    ('powdery', 'powdery_mildew'),
    ('nutrient', 'nutrient_deficiency'), ('chlorosis', 'nutrient_deficiency'),
    ('deficien', 'nutrient_deficiency'),
    ('mosaic', 'viral_disease'), ('virus', 'viral_disease'),
    ('bacterial_spot', 'leaf_spot'), ('leaf_spot', 'leaf_spot'),
    ('leafspot', 'leaf_spot'), ('septoria', 'leaf_spot'), ('anthracnose', 'leaf_spot'),
    ('burn', 'leaf_scorch_underwatering'), ('scorch', 'leaf_scorch_underwatering'),
    ('necrosis', 'leaf_scorch_underwatering'),
    ('healthy', 'healthy'),
]
def _doctorp_map_folder(folder_norm):
    if any(k in folder_norm for k in DOCTORP_DROP):
        return None
    for kw, cls in DOCTORP_KEYWORDS:
        if kw in folder_norm and cls in CLASSES:
            return cls
    return None

def fetch_doctorp_disease(slug=DOCTORP_SLUG, per_class_cap=400, label_confidence=0.8):
    """DoctorP app photos -> TRAIN + held-out eval (source 'doctorp', the only
    EVAL-ELIGIBLE Kaggle source). Maps each condition folder by keyword
    (dropping out-of-taxonomy / deferred conditions), draws a diverse
    cross-folder sample per class to per_class_cap, SHA-dedups against every
    other source, prints the folder mapping + per-class totals. Fail-soft."""
    root = _kaggle_download(slug)
    if root is None:
        return 0
    by_folder = {}
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_folder.setdefault(p.parent.name, []).append(p)
    if not by_folder:
        print(f'  Kaggle {slug}: no images found under {root}')
        return 0
    pools = {}
    print(f'  Kaggle {slug}: discovered folders (folder -> mapped class) ->')
    for folder, files in sorted(by_folder.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        norm = _pv_normalise(folder)
        mapped = _doctorp_map_folder(norm)
        if mapped is not None:
            print(f'    {folder!r:34s} n={len(files):5d} -> {mapped}')
            pools.setdefault(mapped, []).extend((f, norm) for f in files)
    saved_per_class = {}
    for cls, items in pools.items():
        (TRAIN_ROOT / cls).mkdir(parents=True, exist_ok=True)
        order = _stratified_folder_order(items)
        got = 0
        for idx in order:
            if got >= per_class_cap:
                break
            path, norm = items[idx]
            try:
                body = path.read_bytes()
            except Exception:
                continue
            got += _save_archive_member(
                cls, 'doctorp', f'doctorp_{norm}', path.name, body,
                label_confidence=label_confidence,
                species=f'plant (DoctorP {norm})',
                # DoctorP app photos are uniformly ~128px; the default 160px
                # floor rejected EVERY one (the source contributed 0). It's the
                # eval-eligible real-photo source, so accept its native size.
                min_side=120)
        saved_per_class[cls] = got
    total = sum(saved_per_class.values())
    print(f'  DoctorP per-class: {saved_per_class}')
    print(f'  DoctorP total: += {total}')
    return total


# ── PlantDoc (HF) → real-world train + val_realworld seed ───────
_PD_FUNGAL_KEYWORDS = (
    # Indoor-relevant leaf-spot morphologies. Rust, scab, and whole-plant
    # systemic patterns are intentionally OUT — pooling them with cercospora
    # / anthracnose / alternaria collapses recall on the actual indoor pattern.
    'leaf spot', 'mold', 'anthracnose', 'septoria', 'target spot',
    'early blight', 'late blight', 'leaf blight',
)
_PD_DROP_KEYWORDS = (
    'spider mite', 'mites',
    'rust',                     # very different morphology — drop
    'scab',                     # apple scab is more rust-like
    'black rot', 'fruit rot',   # fruit-stage disease, not leaf
)

def map_plantdoc_label(raw):
    s = re.sub(r'\s+', ' ', str(raw).strip().lower())
    if any(k in s for k in _PD_DROP_KEYWORDS):
        return None
    if 'bacterial spot' in s or 'bacterial blight' in s:
        return 'leaf_spot'
    if 'powdery mildew' in s:
        return 'powdery_mildew'
    if 'yellow virus' in s or 'mosaic virus' in s or 'curl virus' in s:
        return 'viral_disease'
    if any(k in s for k in _PD_FUNGAL_KEYWORDS):
        return 'leaf_spot'
    if 'leaf' in s:
        return 'healthy'
    return None

def _clip_bbox_xywh(bbox, w, h, pad=0.06):
    """Clip an XYWH (COCO-style) bbox to image bounds with padding.

    Strictly assumes XYWH (the HF card for `agyaatcoder/PlantDoc` confirms
    XYWH). Reject bboxes whose bottom-right corner falls outside the image:
    in valid XYWH `x + bw ≤ w` always holds, so an XYXY tuple sneaking
    through past the image centre will overflow this check.
    """
    x, y, bw, bh = [float(v) for v in bbox]
    if bw <= 1 or bh <= 1 or w <= 1 or h <= 1:
        return None
    # 5 % slack absorbs off-by-one annotations; anything past that is
    # either a real out-of-bounds bbox or XYXY misread as XYWH.
    if (x + bw) > w * 1.05 or (y + bh) > h * 1.05:
        return None
    if bw > w or bh > h:
        return None
    px, py = bw * pad, bh * pad
    x0, y0 = max(0, int(round(x - px))), max(0, int(round(y - py)))
    x1, y1 = min(w, int(round(x + bw + px))), min(h, int(round(y + bh + py)))
    if x1 - x0 < 32 or y1 - y0 < 32:
        return None
    return x0, y0, x1, y1

def _plantdoc_image_label(row, ds_split):
    """Resolve a PlantDoc image-level class label as a string, used as a
    fallback when the HF revision ships image-level labels instead of
    bbox-level ``objects``."""
    if 'label' in row:
        feat = ds_split.features.get('label') if hasattr(ds_split, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

def fetch_plantdoc(max_train_per_class=500, max_val_per_class=120):
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as e:
        print(f'PlantDoc HF download failed after retries: {e}')
        print('  HuggingFace 500/503 on /api/datasets/.../tree/... resolves in '
              'a few minutes — re-run this cell shortly.')
        return
    counts, dropped, no_obj_fallback = {}, 0, 0
    for split in [s for s in ('train', 'test') if s in ds]:
        out_base = VAL_PRIMARY if split == 'test' else TRAIN_ROOT
        cap = max_val_per_class if split == 'test' else max_train_per_class
        # HF returns rows clustered by class — shuffle so the cap samples
        # a wider mix.
        order = list(range(len(ds[split])))
        random.Random(SEED + (10 if split == 'test' else 11)).shuffle(order)
        for i in tqdm(order, desc=f'PlantDoc/{split}'):
            row = ds[split][i]
            img = row['image'].convert('RGB')
            w, h = img.size
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            cats = objects.get('category') or []
            # Object-level path (preferred — HF revisions ≥ 1.x).
            wrote_any = False
            for j, (bbox, cat) in enumerate(zip(bboxes, cats)):
                target = map_plantdoc_label(cat)
                if target is None or target not in CLASSES:
                    dropped += 1; continue
                key = (split, target)
                if counts.get(key, 0) >= cap:
                    continue
                box = _clip_bbox_xywh(bbox, w, h)
                if box is None: dropped += 1; continue
                crop = img.crop(box)
                image_id = row.get('image_id', i)
                stem = f'plantdoc_{split}_{image_id}_{j}'
                out = out_base / target / f'{stem}.jpg'
                if out.exists(): wrote_any = True; continue
                _save_rgb(crop, out)
                write_sidecar(out, CropAnnotation(
                    plant_id=f'plantdoc_{split}_{image_id}',
                    species=str(cat).split(' leaf')[0],
                    diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                    capture_session=f'plantdoc_{split}', source='plantdoc',
                    label_confidence=0.85, bbox=[int(v) for v in bbox]))
                counts[key] = counts.get(key, 0) + 1
                wrote_any = True
            if wrote_any:
                continue
            # Image-level fallback. Triggered when the HF revision drops
            # object-detection annotations and ships one category per image.
            raw = _plantdoc_image_label(row, ds[split])
            target = map_plantdoc_label(raw) if raw is not None else None
            if target is None or target not in CLASSES:
                dropped += 1; continue
            key = (split, target)
            if counts.get(key, 0) >= cap:
                continue
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split}_{image_id}_full'
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            _save_rgb(img, out)
            write_sidecar(out, CropAnnotation(
                plant_id=f'plantdoc_{split}_{image_id}',
                species=str(raw).split(' leaf')[0] if raw else None,
                diagnosis=target, severity='unknown', leaf_or_whole_plant='whole_plant',
                capture_session=f'plantdoc_{split}', source='plantdoc',
                label_confidence=0.75, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
            no_obj_fallback += 1
    print(f'PlantDoc counts: {counts}\nPlantDoc dropped: {dropped}'
          f'  image-level fallbacks: {no_obj_fallback}')

# ── PlantVillage (HF) → capped train + val_studio ───────────────
def _pv_normalise(s):
    return re.sub(r'[^a-z0-9_]+', '_', str(s).strip().lower()).strip('_')

PV_LABEL_MAP_RAW = {
    'Pepper,_bell___Bacterial_spot': 'leaf_spot',
    'Pepper__bell___Bacterial_spot': 'leaf_spot',
    'Tomato___Bacterial_spot':       'leaf_spot',
    'Peach___Bacterial_spot':        'leaf_spot',
    'Apple___healthy': 'healthy', 'Blueberry___healthy': 'healthy',
    'Cherry_(including_sour)___healthy': 'healthy', 'Corn_(maize)___healthy': 'healthy',
    'Grape___healthy': 'healthy',
    'Pepper,_bell___healthy': 'healthy', 'Pepper__bell___healthy': 'healthy',
    'Potato___healthy': 'healthy', 'Raspberry___healthy': 'healthy',
    'Soybean___healthy': 'healthy', 'Strawberry___healthy': 'healthy',
    'Tomato___healthy': 'healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'leaf_spot',
    'Tomato___Septoria_leaf_spot': 'leaf_spot',
    'Tomato___Target_Spot':        'leaf_spot',
    'Tomato___Leaf_Mold':          'leaf_spot',
    'Tomato___Early_blight':       'leaf_spot',
    'Tomato___Late_blight':        'leaf_spot',
    'Potato___Early_blight':       'leaf_spot',
    'Potato___Late_blight':        'leaf_spot',
    # Apple scab is rust-family morphology; keep OUT of leaf_spot.
    'Apple___Apple_scab':          None,
    # Apple black rot is a fruit disease — many images are whole-fruit.
    'Apple___Black_rot':           None,
    # Rust morphology is dropped wholesale.
    'Apple___Cedar_apple_rust':    None,
    'Corn_(maize)___Common_rust_': None,
    'Corn_(maize)___Northern_Leaf_Blight': 'leaf_spot',
    'Grape___Black_rot':           None,                  # fruit disease
    'Grape___Esca_(Black_Measles)':'leaf_spot',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'leaf_spot',
    'Orange___Haunglongbing_(Citrus_greening)':   None,   # HLB ≠ leaf-spot
    'Cherry_(including_sour)___Powdery_mildew': 'powdery_mildew',
    'Squash___Powdery_mildew':                  'powdery_mildew',
    'Tomato___Tomato_mosaic_virus':           'viral_disease',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'viral_disease',
    'Tomato___Spider_mites Two-spotted_spider_mite': None,
    # Strawberry leaf scorch is physiological / abiotic, not fungal.
    'Strawberry___Leaf_scorch':  None,
    'Background_without_leaves': 'not_a_plant',
}
PV_LABEL_MAP = {_pv_normalise(k): v for k, v in PV_LABEL_MAP_RAW.items()}
# Each candidate is (hf_repo, hf_config_name_or_None, splits_tuple). The
# config name is required for mohanty/PlantVillage (ships three configs;
# load_dataset() refuses to pick one). BrandonFors has a single default
# config, so explicit name is None there.
PV_DATASET_CANDIDATES = [
    ('BrandonFors/Plant-Diseases-PlantVillage-Dataset', None,    ('train', 'test')),
    ('mohanty/PlantVillage',                            'color', ('train', 'test')),
]

def _hf_load_with_retry(name, config_name=None, *, split=None, retries=3, base_delay=5.0):
    """`datasets.load_dataset` with exponential-backoff retry on transient
    HuggingFace 5xx errors."""
    from datasets import load_dataset
    last = None
    delay = base_delay
    for attempt in range(retries):
        try:
            return load_dataset(name, config_name, split=split, token=False)
        except Exception as e:
            last = e
            msg = str(e)
            transient = (
                '500' in msg or '502' in msg or '503' in msg or '504' in msg
                or 'temporarily unavailable' in msg.lower()
                or 'connection' in msg.lower() or 'timeout' in msg.lower()
            )
            if attempt + 1 == retries or not transient:
                raise
            print(f'  HF {name} attempt {attempt + 1}/{retries} failed ({type(e).__name__}); '
                  f'retrying in {delay:.0f}s…')
            time.sleep(delay)
            delay *= 3
    if last is not None:
        raise last
    raise RuntimeError(f'_hf_load_with_retry exhausted retries for {name}')

def _load_first_available_hf_dataset(candidates):
    errors = []
    for entry in candidates:
        if len(entry) == 2:
            name, splits = entry; config_name = None
        else:
            name, config_name, splits = entry
        try:
            out = {}
            for split in [s for s in splits if s]:
                out[split] = _hf_load_with_retry(name, config_name, split=split)
            label = name + (f' [{config_name}]' if config_name else '')
            print(f'PlantVillage source: {label}')
            return name, out
        except Exception as e:
            errors.append(f'{name}: {e}')
    raise RuntimeError('All PlantVillage HF downloads failed:\n' + '\n'.join(errors))

def _row_label(row, ds):
    if 'label' in row:
        feat = ds.features.get('label') if hasattr(ds, 'features') else None
        if hasattr(feat, 'names') and feat.names:
            return feat.names[int(row['label'])]
        return row.get('label_name') or str(row['label'])
    return row.get('label_name') or row.get('class') or row.get('disease')

# PV `healthy` is uniform-bg / agar-plate; iNat `healthy` is real foliage.
# Cap PV-only contribution low so iNat dominates healthy.
PV_PER_CLASS_TRAIN_OVERRIDE = {'healthy': 250}

def fetch_plantvillage_into_train_and_studio(max_train_per_class=900, max_studio_per_class=300, studio_hash_frac=0.15):
    try:
        name, dsets = _load_first_available_hf_dataset(PV_DATASET_CANDIDATES)
    except Exception as e:
        print(f'PlantVillage HF download failed: {e}'); return
    # When the candidate ships a real `test` split, use IT as the canonical
    # val_studio source so train rows always go to TRAIN_ROOT.
    test_present = 'test' in dsets
    # Skip image DECODE for the ~90% of rows we drop or cap: read labels off
    # the undecoded column and decode only the few we actually save. Falls back
    # to normal decoding if the cast isn't applicable.
    try:
        from datasets import Image as _HFImage
        for _k in list(dsets):
            if 'image' in dsets[_k].features and isinstance(dsets[_k].features['image'], _HFImage):
                dsets[_k] = dsets[_k].cast_column('image', _HFImage(decode=False))
    except Exception as _e:  # noqa: BLE001 — degrade to full decode
        print(f'  PV: decode-skip unavailable ({type(_e).__name__}); decoding all rows')
    counts, dropped = {}, 0
    for split, ds in dsets.items():
        order = list(range(len(ds)))
        random.Random(SEED + (0 if split == 'train' else 1)).shuffle(order)
        for i in tqdm(order, desc=f'PlantVillage/{split}'):
            row = ds[i]
            raw = _row_label(row, ds)
            norm = _pv_normalise(raw)
            target = PV_LABEL_MAP.get(norm)
            if target is None or target not in CLASSES:
                dropped += 1; continue
            _imgcol = row['image']
            filename = (row.get('image/filename') or row.get('filename')
                        or (_imgcol.get('path') if isinstance(_imgcol, dict) else None)
                        or f'{split}_{i}')
            stem = f'pv_{_pv_normalise(filename)}_{i}'
            if test_present:
                use_studio = (split == 'test')
            else:
                use_studio = (_stable_frac(stem) < studio_hash_frac)
            out_base = VAL_STUDIO if use_studio else TRAIN_ROOT
            if use_studio:
                cap = max_studio_per_class
            else:
                cap = PV_PER_CLASS_TRAIN_OVERRIDE.get(target, max_train_per_class)
            key = ('studio' if use_studio else 'train', target)
            if counts.get(key, 0) >= cap:
                continue
            out = out_base / target / f'{stem}.jpg'
            if out.exists(): continue
            img = _imgcol
            if isinstance(img, dict):          # decode=False -> {'bytes','path'}
                img = (PILImage.open(BytesIO(img['bytes'])) if img.get('bytes')
                       else PILImage.open(img['path']))
            elif not hasattr(img, 'convert'):
                img = PILImage.open(img)
            _save_rgb(img.convert('RGB'), out)
            write_sidecar(out, CropAnnotation(
                plant_id=stem, species=str(raw).split('___')[0].replace('_', ' '),
                diagnosis=target, severity='unknown', leaf_or_whole_plant='leaf',
                capture_session=f'plantvillage_{split}', source='plantvillage',
                label_confidence=0.9, bbox=[0, 0, 0, 0]))
            counts[key] = counts.get(key, 0) + 1
    print(f'PV counts: {counts}\nPV dropped/ignored: {dropped}')

# ── Group-aware splits ──────────────────────────────────────────
def make_group_split(dest_root, ratio, seed, label, allowed_sources=None):
    """Move a fraction of TRAIN_ROOT into dest_root.

    Every `(source, plant_id)` group lands together IN ONE SPLIT — including
    when its images span MULTIPLE classes. PlantDoc is a detection set whose
    crops from a single photo can map to different diseases (one box
    leaf_spot, another healthy); they share a plant_id, so deciding per-class
    independently would leak the same photo across train and a held-out split
    (exactly what the §2 leakage guard caught). We therefore collect groups
    GLOBALLY across classes, choose per-class by `ratio`, and move EVERY file
    of a chosen group regardless of which class dir it sits in.

    `allowed_sources` constrains which sources can populate the split — used
    to keep PlantVillage out of the real-world / calibration splits.
    """
    rng = random.Random(seed)
    # Pass 1 — collect every group's files across ALL classes.
    group_files = {}                 # (source, pid) -> [(cls, img_path), ...]
    per_class_ids = {}               # cls -> [ (source, pid), ... ] (unique)
    for cls in CLASSES:
        cls_dir = TRAIN_ROOT / cls
        if not cls_dir.exists():
            continue
        seen_in_cls = set()
        for img in iter_images(cls_dir):
            sc = read_sidecar(img) or {}
            source = sc.get('source', 'unknown')
            if allowed_sources is not None and source not in allowed_sources:
                continue
            g = (source, sc.get('plant_id', img.stem))
            group_files.setdefault(g, []).append((cls, img))
            if g not in seen_in_cls:
                per_class_ids.setdefault(cls, []).append(g)
                seen_in_cls.add(g)
    # Pass 2 — choose groups per class by `ratio`. A group already chosen via
    # another class counts toward this class's quota (it's one shared photo),
    # so cross-class groups never double-draw and always co-locate.
    chosen = set()
    for cls in CLASSES:
        ids = list(per_class_ids.get(cls, []))
        if not ids:
            continue
        rng.shuffle(ids)
        k = int(len(ids) * ratio)
        if ratio > 0:
            k = max(1, k)
        if k <= 0:
            continue
        taken = [g for g in ids if g in chosen]          # already destined here
        for g in ids:
            if len(taken) >= k:
                break
            if g not in chosen:
                taken.append(g)
        chosen.update(taken)
    # Pass 3 — move every file of every chosen group (across all its classes).
    moved = 0
    for g in chosen:
        for cls, img in group_files[g]:
            out = dest_root / cls
            out.mkdir(parents=True, exist_ok=True)
            dst = out / img.name
            if dst.exists():
                raise RuntimeError(f'Destination already exists: {dst}')
            shutil.move(str(img), str(dst))
            sc_path = img.with_suffix('.json')
            if sc_path.exists():
                shutil.move(str(sc_path), str(dst.with_suffix('.json')))
            moved += 1
    print(f'{label} populated  (moved={moved})')

def _sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def _phash_file(p: Path):
    import imagehash
    try:
        return imagehash.phash(PILImage.open(p).convert('RGB'))
    except Exception:
        return None

def _dedup_train_in_place(root: Path, phash_threshold: int = 6) -> int:
    """Per-class SHA-256 + pHash dedup on TRAIN_ROOT before the splitters
    run. PIL.open + hashing both release the GIL, so we parallelise the
    per-file work and keep the per-class hash-set membership check serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists():
            continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas   = list(pool.map(_sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_ph = set(), []
        for p, sha, ph in zip(paths, shas, phashes):
            if sha in seen_sha:
                drop.append(p); continue
            seen_sha.add(sha)
            if ph is None:
                drop.append(p); continue
            if any(ph - existing <= phash_threshold for existing in seen_ph):
                drop.append(p); continue
            seen_ph.append(ph)
    for p in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except OSError:
            pass
    print(f'pre-split dedup({root}): dropped={len(drop)}')
    return len(drop)

def make_val_realworld(ratio=0.15, seed=2):
    # Physiological-class sources join the realworld pool so the ship-gate
    # per-class recall check sees real rows. Every NAP source is included
    # so NAP recall is measured against indoor clutter, not just COCO.
    make_group_split(VAL_PRIMARY, ratio, seed, 'val_realworld',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp'})
def make_val_indomain(ratio=0.10, seed=0):
    make_group_split(VAL_INDOMAIN, ratio, seed, 'val_indomain',
                     allowed_sources={'inat', 'coco', 'plantdoc',
                                      'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp'})
def make_temperature_cal(ratio=0.12, seed=3):
    # All trained classes need calibration samples. Include every source
    # that feeds a trained class (NAP + physiological).
    make_group_split(TEMP_CAL, ratio, seed, 'temperature_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp'})
def make_conformal_cal(ratio=0.15, seed=1):
    make_group_split(CONFORMAL_CAL, ratio, seed, 'conformal_cal',
                     allowed_sources={'plantdoc', 'inat',
                                      'coco', 'inat_animalia',
                                      'openimages_indoor',
                                      'coco_hard_partial_plant',
                                      'inat_hard_greentexture', 'inat_pest', 'doctorp'})

# ── NAP hard negatives (leaf-like-but-not-leaf / partial plants /
#    blurry green textures) ─────────────────────────────────────────
# The easy COCO / Animalia / Open-Images negatives sit far from the plant
# MaxLogit distribution, so plant and NAP MaxLogit overlap and the §6 OOD
# gate auto-disables (negative separation). These two fetchers add the HARD
# negatives that actually overlap real leaf close-ups — green, textured,
# leaf-adjacent frames — which is what flips separation positive and lets
# the MaxLogit floor switch on.
def fetch_coco_hard_negatives(target_n=600, max_plant_area_frac=0.25):
    """COCO indoor frames that DO contain a potted plant / vase but only as
    a small / partial element (summed plant bbox area < max_plant_area_frac
    of the image). These 'partial plant in clutter' shots are exactly what
    fetch_coco_for_nap deliberately excludes — and exactly what the OOD gate
    confuses with a real leaf. Reuses the COCO cache fetch_coco_for_nap
    already downloaded."""
    ann_path = COCO_CACHE / 'annotations' / 'instances_val2017.json'
    img_dir = COCO_CACHE / 'val2017'
    if not ann_path.exists() or not img_dir.exists():
        print('  COCO cache missing (run fetch_coco_for_nap first) — skipping hard negatives')
        return
    ann = json.loads(ann_path.read_text())
    plant_cat = {c['id'] for c in ann['categories'] if c['name'] in {'potted plant', 'vase'}}
    meta = {im['id']: (im['width'], im['height'], im['file_name']) for im in ann['images']}
    frac = {}
    for a in ann['annotations']:
        if a['category_id'] not in plant_cat:
            continue
        iw, ih, _ = meta.get(a['image_id'], (0, 0, None))
        if not iw or not ih:
            continue
        bw, bh = a['bbox'][2], a['bbox'][3]
        frac[a['image_id']] = frac.get(a['image_id'], 0.0) + (bw * bh) / float(iw * ih)
    eligible = [iid for iid, f in frac.items() if 0.0 < f < max_plant_area_frac]
    rng = random.Random(SEED + 23); rng.shuffle(eligible)
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    n = 0
    for iid in eligible:
        if n >= target_n:
            break
        _iw, _ih, fname = meta[iid]
        src, dst = img_dir / fname, dest / f'coco_hardneg_{iid}.jpg'
        if not src.exists() or dst.exists():
            continue
        shutil.copy(src, dst)
        write_sidecar(dst, CropAnnotation(
            plant_id=f'coco_hardneg_{iid}', species=None, diagnosis='not_a_plant',
            severity='low', leaf_or_whole_plant='whole_plant',
            capture_session='coco_hard_partial_plant', source='coco_hard_partial_plant',
            label_confidence=0.95, bbox=[0, 0, 0, 0]))
        n += 1
    print(f'COCO hard negatives (partial plant): not_a_plant += {n}')

# Green, leaf-like textures that are NOT a houseplant leaf and are
# taxonomically disjoint from every disease / healthy genus: grasses,
# mosses, liverworts, lichens, green algae. These overlap the plant
# MaxLogit distribution far more than animals / furniture do.
INAT_HARD_NEGATIVE_TAXA = [
    ('Poaceae', 'family'),          # grasses / lawn blades
    ('Bryophyta', 'phylum'),        # mosses
    ('Marchantiophyta', 'phylum'),  # liverworts
    ('Lecanoromycetes', 'class'),   # lichens
    ('Chlorophyta', 'phylum'),      # green algae
]
def fetch_inat_hard_negatives(per_taxon=200, target_n=900):
    """iNat research-grade green-texture hard negatives -> not_a_plant."""
    dest = TRAIN_ROOT / 'not_a_plant'; dest.mkdir(parents=True, exist_ok=True)
    saved = 0
    for name, rank in INAT_HARD_NEGATIVE_TAXA:
        if saved >= target_n:
            break
        tid = inat_resolve_taxon(name, rank)
        if not tid:
            print(f'  iNat hard-neg taxon {name!r} not resolvable; skipping')
            continue
        obs = inat_observations(tid, per_taxon, leaf_only=False)
        jobs = []
        for ob in obs:
            for ph in (ob.get('photos') or [])[:1]:
                url = _photo_url(ph, 'medium')
                if not url:
                    continue
                fp = dest / f'inat_hardneg_{ob["id"]}_{ph["id"]}.jpg'
                obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                ann = lambda obs_id=obs_id, taxon_name=taxon_name, name=name: CropAnnotation(
                    plant_id=f'inat_hardneg_{obs_id}', species=taxon_name,
                    diagnosis='not_a_plant', severity='low',
                    leaf_or_whole_plant='whole_plant',
                    capture_session=f'inat_hardneg_{name}',
                    source='inat_hard_greentexture', label_confidence=0.97,
                    bbox=[0, 0, 0, 0])
                jobs.append((url, fp, ann))
        n = 0
        pool = ThreadPoolExecutor(max_workers=8)
        try:
            futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
            for fut in as_completed(futs):
                r = fut.result(); n += r; saved += r
                if saved >= target_n:
                    break
        finally:
            pool.shutdown(wait=False, cancel_futures=True)
        print(f'  iNat hard-neg {name}: += {n}')
    print(f'iNat hard negatives (green texture): not_a_plant += {saved}')

# ── iNat → pest classes (mealybug / scale) ───────────────
# The two most visually-separable houseplant pests: mealybugs
# (Pseudococcidae — cottony white masses) and scale (Diaspididae /
# Coccidae — waxy / armoured bumps). Research-grade observations are the
# insect ON a plant, so leaf_only=False and the green-content gate is NOT
# applied (the pest mass, not foliage green, is the signal). Tagged
# source=\'inat_pest\' (in-domain weight) so the sampler treats them as real
# houseplant rows, not crop supplements (full source weight in §4).
INAT_PEST_TAXA = {
    # Single combined 'pests' class — pull from every common houseplant
    # mealybug + scale family into one label: mealybugs = Pseudococcidae;
    # armoured scale = Diaspididae; soft scale = Coccidae; felt scale =
    # Eriococcidae. Broad family coverage makes 'pests' a strong,
    # well-supplied class rather than a thin one.
    'pests': [('Pseudococcidae', 'family'), ('Diaspididae', 'family'),
              ('Coccidae', 'family'), ('Eriococcidae', 'family')],
}
def fetch_inat_pests(per_taxon=800, target_per_class=900):
    for cls, taxa in INAT_PEST_TAXA.items():
        dest = TRAIN_ROOT / cls; dest.mkdir(parents=True, exist_ok=True)
        saved = 0
        for name, rank in taxa:
            if saved >= target_per_class:
                break
            tid = inat_resolve_taxon(name, rank)
            if not tid:
                print(f'  iNat pest taxon {name!r} not resolvable; skipping'); continue
            obs = inat_observations(tid, per_taxon, leaf_only=False)
            jobs = []
            for ob in obs:
                for ph in (ob.get('photos') or [])[:1]:
                    url = _photo_url(ph, 'medium')
                    if not url:
                        continue
                    fp = dest / f'inat_pest_{cls}_{ob["id"]}_{ph["id"]}.jpg'
                    obs_id, taxon_name = ob['id'], (ob.get('taxon') or {}).get('name')
                    ann = lambda obs_id=obs_id, taxon_name=taxon_name, name=name, cls=cls: CropAnnotation(
                        plant_id=f'inat_pest_{obs_id}', species=taxon_name,
                        diagnosis=cls, severity='medium', leaf_or_whole_plant='leaf',
                        capture_session=f'inat_pest_{name}', source='inat_pest',
                        label_confidence=0.7, bbox=[0, 0, 0, 0])
                    jobs.append((url, fp, ann))
            pool = ThreadPoolExecutor(max_workers=8)
            try:
                futs = [pool.submit(_fetch_save_remote, u, f, a) for u, f, a in jobs]
                for fut in as_completed(futs):
                    saved += fut.result()
            finally:
                pool.shutdown(wait=False, cancel_futures=True)
        print(f'iNat pest ({cls}): += {saved}')


# ── Run the pipeline ────────────────────────────────────────────
def _try_fetch(label, fn, *args, **kwargs):
    """Run a fetcher, log + continue on any exception. Upstream availability
    is the dominant failure mode for §2; a single transient failure after
    the slow fetches wastes ten minutes of work."""
    try:
        return fn(*args, **kwargs)
    except KeyboardInterrupt:
        raise
    except Exception as exc:  # noqa: BLE001 — by design; see above
        print(f'{label} skipped: {type(exc).__name__}: {exc}')
        log_run({'phase': 'fetch_skipped', 'label': label,
                 'error_type': type(exc).__name__, 'error': str(exc)})
        return None

_try_fetch('inat_supplement', fetch_inat_supplement,
           healthy_per_taxon=180, botrytis_per_taxon=2400)
_try_fetch('coco_nap', fetch_coco_for_nap, target_n=1500)
_try_fetch('inat_animalia_nap', fetch_inat_animalia_for_nap, target_n=500)
_try_fetch('openimages_indoor', fetch_openimages_indoor, target_n=600)
_try_fetch('coco_hard_negatives', fetch_coco_hard_negatives, target_n=600)
_try_fetch('inat_hard_negatives', fetch_inat_hard_negatives, per_taxon=200, target_n=900)
_try_fetch('plantdoc', fetch_plantdoc,
           max_train_per_class=500, max_val_per_class=120)
_try_fetch('plantvillage', fetch_plantvillage_into_train_and_studio,
           max_train_per_class=900, max_studio_per_class=300)
if INCLUDE_PHYSIOLOGICAL_CLASSES:
    # Stage 0: higher target so the broadened query set is fully drawn.
    _try_fetch('inat_physiological', fetch_inat_physiological, per_class=1500)
    if INCLUDE_ABIOTIC_SUPPLEMENTS:
        # CoLeaf-DB (Mendeley) is intentionally NOT fetched: Mendeley/Cloudflare
        # 403s datacenter IPs (incl. Colab), so it always failed and only added
        # ~4 dead download attempts (with backoff) per run. nutrient_deficiency's
        # crop signal comes from MPLD + the Kaggle nutrient/ornamental sets +
        # DoctorP. To restore CoLeaf, mirror DOI 10.17632/brfgw46wzb.1 to a
        # Kaggle dataset and add it to the Kaggle supplement.
        # MPLD (maize P-deficiency, Zenodo) downloads fine. Capped LOW on
        # purpose: a crop-domain representation aid for the deficiency *signal*,
        # not in-domain samples — kept comparable to the other thin classes so
        # it doesn't inflate nutrient_deficiency's logit-adjustment prior.
        _try_fetch('mpld_phosphorus_deficiency', fetch_mpld_phosphorus_deficiency,
                   cap=150)

if INCLUDE_KAGGLE_SUPPLEMENTS:
    # Operator opt-in; fail-soft without Kaggle creds. Runs BEFORE the
    # pre-split dedup so its rows are deduped + train-scoped like the rest.
    _try_fetch('kaggle_houseplant', fetch_kaggle_houseplant_disease,
               per_class_cap=250)
    # Crop-domain Botrytis proxy to lift the supply-bound gray_mold class.
    _try_fetch('kaggle_strawberry_graymold', fetch_kaggle_strawberry_graymold,
               cap=250)
    # Colab-reachable nutrient-deficiency source (replaces the IP-blocked
    # Mendeley CoLeaf; adds N/K diversity to MPLD's maize-P).
    _try_fetch('kaggle_nutrient_deficiency', fetch_kaggle_nutrient_deficiency,
               cap=200)
    # v2.5.1: melon BROADLEAF nutrient proxy — dicot leaf morphology transfers
    # to houseplants better than the maize/rice grass-blade proxies (verified
    # clean, non-segmented). TRAIN-ONLY, down-weighted + prior-decoupled.
    _try_fetch('kaggle_melon_nutrient', fetch_kaggle_melon_nutrient, cap=150)
    # Curated real-houseplant species photos -> healthy (variegation diversity).
    # SKIPPED by default: it's a 4.85 GB Kaggle download — the single biggest
    # time sink in this cell (~5 min, bandwidth-bound) — for only ~400 images of
    # the ALREADY-abundant `healthy` class. iNat (Ficus/Calathea/Sansevieria/…)
    # + PlantVillage + ornamental already supply healthy variegation. Flip to
    # True to re-enable.
    INCLUDE_KAGGLE_HEALTHY_SPECIES = False
    if INCLUDE_KAGGLE_HEALTHY_SPECIES:
        _try_fetch('kaggle_houseplant_species', fetch_kaggle_houseplant_species,
                   cap=400)
    # v2.4: ornamental/houseplant disease set (rose/hibiscus/money-plant) —
    # real disease morphology on natural backgrounds for the weak classes.
    _try_fetch('kaggle_ornamental', fetch_kaggle_ornamental_disease,
               per_class_cap=300)
    # v2.4: DoctorP — REAL plant-doctor app photos. Supplies the starved
    # classes (gray mold / mealybug / scale / nutrient deficiency) in their
    # deployment framing AND is eval-eligible (seeds the real-photo eval slice).
    _try_fetch('doctorp', fetch_doctorp_disease, per_class_cap=400)

# 'pests' is a first-class trainable class — always fetch its supply.
_try_fetch('inat_pests', fetch_inat_pests, per_taxon=800, target_per_class=900)

# Dedup BEFORE splitting. The previous order — split first, dedup TRAIN_ROOT
# in §3 — left near-duplicates straddling train and held-out splits because
# make_group_split moves bytes by (source, plant_id) and only a SHA-256
# audit catches identical bytes (not pHash near-dupes).
_dedup_train_in_place(TRAIN_ROOT, phash_threshold=6)

make_val_realworld(); make_val_indomain(); make_temperature_cal(); make_conformal_cal()

def report_class_counts(root):
    print(f'\n{root}:')
    for c in CLASSES:
        print(f'  {c:25s} {count_images(root/c):5d}')
for r in (TRAIN_ROOT, VAL_PRIMARY, VAL_INDOMAIN, TEMP_CAL, CONFORMAL_CAL, VAL_STUDIO):
    report_class_counts(r)

# Per-class minimum training count. Most classes share the 150 default —
# the smallest count where KD distillation has produced trainable signal
# in practice for this taxonomy. ``gray_mold_botrytis`` is supplied by
# iNat alone (PlantDoc has no Botrytis label, PV's standard release has
# no Botrytis label) and ships chronically thin. Measured May 2026: the
# full Botrytis taxa set yields ~138 foliage images after the
# green-content gate, of which ~81 land in train after the four-way
# split drainage — that is the genuine iNat-only ceiling, not a
# starvation symptom. The 70 floor sits ~11 rows below it so a normal
# run passes while a real iNat outage (ConnectTimeout truncation,
# rate-limit wall) still fails loud. The §6 ship gate — disease
# macro-F1 ≥ 0.70 with no zero-recall class — is the real quality
# backstop for an under-supplied class. Openverse used to top this
# class up; removed in v2.2 because its anonymous tier was rate-capped
# below the notebook's burst rate.
THRESH_TRAIN_DEFAULT = 150
THRESH_TRAIN_PER_CLASS = {
    # v2.4: gray_mold is no longer iNat-only. It is now fed by (a) iNat
    # research-grade botrytis at 3 photos/obs (~1.6x the old ~138 ceiling),
    # (b) DoctorP real app photos (Grey mold + Botrytis cinerea, ~96), and
    # (c) the Kaggle strawberry crop proxy (down-weighted). Floor kept low
    # so a partial-source outage still trains, but the realistic train
    # count is several hundred; the ship-gate per-class recall CI on
    # val_realworld (now seeded with REAL DoctorP gray-mold rows) is the
    # honest quality backstop for this structurally-hardest class.
    'gray_mold_botrytis': 70,
    # 60 is the floor at which source-weighted sampling produces trainable
    # signal for thin iNat-free-text classes; nutrient_deficiency is now
    # additionally supplemented by the curated CoLeaf set (train-only).
    'leaf_scorch_underwatering': 90,
    'nutrient_deficiency': 90,
    # 'pests' is a strong, broadly-supplied class — it uses the
    # THRESH_TRAIN_DEFAULT (150) floor like the other real classes.
}
THRESH_VAL = 5

def _train_floor_for(cls):
    return THRESH_TRAIN_PER_CLASS.get(cls, THRESH_TRAIN_DEFAULT)

under_train = [c for c in CLASSES if count_images(TRAIN_ROOT/c) < _train_floor_for(c)]
under_val   = [c for c in CLASSES if count_images(VAL_PRIMARY/c) < THRESH_VAL]
if under_train or under_val:
    msg = []
    if under_train:
        floors = {c: _train_floor_for(c) for c in under_train}
        msg.append(f'TRAIN_ROOT under per-class floor: {floors}')
    if under_val:   msg.append(f'VAL_PRIMARY < {THRESH_VAL}: {under_val}')
    raise RuntimeError('\n'.join(msg) +
        '\nTRAIN_ROOT < threshold means an upstream source for that class is '
        'starved. For gray_mold_botrytis the supply chain is iNat alone '
        '(PlantDoc has no Botrytis label); other classes pull from '
        'PlantDoc + PV + iNat. Check the per-source log lines above for '
        'download failures or rate-limit backoffs, then re-run §2 or '
        'drop additional real-world photos into TRAIN_ROOT/<class>/.')

# ── Leakage guard + per-source coverage report (v2.5) ───────────
# Group-aware splits MOVE each (source, plant_id) group wholesale out of a
# shrinking TRAIN_ROOT, so by construction no group can span two splits. This
# is the runtime assertion that PROVES it for this run: a future refactor that
# accidentally copies instead of moves, or splits before dedup, trips here
# instead of silently inflating eval scores.
_SPLIT_ROOTS = {'train': TRAIN_ROOT, 'val_realworld': VAL_PRIMARY,
                'val_indomain': VAL_INDOMAIN, 'temperature_cal': TEMP_CAL,
                'conformal_cal': CONFORMAL_CAL}
_group_to_splits, _per_source = {}, {}
for _split, _root in _SPLIT_ROOTS.items():
    for _cls in CLASSES:
        _d = _root / _cls
        if not _d.exists():
            continue
        for _img in iter_images(_d):
            _sc = read_sidecar(_img) or {}
            _src = _sc.get('source', 'unknown')
            _gid = (_src, _sc.get('plant_id', _img.stem))
            _group_to_splits.setdefault(_gid, set()).add(_split)
            _per_source[_src] = _per_source.get(_src, 0) + 1
_leaked = {g: sorted(v) for g, v in _group_to_splits.items() if len(v) > 1}
if _leaked:
    _ex = list(_leaked.items())[:5]
    raise RuntimeError(
        f'DATA LEAKAGE: {len(_leaked)} (source, plant_id) group(s) span multiple '
        f'splits - e.g. {_ex}. Splits must be disjoint by group; verify dedup ran '
        'BEFORE make_*_split and that splits MOVE (not copy) files.')
print('leakage guard OK - every (source, plant_id) group lands in exactly one split')
print('per-source image counts:', dict(sorted(_per_source.items(), key=lambda kv: -kv[1])))


## §3 — Dedup + Stage 1 detector + DINOv2 diagnostic probe

Three sanity passes before we burn GPU on the teacher:

1. **Dedup** the train set (per-class SHA-256 + pHash with Hamming ≤ 6).
   Cross-class similarities are kept on purpose — they are useful as
   hard negatives. Held-out splits are audited (SHA-256 only) but
   never pruned, because pruning a held-out set after the fact biases
   the gate.

2. **Stage 1 YOLOv8n** is re-trained from PlantDoc bboxes **by default**
   (`TRAIN_DETECTOR_FROM_SCRATCH = True`): `find_detector_yaml()` first
   looks for a user-supplied YOLO `data.yaml` under `data_v2/`, and when
   none exists `build_yolo_detection_dataset_from_plantdoc()` generates one
   from the same PlantDoc bboxes used for the classifier crops. Set
   `TRAIN_DETECTOR_FROM_SCRATCH = False` to skip Stage 1 and keep the
   shipped `detector.{pt,onnx}` pair instead.

3. **DINOv2-S linear probe** is the cheap "is the data even
   separable?" check. If linear-on-frozen-DINOv2 cannot get past 30 %
   on `val_realworld` after 8 epochs, the data is the problem, not
   the model.


In [ ]:
# §3 — Dedup + Stage 1 detector + DINOv2 probe
import imagehash
from PIL import Image as PILImage
from ultralytics import YOLO
import yaml

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

def dedup_directory(root, phash_threshold=6):
    """Per-class SHA-256 + pHash dedup. Per-file work runs in a thread pool
    (PIL + hashlib release the GIL); the per-class hash-set membership
    check stays serial."""
    drop = []
    for cls in CLASSES:
        cls_dir = Path(root) / cls
        if not cls_dir.exists(): continue
        paths = sorted(iter_images(cls_dir))
        if not paths:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            shas    = list(pool.map(sha256_file, paths))
            phashes = list(pool.map(_phash_file, paths))
        seen_sha, seen_phash = set(), []
        for p, s, ph in zip(paths, shas, phashes):
            if s in seen_sha:
                drop.append((p, 'sha256_dup')); continue
            seen_sha.add(s)
            if ph is None:
                drop.append((p, 'unreadable')); continue
            if any(ph - existing <= phash_threshold for existing in seen_phash):
                drop.append((p, 'phash_dup')); continue
            seen_phash.append(ph)
    for p, _reason in drop:
        sc = p.with_suffix('.json')
        try:
            p.unlink()
            if sc.exists(): sc.unlink()
        except Exception:
            pass
    print(f'dedup({root}): dropped={len(drop)}')

def assert_no_exact_split_leakage(train_root, heldout_roots):
    """SHA-256 cross-check between train and every held-out split."""
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_shas = list(pool.map(sha256_file, train_paths))
    train_hash = dict(zip(train_shas, train_paths))
    problems = []
    for root in heldout_roots:
        if not Path(root).exists(): continue
        for p in iter_images(root):
            h = sha256_file(p)
            if h in train_hash:
                problems.append((str(train_hash[h]), str(p)))
                if len(problems) >= 10: break
        if len(problems) >= 10: break
    if problems:
        detail = '\n'.join(f'train={a}  heldout={b}' for a, b in problems)
        raise RuntimeError('Exact train/heldout leakage detected:\n' + detail)
    print('leakage audit (SHA): no exact overlap between train and held-out splits')

def resolve_phash_split_leakage(train_root, heldout_roots, threshold=6,
                                 max_drop_frac=0.25, max_report=10):
    """Perceptual-hash cross-check + auto-resolve. Source datasets (PlantDoc,
    PlantVillage) split near-duplicates across their published train/test
    boundaries; the §2 splitter can't stop those because they live on
    opposite sides of the HF split. This is the last-line defence before
    training reads the splits.

    Train data is never modified — held-out leaks are cheaper to lose
    than diverse training samples. A safety cap (``max_drop_frac``)
    raises if more than the given fraction of any held-out split would be
    removed.
    """
    # Group train pHashes by class. A held-out image is only checked
    # against train images of the **same class** so a leaf-spot photo
    # that pHash-collides with a (visually similar) bacterial photo
    # isn't dropped from val.
    train_hashes_by_class = {}
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_phashes = list(pool.map(_phash_file, train_paths))
    for p, ph in zip(train_paths, train_phashes):
        if ph is not None:
            train_hashes_by_class.setdefault(p.parent.name, []).append(ph)
    if not any(train_hashes_by_class.values()):
        print('pHash leakage resolution: TRAIN_ROOT empty — skipped')
        return
    total_dropped, samples = 0, []
    for root in heldout_roots:
        if not Path(root).exists():
            continue
        held = list(iter_images(root))
        if not held:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            held_phashes = list(pool.map(_phash_file, held))
        to_drop = []
        for p, ph in zip(held, held_phashes):
            if ph is None:
                continue
            same_class_hashes = train_hashes_by_class.get(p.parent.name, [])
            best = min((ph - tph for tph in same_class_hashes), default=None)
            if best is not None and best <= threshold:
                to_drop.append((p, int(best)))
        frac = len(to_drop) / max(len(held), 1)
        if frac > max_drop_frac:
            preview = '\n'.join(f'  heldout={p}  pHash_dist={d}'
                                for p, d in to_drop[:max_report])
            raise RuntimeError(
                f'pHash leakage in {root}: would drop {len(to_drop)}/{len(held)} '
                f'({frac:.0%}) — exceeds max_drop_frac={max_drop_frac:.0%}. '
                f'Investigate the upstream split before re-running.\n' + preview
            )
        for p, _d in to_drop:
            sc = p.with_suffix('.json')
            try:
                p.unlink()
                if sc.exists(): sc.unlink()
            except OSError:
                pass
        if to_drop:
            print(f'  resolved {root}: dropped {len(to_drop)} of {len(held)} '
                  f'(near-duplicate of train, Hamming ≤ {threshold})')
            samples.extend((str(p), d) for p, d in to_drop[:max_report])
        total_dropped += len(to_drop)
    if total_dropped == 0:
        print(f'leakage audit (pHash): no near-duplicates within Hamming {threshold}')
    else:
        preview = '\n'.join(f'  {p}  pHash_dist={d}' for p, d in samples[:max_report])
        print(f'leakage resolution (pHash): dropped {total_dropped} held-out '
              f'near-duplicate(s) within Hamming {threshold}\n' + preview)

def audit_cross_class_phash_leakage(train_root, heldout_roots, threshold=6,
                                    drop=True, max_report=12):
    """Class-AGNOSTIC cross-split pHash audit. resolve_phash_split_leakage
    only compares a held-out image to train images of the SAME class, so a
    PlantDoc near-duplicate that landed under DIFFERENT labels on each side
    of the upstream train/test boundary (e.g. train=leaf_spot,
    val=viral_disease) is never caught and silently inflates the ship-gate
    F1. This pass compares every held-out image to ALL train pHashes
    regardless of class; by default it drops the held-out near-dup (train is
    never touched) and reports both labels so an intentional KD hard-negative
    pair can be audited by eye. Vectorised popcount keeps the full
    held x train comparison cheap."""
    train_paths = list(iter_images(train_root))
    with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
        train_ph = list(pool.map(_phash_file, train_paths))
    train_cls, train_bits = [], []
    for p, ph in zip(train_paths, train_ph):
        if ph is None:
            continue
        train_cls.append(p.parent.name)
        train_bits.append(ph.hash.flatten())
    if not train_bits:
        print('cross-class pHash audit: TRAIN_ROOT empty — skipped')
        return 0
    train_bits = np.asarray(train_bits, dtype=bool)
    train_cls = np.asarray(train_cls)
    total_cross, samples = 0, []
    for root in heldout_roots:
        if not Path(root).exists():
            continue
        held = list(iter_images(root))
        if not held:
            continue
        with ThreadPoolExecutor(max_workers=EVAL_WORKERS) as pool:
            held_ph = list(pool.map(_phash_file, held))
        to_drop = []
        for p, ph in zip(held, held_ph):
            if ph is None:
                continue
            dists = np.logical_xor(train_bits, ph.hash.flatten()).sum(axis=1)
            j = int(dists.argmin()); d = int(dists[j])
            tcls, hcls = str(train_cls[j]), p.parent.name
            if d <= threshold and tcls != hcls:
                to_drop.append((p, d, tcls, hcls))
        for p, d, tcls, hcls in to_drop:
            samples.append((str(p), d, tcls, hcls))
            if drop:
                sc = p.with_suffix('.json')
                try:
                    p.unlink()
                    if sc.exists(): sc.unlink()
                except OSError:
                    pass
        total_cross += len(to_drop)
    if total_cross == 0:
        print(f'cross-class pHash audit: no cross-label near-dups within Hamming {threshold}')
    else:
        verb = 'dropped' if drop else 'flagged'
        preview = '\n'.join(f'  heldout[{h}]={p}  ~train[{t}]  dist={d}'
                            for p, d, t, h in samples[:max_report])
        print(f'cross-class pHash audit: {verb} {total_cross} cross-label held-out '
              f'near-dup(s) within Hamming {threshold}\n' + preview)
    return total_cross


def dedup_each_split(heldout_roots, threshold=6):
    """Per-split internal dedup, in case the splitters moved a near-dup
    pair into the same held-out slice."""
    for root in heldout_roots:
        if Path(root).exists():
            dedup_directory(root, phash_threshold=threshold)

# §2 already ran _dedup_train_in_place(TRAIN_ROOT) before splitting; only
# dedup the held-out splits here.
dedup_each_split(SPLIT_ROOTS)
assert_no_exact_split_leakage(TRAIN_ROOT, SPLIT_ROOTS)
resolve_phash_split_leakage(TRAIN_ROOT, SPLIT_ROOTS, threshold=6)
audit_cross_class_phash_leakage(TRAIN_ROOT, SPLIT_ROOTS, threshold=6, drop=True)

# ── Stage 1 — YOLOv8n single-class plant_leaf ───────────────────
DETECTOR_DATA_ROOT = ROOT / 'detector'
TRAIN_DETECTOR_FROM_SCRATCH = True


def find_detector_yaml():
    """Look for a pre-existing YOLO ``data.yaml`` under ``data_v2/``.

    A user can drop their own labelled detection dataset here; when none
    exists we fall back to ``build_yolo_detection_dataset_from_plantdoc()``
    so Stage 1 can still train end-to-end from the same source we used
    for the classifier crops.
    """
    if not ROOT.exists():
        return None
    cands = [p for p in ROOT.rglob('data.yaml') if 'cache_coco' not in str(p)]
    return cands[0] if cands else None


def _yolo_label_ids(label_dir):
    ids = set()
    for txt in Path(label_dir).rglob('*.txt'):
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if parts:
                try:
                    ids.add(int(float(parts[0])))
                except ValueError:
                    pass
    return ids


def _add_background_negatives(out_root: Path, n_train_pos: int, n_val_pos: int,
                              bg_frac: float = 0.12, seed: int = 29):
    """Add NAP images as YOLOv8 background negatives (image + EMPTY label
    .txt). Every PlantDoc detection row carries >=1 leaf bbox, so without
    negatives the detector never learns 'no leaf here' and fires at 0.9 on
    pure noise (the actual source of confident-garbage verdicts). An empty
    label file is the explicit YOLO 'background' marker. Pulls from
    not_a_plant — which now includes the §2 hard negatives — so the detector
    also sees leaf-like-but-not-leaf frames as backgrounds."""
    nap_dir = TRAIN_ROOT / 'not_a_plant'
    naps = sorted(iter_images(nap_dir)) if nap_dir.exists() else []
    if not naps:
        print('  no not_a_plant images for detector backgrounds; skipping')
        return (0, 0)
    rng = random.Random(seed); rng.shuffle(naps)
    n_train_bg, n_val_bg = int(n_train_pos * bg_frac), int(n_val_pos * bg_frac)
    take = naps[:n_train_bg + n_val_bg]
    added_tr = added_val = 0
    for k, src in enumerate(take):
        target = 'train' if k < n_train_bg else 'val'
        stem = f'bg_{src.stem}'
        img_path = out_root / 'images' / target / f'{stem}.jpg'
        lbl_path = out_root / 'labels' / target / f'{stem}.txt'
        if img_path.exists():
            continue
        try:
            _save_rgb(PILImage.open(src), img_path)
        except Exception:
            continue
        lbl_path.write_text('')   # empty label = background negative
        if target == 'train':
            added_tr += 1
        else:
            added_val += 1
    print(f'  detector backgrounds added: train={added_tr} val={added_val} (bg_frac={bg_frac})')
    return (added_tr, added_val)


def build_yolo_detection_dataset_from_plantdoc(
    out_root: Path = DETECTOR_DATA_ROOT,
    train_ratio: float = 0.85,
    seed: int = 7,
) -> Optional[Path]:
    """Generate a YOLOv8 single-class detection dataset from PlantDoc.

    PlantDoc HF rows ship the full source image plus one or more
    ``{bbox: [x, y, w, h], category: <leaf-class>}`` objects. This helper
    walks the HF dataset (cached locally after the first §2 run), writes
    the full image to ``out_root/images/{train,val}/`` and a parallel
    YOLO ``.txt`` to ``out_root/labels/{train,val}/`` with one
    ``0 cx cy w h`` row per bbox — every category collapsed to a single
    class ``plant_leaf`` (id 0).

    PlantDoc's own ``test`` split lands entirely in YOLO ``val`` so the
    eval set matches the upstream hold-out. ``train`` rows are sub-split
    deterministically by ``seed`` at ``train_ratio``.

    Returns the generated ``data.yaml`` path, or ``None`` when the HF
    pull failed.
    """
    try:
        ds = _hf_load_with_retry('agyaatcoder/PlantDoc')
    except Exception as exc:
        print(f'PlantDoc HF pull for YOLO dataset failed: {exc}')
        return None

    if out_root.exists():
        shutil.rmtree(out_root)
    for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
        (out_root / sub).mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    train_count = val_count = skipped_no_bbox = 0
    for split_name in [s for s in ('train', 'test') if s in ds]:
        split = ds[split_name]
        order = list(range(len(split)))
        rng.shuffle(order)
        for i in tqdm(order, desc=f'YOLO/{split_name}'):
            row = split[i]
            objects = row.get('objects') or {}
            bboxes = objects.get('bbox') or []
            if not bboxes:
                skipped_no_bbox += 1
                continue
            img = row['image'].convert('RGB')
            w, h = img.size
            lines = []
            for bbox in bboxes:
                x, y, bw, bh = (float(v) for v in bbox)
                if bw <= 1 or bh <= 1:
                    continue
                cx = (x + bw / 2) / w
                cy = (y + bh / 2) / h
                nw = bw / w
                nh = bh / h
                if not (0 < cx < 1 and 0 < cy < 1 and 0 < nw <= 1 and 0 < nh <= 1):
                    continue
                lines.append(f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
            if not lines:
                skipped_no_bbox += 1
                continue
            if split_name == 'test' or rng.random() > train_ratio:
                target = 'val'
            else:
                target = 'train'
            image_id = row.get('image_id', i)
            stem = f'plantdoc_{split_name}_{image_id}'
            img_path = out_root / 'images' / target / f'{stem}.jpg'
            lbl_path = out_root / 'labels' / target / f'{stem}.txt'
            if img_path.exists():
                continue
            _save_rgb(img, img_path)
            lbl_path.write_text('\n'.join(lines) + '\n')
            if target == 'train':
                train_count += 1
            else:
                val_count += 1

    _add_background_negatives(out_root, train_count, val_count, bg_frac=0.12)

    if train_count == 0 or val_count == 0:
        print(f'PlantDoc → YOLO produced too few rows '
              f'(train={train_count} val={val_count}); skipping.')
        return None

    yaml_path = out_root / 'data.yaml'
    yaml_path.write_text(yaml.safe_dump({
        'path': str(out_root.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'names': ['plant_leaf'],
        'nc': 1,
    }, sort_keys=False))
    print(f'YOLO dataset built: train={train_count}  val={val_count}  '
          f'skipped_no_bbox={skipped_no_bbox}')
    print(f'  data.yaml = {yaml_path}')
    return yaml_path


yaml_path = find_detector_yaml()
if yaml_path is None and TRAIN_DETECTOR_FROM_SCRATCH:
    print('No YOLO data.yaml under data_v2/ — building one from PlantDoc bboxes.')
    yaml_path = build_yolo_detection_dataset_from_plantdoc()
if yaml_path is None:
    print(f'No YOLO dataset under {ROOT}.  Stage 1 training skipped — '
          f'the shipped detector.{{pt,onnx}} will be used at inference.')
else:
    spec = yaml.safe_load(yaml_path.read_text())
    for key in ('train', 'val'):
        p = spec.get(key)
        if isinstance(p, str):
            labels_dir = Path(yaml_path.parent, p).as_posix().replace('/images/', '/labels/')
            if Path(labels_dir).exists():
                ids = _yolo_label_ids(labels_dir)
                if any(i != 0 for i in ids):
                    raise RuntimeError(
                        f'{key} labels contain class ids {sorted(ids)}; remap to single class 0 first.'
                    )
    spec['names'], spec['nc'] = ['plant_leaf'], 1
    patched = yaml_path.parent / 'plant_leaf.yaml'
    patched.write_text(yaml.safe_dump(spec, sort_keys=False))
    detector = YOLO('yolov8n.pt')
    detector.train(
        # batch=128 saturates an A100 at imgsz=640 (~12 GB peak). YOLOv8
        # auto-falls back to a smaller batch on OOM so this stays safe on
        # smaller cards too.
        data=str(patched), epochs=60, patience=12, batch=128, imgsz=640,
        optimizer='AdamW', lr0=1e-3, cos_lr=True, amp=True,
        # Indoor leaf bboxes are tight — heavy mosaic = leaks across boxes.
        mosaic=0.1, hsv_h=0.0, hsv_s=0.4, hsv_v=0.4,
        device=0 if device == 'cuda' else 'cpu', workers=8,
        project='runs/detect', name='plant_leaf_indoor_v1', exist_ok=True,
    )
    best_pt = Path(detector.trainer.best) if getattr(detector, 'trainer', None) else next(
        Path('runs/detect').rglob('best.pt'))
    exported = YOLO(str(best_pt)).export(format='onnx', opset=13, dynamic=False)
    Path('models').mkdir(exist_ok=True)
    shutil.copy(best_pt, 'models/detector.pt')
    if exported and Path(exported).exists():
        shutil.copy(exported, 'models/detector.onnx')
    else:
        src = next(best_pt.parent.rglob('*.onnx'), None)
        if src: shutil.copy(src, 'models/detector.onnx')

# ── DINOv2-S linear probe (diagnostic) ──────────────────────────
# Backbone is frozen — extract features ONCE, then iterate the linear
# head over the cached embeddings. ~30× faster than recomputing the
# ViT-S forward pass per epoch on an A100.
PROBE_SIZE = 224
PROBE_TX = T.Compose([T.Resize(int(PROBE_SIZE * EVAL_RESIZE_RATIO)),
                      T.CenterCrop(PROBE_SIZE), T.ToTensor(), NORM])
train_ds_probe = ImageFolder(str(TRAIN_ROOT), transform=PROBE_TX)
assert train_ds_probe.classes == sorted(CLASSES), (train_ds_probe.classes, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds_probe.class_to_idx)
train_dl_probe = DataLoader(
    train_ds_probe, batch_size=256, shuffle=False,
    **_loader_kwargs(EVAL_WORKERS, drop_last=False),
)
_, val_dl_probe = make_eval_loader(
    VAL_PRIMARY, PROBE_SIZE, batch=256, class_to_idx=TRAIN_CLASS_TO_IDX,
)

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval().to(device)
for p in backbone.parameters(): p.requires_grad_(False)

@torch.inference_mode()
def _extract_features(dl):
    feats, labels = [], []
    for x, y in dl:
        f = backbone(x.to(device, non_blocking=True))
        feats.append(f.detach().cpu())
        labels.append(y.detach().cpu())
    return torch.cat(feats), torch.cat(labels)

print('caching DINOv2 features (train + val)…')
train_feats, train_labels = _extract_features(train_dl_probe)
val_feats,   val_labels   = _extract_features(val_dl_probe)
print(f'  train_feats={tuple(train_feats.shape)}  val_feats={tuple(val_feats.shape)}')

# Move to device for the head loop — small enough (<20 MB even at v2.2 scale).
train_feats_g = train_feats.to(device); train_labels_g = train_labels.to(device)
val_feats_g   = val_feats.to(device);   val_labels_g   = val_labels.to(device)

head = nn.Linear(train_feats.shape[1], len(train_ds_probe.classes)).to(device)
opt  = torch.optim.AdamW(head.parameters(), lr=3e-3, weight_decay=1e-4)
loss_fn_probe = nn.CrossEntropyLoss(label_smoothing=0.05)
PROBE_BATCH = 1024

probe_curve = []
g = torch.Generator(device='cpu').manual_seed(SEED)
for ep in range(8):
    head.train()
    perm = torch.randperm(train_feats_g.size(0), generator=g)
    for i in range(0, perm.size(0), PROBE_BATCH):
        idx = perm[i:i + PROBE_BATCH].to(device)
        opt.zero_grad(set_to_none=True)
        loss_fn_probe(head(train_feats_g[idx]), train_labels_g[idx]).backward()
        opt.step()
    head.eval()
    with torch.inference_mode():
        pred = head(val_feats_g).argmax(1)
        top1 = float((pred == val_labels_g).float().mean().item())
    probe_curve.append({'epoch': ep + 1, 'val_realworld_top1': top1})
    print(f'ep={ep+1}  val_realworld_top1={top1:.3f}')

Path('models/dinov2_probe.json').write_text(json.dumps(probe_curve, indent=2))
log_run({'phase': 'dinov2_probe', 'curve': probe_curve})
if probe_curve[-1]['val_realworld_top1'] < 0.30:
    print('WARN: DINOv2 frozen+probe < 0.30 — your data is hard. '
          'Audit labels in §2 before spending GPU on the teacher.')

# Per-class recall on val_realworld for the thin classes — surfaces a dead
# thin class here (cheap, on the frozen probe) instead of ~40 min later at
# the §6 ship gate. Warn-only by default; flip the flag to abort early.
PROBE_FAIL_FAST_ON_DEAD_THIN = False
_THIN_CLASSES = ('gray_mold_botrytis', 'leaf_scorch_underwatering', 'nutrient_deficiency')
with torch.inference_mode():
    _probe_pred = head(val_feats_g).argmax(1).cpu().numpy()
_probe_true = val_labels_g.cpu().numpy()
_dead_thin = []
print('thin-class probe recall (val_realworld):')
for _c in _THIN_CLASSES:
    _ci = TRAIN_CLASS_TO_IDX.get(_c)
    if _ci is None:
        continue
    _support = int((_probe_true == _ci).sum())
    if _support == 0:
        print(f'    recall[{_c:25s}] =   n/a  (no val_realworld samples)')
        continue
    _rec = float(((_probe_pred == _ci) & (_probe_true == _ci)).sum() / _support)
    print(f'    recall[{_c:25s}] = {_rec:.3f}  (support={_support})')
    if _rec == 0.0:
        _dead_thin.append(_c)
if _dead_thin:
    _msg = f'DINOv2 probe: zero recall on thin class(es) {_dead_thin} on val_realworld'
    if PROBE_FAIL_FAST_ON_DEAD_THIN:
        raise RuntimeError(_msg + ' — PROBE_FAIL_FAST_ON_DEAD_THIN=True. '
                           'Audit the source for these classes in §2 first.')
    print(f'WARN: {_msg} (warn-only; set PROBE_FAIL_FAST_ON_DEAD_THIN=True to abort).')

# Free memory before §4.
del backbone, head, train_dl_probe, val_dl_probe
del train_feats, train_feats_g, train_labels, train_labels_g
del val_feats,   val_feats_g,   val_labels,   val_labels_g
torch.cuda.empty_cache() if device == 'cuda' else None


## §4 — Loss, augs, frozen DINOv2 ViT-S/14 + trained head

* **Loss:** `LogitAdjustmentLoss(τ=0.5, label_smoothing=0.05)`
  (Menon et al. ICLR 2021). Loss-side rebalancing matched to the
  **training prior** — no `WeightedRandomSampler` by default, because
  stacking the two over-corrects rare classes (Liu et al. 2024,
  *frontiersin*).
* **Sampler off by default.** Set `USE_WEIGHTED_SAMPLER = True` only
  if a single class dominates by > 10×.
* **Model:** frozen DINOv2 ViT-S/14 backbone (`torch.hub`,
  `dinov2_vits14`; flip `USE_REGISTERS` for the `_reg` weights) + a small
  MLP head (`LayerNorm → Linear → GELU → Dropout → Linear`) on the 384-d
  CLS embedding. **Only the head trains**; the backbone stays frozen and
  in `eval()` so its (LayerNorm-only) statistics are identical at train,
  eval and export time. Input is **224 px** (16×14 — DINOv2 patches are
  14 px). Early-stop on `val_indomain` disease macro-F1.
* **No teacher, no KD.** Directly fitting a strong frozen backbone beats
  distilling into a weaker student when there is no size constraint; the
  former EfficientNetV2-S teacher fine-tune was removed.
* **Augs:** indoor-targeted — WB shift, glare, defocus / motion blur,
  JPEG re-compression, mild ColorJitter, RandomErasing, horizontal flip.
  **No** strong hue rotation / Solarize — colour is a disease signal.


In [ ]:
# §4 — Loss, augs, frozen DINOv2 ViT-S/14 + trained head
import torch.nn.functional as F
from collections import Counter
from contextlib import nullcontext
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import WeightedRandomSampler
from PIL import ImageFilter, ImageEnhance, Image as PILImage
from io import BytesIO

# ── Backbone / head config ──────────────────────────────────────
# DINOv2 patches are 14 px, so the input side must be a multiple of 14.
# 224 = 16×14 matches the §3 probe that demonstrated the ~0.91 ceiling.
IMG, BATCH = FAMILY_CONFIG['img_size'], 128
USE_REGISTERS = False   # dinov2_vits14_reg gives slightly cleaner features,
                        # but the §3 probe (and thus the demonstrated 0.91
                        # ceiling) used the non-register weights — default
                        # matches what we validated.
HEAD_HIDDEN = 384       # ViT-S embedding dim → 1-hidden-layer MLP head.
HEAD_DROPOUT = 0.2
HEAD_EPOCHS = 30
HEAD_LR = 1e-3
HEAD_PATIENCE = 5
# Precision-aware checkpoint selection — OPT-IN, OFF BY DEFAULT (v2.5.0).
# The physiological classes are precision-bound (decent recall, low precision —
# they over-trigger). When enabled, the best checkpoint maximises disease
# macro-F1 PLUS a small bonus for the weak classes' F0.5 (precision-weighted,
# recall-aware so a barely-predicting epoch earns ~0) on val_indomain (which
# holds only REAL in-domain sources, never the crop proxies).
#
# DEFAULT 0.0 == pure macro-F1 (the proven selection). A non-zero weight can
# trade up to `weight` of indomain macro-F1 for weak-class precision, and that
# trade is NOT yet A/B-validated to transfer to the val_realworld SHIP metric —
# so it stays OFF until measured on GPU. Raise it (e.g. 0.05) only as a
# deliberate, validated A/B. Precision + CIs are reported/shipped either way
# (§6), so the over-trigger is always visible regardless of this knob.
SELECT_WEAKCLASS_PRECISION_WEIGHT = 0.0
SELECT_WEAKCLASS_PRECISION_CLASSES = ('leaf_scorch_underwatering', 'nutrient_deficiency')
def _weakclass_fbeta(precision_by_cls, recall_by_cls, beta2=0.25):
    """Mean F0.5 (precision-weighted, recall-aware) over the precision-bound
    weak classes — the selection bonus metric. F0.5 -> ~0 as recall -> 0, so a
    degenerate 'predict the class once, precision 1.0' epoch earns no bonus."""
    vals = []
    for c in SELECT_WEAKCLASS_PRECISION_CLASSES:
        if c not in precision_by_cls:
            continue
        p = float(precision_by_cls[c]); r = float(recall_by_cls.get(c, 0.0))
        denom = beta2 * p + r
        vals.append((1.0 + beta2) * p * r / denom if denom > 0 else 0.0)
    return float(sum(vals) / len(vals)) if vals else 0.0

# ── LoRA fine-tune of the backbone (report "last few points") ────
# The frozen probe capped in-domain nutrient_deficiency at ~0.32 — the
# FEATURES, not the head, are the ceiling. LoRA adapts the LATER DINOv2
# blocks to leaf imagery at low rank/LR with minimal forgetting. Validated
# end-to-end on dinov2_vits14 (wrap->train->merge_and_unload->ONNX, parity
# ~1e-6). USE_LORA=False falls back to the pure frozen-head path.
USE_LORA = True
LORA_RANK = 16             # v2.4: ↑ from 8 — the §3 probe showed the FEATURES
LORA_ALPHA = 32            #   (not the head) cap weak-class recall, so give the
LORA_DROPOUT = 0.05        #   adapter more capacity to specialise DINOv2 to leaf
LORA_LR = 1e-4             #   imagery. A100-40GB has the headroom; the merged
                           #   weights still export to a single ONNX graph.
LORA_LAST_K_BLOCKS = 8     # adapt the last K of the ViT's blocks (↑ from 6)

USE_WEIGHTED_SAMPLER = False   # class-balanced sampler — off by default
# Per-image source weighting attacks the agricultural / houseplant domain
# mismatch. PlantDoc is field crops; the deployment target is indoor
# houseplants. Down-weighting PlantDoc rows in the sampler pulls the
# empirical training prior toward the real distribution without dropping
# the hard negatives the NAP head leans on. The LA loss prior is recomputed
# from the sampler weights (``class_priors_from_source_weights``) so the
# correction stays aligned with the empirical per-batch distribution.
# Never enable alongside USE_WEIGHTED_SAMPLER (double rebalance).
USE_SOURCE_WEIGHTED_SAMPLER = True
SOURCE_WEIGHTS = {
    # Rebalance toward the deployment/eval distribution. PlantDoc (field
    # photos) is what val_realworld is seeded from, so favour it;
    # PlantVillage studio images barely transfer; the NAP-only sources
    # (coco / inat_animalia / openimages_indoor) were over-weighted at 1.0.
    'plantdoc':            1.1,
    'plantvillage':        0.35,
    'inat':                1.0,
    # DoctorP real app photos — the deployment distribution itself; full
    # weight, and the only Kaggle source allowed into the eval/cal splits.
    'doctorp':             1.0,
    'inat_animalia':       0.5,
    'inat_physiological':  1.2,
    'coco':                0.5,
    'openimages_indoor':   0.5,
    # CoLeaf coffee-leaf nutrient-deficiency: curated/clean labels but a
    # crop-domain supplement, so down-weighted like PlantVillage — the
    # deficiency signal is learned without pulling the prior off the
    # in-domain houseplant distribution.
    'coleaf':              0.5,
    # MPLD maize phosphorus deficiency: same crop-domain down-weight
    # as CoLeaf (curated labels, out-of-domain species).
    'mpld':                0.5,
    # Pest classes (mealybug / scale) from iNat research-grade — in-domain
    # houseplant pests, so full weight (not a crop supplement). Inert until
    'inat_pest':           1.0,
    # Kaggle indoor houseplant disease supplement: in-domain (indoor
    # houseplants) but possibly partly studio-framed and unverified, so
    # a moderate weight between in-domain (1.0) and crop supplement (0.5).
    # v2.4: down-weighted further. The set is black-bg-segmented + augmented
    # Aloe (now rejected at fetch) plus a few small real-photo folders.
    'kaggle_houseplant':   0.35,
    # Ornamental/houseplant disease set (pappu54): rose / hibiscus / money-plant
    # (Epipremnum = pothos, a true houseplant) leaves with REAL disease
    # morphology on NATURAL backgrounds — much closer to the deployment domain
    # than the field-crop proxies. Moderate weight (partly augmented).
    'kaggle_ornamental':   0.8,
    # Strawberry Botrytis: crop-domain gray-mold proxy, down-weighted like
    # the CoLeaf/MPLD nutrient proxies so it teaches the texture without
    # pulling the prior off the in-domain houseplant gray-mold.
    'kaggle_strawberry':   0.5,
    # Rice N/P/K nutrient-deficiency: crop-domain proxy (CoLeaf replacement),
    # down-weighted like the other crop nutrient sources.
    'kaggle_nutrient':     0.5,
    # Melon BROADLEAF macronutrient deficiency (N/Ca/K, clean neutral bg):
    # a dicot nutrient proxy, same crop-domain down-weight as rice/maize.
    'kaggle_melon':        0.5,
    # Real-houseplant healthy species photos: in-domain but whole-plant/
    # species-labelled, so a moderate weight (diversity, not volume).
    'kaggle_species':      0.7,
}

# ── Logit-adjustment prior weights (DECOUPLED from the sampler) ──
# The LA prior should track the DEPLOYMENT class frequency, not the crop-
# inflated training count. CoLeaf/MPLD are out-of-domain crop sources that
# teach the deficiency *signal* but say nothing about how often a houseplant
# is nutrient-deficient in production. In v2.3.0 they entered the LA prior at
# their sampler weight (0.5) and, with ~1000 rows, pushed nutrient_deficiency
# to a head-class prior (0.1165) -> LA SUPPRESSED it -> in-domain recall fell
# 0.55->0.26. Here the crop sources get a SMALLER prior weight than their
# sampler weight, so the model still TRAINS on them (sampler 0.5) but LA
# treats nutrient_deficiency as the deployment-rare class it is and applies
# the rare-class boost. The §2 caps keep the sampler/prior gap small enough
# that this does not over-predict the class. (Report Stage 2: decouple
# representation sampling from the rebalancing prior.)
PRIOR_SOURCE_WEIGHTS = dict(SOURCE_WEIGHTS)
PRIOR_SOURCE_WEIGHTS['coleaf'] = 0.25
PRIOR_SOURCE_WEIGHTS['mpld'] = 0.25
# Kaggle houseplant: decouple the LA prior below the sampler weight too
# (unverified domain mix) so it teaches signal without inflating the prior.
PRIOR_SOURCE_WEIGHTS['kaggle_houseplant'] = 0.25
PRIOR_SOURCE_WEIGHTS['kaggle_ornamental'] = 0.6
PRIOR_SOURCE_WEIGHTS['kaggle_strawberry'] = 0.25
PRIOR_SOURCE_WEIGHTS['kaggle_nutrient'] = 0.25
PRIOR_SOURCE_WEIGHTS['kaggle_melon'] = 0.25
PRIOR_SOURCE_WEIGHTS['kaggle_species'] = 0.5
LA_TAU = 0.5                   # logit adjustment τ — flattened post-cap
                                #   distribution justifies the lower value.
LABEL_SMOOTHING = 0.05

class LogitAdjustmentLoss(nn.Module):
    """Loss-side Logit Adjustment, Menon et al. ICLR 2021 eq. 4."""
    def __init__(self, class_priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.register_buffer('log_prior', torch.log(class_priors.clamp(min=1e-12)))
        self.tau, self.ls = tau, label_smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.tau * self.log_prior, target,
                               label_smoothing=self.ls)

def class_priors_from_imagefolder(ds):
    counts = Counter([y for _, y in ds.samples])
    pri = torch.tensor([counts.get(i, 0) for i in range(len(ds.classes))], dtype=torch.float32)
    return pri.clamp(min=1.0) / pri.clamp(min=1.0).sum()

def class_priors_from_source_weights(ds, source_weights, default_weight=1.0):
    """Per-class prior under the source-weighted sampler.

    LogitAdjustment (Menon et al. ICLR 2021 eq. 4) assumes the ``log_prior``
    fed to the loss equals the per-batch class distribution the model
    actually sees during training. Under the source-weighted sampler the
    file-count prior is wrong; this helper sums each image's source weight
    into its class bucket. Pass the sampler's SOURCE_WEIGHTS to mirror the
    draw distribution, OR pass PRIOR_SOURCE_WEIGHTS to intentionally DECOUPLE
    the prior from the sampler: out-of-domain representation-only sources
    (coleaf, mpld) are down-weighted in the prior so the deployment-rare
    classes they pad (nutrient_deficiency) still receive the LA rare-class
    boost while the sampler keeps training on their signal.
    """
    counts = torch.zeros(len(ds.classes), dtype=torch.float32)
    for path, y in ds.samples:
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        w = max(float(source_weights.get(src_name, default_weight)), 0.0)
        counts[y] += w
    return counts.clamp(min=1e-6) / counts.clamp(min=1e-6).sum()

def make_class_balanced_sampler(ds, power=0.5):
    counts = Counter([y for _, y in ds.samples])
    weights = torch.DoubleTensor([(1.0 / counts[y]) ** power for _, y in ds.samples])
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def make_source_weighted_sampler(ds, source_weights, default_weight=1.0):
    """Sample images at a rate proportional to their sidecar `source`. Images
    without a sidecar (or with an unknown source) get ``default_weight``."""
    drawn = Counter()
    sources_per_path = []
    weights = torch.zeros(len(ds.samples), dtype=torch.float64)
    for i, (path, _) in enumerate(ds.samples):
        sc = read_sidecar(Path(path)) or {}
        src_name = sc.get('source')
        if not isinstance(src_name, str):
            src_name = 'unknown'
        w = source_weights.get(src_name, default_weight)
        weights[i] = max(float(w), 0.0)
        drawn[src_name] += 1
        sources_per_path.append(src_name)
    total = weights.sum().item()
    if total <= 0:
        raise RuntimeError('source-weighted sampler: all weights are zero — '
                           'check sidecar `source` field and SOURCE_WEIGHTS')
    expected = {s: round(weights[[i for i, x in enumerate(sources_per_path) if x == s]].sum().item() / total, 4)
                for s in drawn}
    print(f'source-weighted sampler  raw_counts={dict(drawn)}  '
          f'expected_draw_fraction={expected}  weight_lookup={source_weights}')
    log_run({'phase': 'sampler', 'tag': 'source_weighted',
             'raw_counts': dict(drawn), 'expected_draw_fraction': expected,
             'weight_lookup': source_weights})
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True,
                                 generator=torch.Generator().manual_seed(SEED))

def amp_ctx():
    return autocast(device_type='cuda', dtype=torch.bfloat16) if device == 'cuda' else nullcontext()

class WhiteBalanceShift:
    def __init__(self, p=0.5, strength=(0.85, 1.15)):
        self.p, self.lo, self.hi = p, *strength
    def __call__(self, img):
        if random.random() > self.p: return img
        r_scale = random.uniform(self.lo, self.hi)
        b_scale = random.uniform(self.lo, self.hi)
        r, g, b = img.split()
        r = r.point(lambda v: max(0, min(255, int(v * r_scale))))
        b = b.point(lambda v: max(0, min(255, int(v * b_scale))))
        return PILImage.merge('RGB', (r, g, b))

class Glare:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        return img if random.random() > self.p else \
               ImageEnhance.Brightness(img).enhance(random.uniform(1.05, 1.35))

class DefocusOrMotion:
    def __init__(self, p=0.3): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        f = (ImageFilter.GaussianBlur(radius=random.uniform(0.6, 1.8))
             if random.random() < 0.5
             else ImageFilter.BoxBlur(random.uniform(0.8, 2.0)))
        return img.filter(f)

class JpegRecompress:
    def __init__(self, p=0.35, q_range=(50, 92)): self.p, self.q_range = p, q_range
    def __call__(self, img):
        if random.random() > self.p: return img
        buf = BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.q_range))
        buf.seek(0)
        return PILImage.open(buf).convert('RGB').copy()

indoor_train_tx = T.Compose([
    T.Resize(int(IMG * 1.15)),
    T.RandomResizedCrop(IMG, scale=(0.70, 1.0), ratio=(0.88, 1.14)),
    T.RandomHorizontalFlip(),
    T.RandomApply([T.RandomPerspective(distortion_scale=0.12, p=1.0)], p=0.20),
    WhiteBalanceShift(p=0.5), Glare(p=0.25), DefocusOrMotion(p=0.3), JpegRecompress(p=0.35),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.0),
    T.ToTensor(), NORM, T.RandomErasing(p=0.12, scale=(0.02, 0.08)),
])

# ── Datasets / loaders ──────────────────────────────────────────
train_ds = ImageFolder(str(TRAIN_ROOT), transform=indoor_train_tx)
if len(train_ds) == 0:
    raise RuntimeError('TRAIN_ROOT empty — re-run §2.')
CLASS_NAMES = list(train_ds.classes)
assert CLASS_NAMES == sorted(CLASSES), (CLASS_NAMES, sorted(CLASSES))
TRAIN_CLASS_TO_IDX = dict(train_ds.class_to_idx)
N_CLASSES = len(CLASS_NAMES)

if USE_SOURCE_WEIGHTED_SAMPLER and not USE_WEIGHTED_SAMPLER:
    priors = class_priors_from_source_weights(train_ds, PRIOR_SOURCE_WEIGHTS).to(device)
    print(f'LA prior (PRIOR_SOURCE_WEIGHTS; crop decoupled from sampler): '
          f'{ {n: round(float(p), 4) for n, p in zip(train_ds.classes, priors.cpu().tolist())} }')
else:
    priors = class_priors_from_imagefolder(train_ds).to(device)
loss_fn = LogitAdjustmentLoss(priors, tau=LA_TAU, label_smoothing=LABEL_SMOOTHING)

if USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER:
    raise RuntimeError('USE_SOURCE_WEIGHTED_SAMPLER and USE_WEIGHTED_SAMPLER are '
                       'mutually exclusive — they would compose into a double '
                       'rebalance on top of the logit-adjustment loss prior.')
LOADER_WORKERS = min(12, EVAL_WORKERS)
if USE_SOURCE_WEIGHTED_SAMPLER:
    sampler = make_source_weighted_sampler(train_ds, SOURCE_WEIGHTS)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
elif USE_WEIGHTED_SAMPLER:
    sampler = make_class_balanced_sampler(train_ds, power=0.5)
    train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))
else:
    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED),
                          **_loader_kwargs(LOADER_WORKERS, drop_last=True))

val_indomain_ds, val_indomain_dl = make_eval_loader(
    VAL_INDOMAIN, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
val_realworld_ds, val_realworld_dl = make_eval_loader(
    VAL_PRIMARY, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
temp_cal_ds, temp_cal_dl = make_eval_loader(
    TEMP_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)
try:
    val_studio_ds, val_studio_dl = make_eval_loader(
        VAL_STUDIO, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
    )
except (RuntimeError, FileNotFoundError):
    print('val_studio not available (no class subdirs); skipping studio monitor.')
    val_studio_ds = val_studio_dl = None

# Studio is a non-blocking transfer monitor, but a sudden drift is worth
# a loud line. Warn when bias falls outside this range.
STUDIO_BIAS_BAND = (0.7, 1.5)

def _maybe_warn_studio_bias(metrics, tag):
    if metrics is None:
        return
    bias = float(metrics.get('healthy_bias', 1.0))
    lo, hi = STUDIO_BIAS_BAND
    if not lo <= bias <= hi:
        print(f'WARN: [{tag}] studio healthy_bias={bias:.2f} outside '
              f'[{lo:.2f}, {hi:.2f}]  (transfer monitor; non-blocking)')

def _eval_under_amp(model, dl, tag):
    with amp_ctx():
        return evaluate(model, dl, CLASS_NAMES, device, tag)

def step_eval(model, tag):
    """Per-epoch eval used for early-stopping decisions only.

    val_realworld is intentionally NOT computed per epoch: it is the ship
    gate, and looking at it every epoch leaks via human inspection of
    run_log.jsonl. The single post-training measurement happens in §6
    against the deployed INT8 ORT session.
    """
    out = {'indomain': _eval_under_amp(model, val_indomain_dl, f'{tag} val_indomain')}
    if val_studio_dl is not None:
        out['studio'] = _eval_under_amp(model, val_studio_dl, f'{tag} val_studio')
        _maybe_warn_studio_bias(out['studio'], tag)
    row = {'phase': 'head', 'tag': tag,
           'indomain_disease_f1': out['indomain']['disease_macro_f1'],
           'indomain_healthy_bias': out['indomain']['healthy_bias']}
    if 'studio' in out:
        row['studio_disease_f1'] = out['studio']['disease_macro_f1']
        row['studio_healthy_bias'] = out['studio']['healthy_bias']
    log_run(row)
    return out

def _make_sched(opt, n_epochs, n_steps_per_epoch, cosine=True):
    """Linear-warmup + cosine schedule. Set ``cosine=False`` for a flat
    post-warmup LR."""
    if n_steps_per_epoch <= 0:
        raise RuntimeError('Training loader has zero batches; lower BATCH or add data.')
    warmup = LinearLR(opt, start_factor=0.1, total_iters=n_steps_per_epoch)
    if not cosine:
        return warmup
    return SequentialLR(opt, schedulers=[
        warmup,
        CosineAnnealingLR(opt, T_max=max(1, (n_epochs - 1) * n_steps_per_epoch)),
    ], milestones=[n_steps_per_epoch])

# ── Frozen DINOv2 ViT-S/14 + trained MLP head ───────────────────
# The §3 frozen DINOv2-S linear probe already reached ~0.91 top-1 on
# val_realworld, proving the features separate the 8 classes. We promote
# that result to the *production* model: freeze the ViT-S/14 backbone and
# train a small MLP head on the CLS embedding under the same logit-
# adjustment loss. No teacher, no knowledge distillation — with no model-
# size constraint and ~10× CPU latency headroom, directly fitting a strong
# frozen backbone is the higher-ceiling, lower-risk path (vanilla KD into a
# weaker, INT8-hostile MobileNetV3 student is exactly the regime where KD
# underperforms; see DIST/DKD). The backbone is exported into the same ONNX
# graph as the head in §6, so serving stays a single self-contained file.
_HUB_ID = FAMILY_CONFIG['hub_id']
if USE_REGISTERS and _HUB_ID == 'dinov2_vits14':
    _HUB_ID = 'dinov2_vits14_reg'
ARCHITECTURE = f'{_HUB_ID}+mlp_head' + ('+lora' if USE_LORA else '')

dino_backbone = torch.hub.load('facebookresearch/dinov2', _HUB_ID).eval().to(device)
for p in dino_backbone.parameters():
    p.requires_grad_(False)
EMBED_DIM = int(getattr(dino_backbone, 'embed_dim', 384))
print(f'DINOv2 backbone={_HUB_ID}  embed_dim={EMBED_DIM}  (frozen)')

# Wrap the LATER blocks with LoRA adapters (base stays frozen + eval). peft
# matches target_modules as a REGEX on module names; we encode both the
# block range and the attn/mlp Linear suffixes. merge_and_unload() in §6
# folds the adapters into the base Linears so the ONNX stays one graph.
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    _n_blocks = len(dino_backbone.blocks)
    _lo = max(0, _n_blocks - LORA_LAST_K_BLOCKS)
    _rng = '|'.join(str(i) for i in range(_lo, _n_blocks))
    LORA_TARGET_REGEX = rf'blocks\.({_rng})\.(attn\.(qkv|proj)|mlp\.(fc1|fc2))'
    dino_backbone = get_peft_model(dino_backbone, LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_REGEX, bias='none'))
    _n_lora = sum(p.numel() for p in dino_backbone.parameters() if p.requires_grad)
    print(f'LoRA enabled: rank={LORA_RANK} alpha={LORA_ALPHA} '
          f'blocks=[{_lo},{_n_blocks}) trainable_backbone_params={_n_lora:,}')

class DINOv2Classifier(nn.Module):
    """Frozen DINOv2 ViT-S/14 backbone + trainable MLP head on the CLS
    embedding.

    The backbone runs under ``torch.no_grad`` so only the head learns and
    the backbone never accumulates an autograd graph. ``torch.onnx.export``
    traces the executed ops regardless of grad mode, so the exported ONNX
    is the full, self-contained backbone+head graph.
    """
    def __init__(self, backbone, embed_dim, num_classes, hidden, p_drop,
                 lora_active=False):
        super().__init__()
        self.backbone = backbone
        self.lora_active = lora_active
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        if self.lora_active:
            feats = self.backbone(x)          # LoRA adapters need a graph
        else:
            with torch.no_grad():
                feats = self.backbone(x)      # (B, embed_dim) CLS embedding
        return self.head(feats)

model = DINOv2Classifier(
    dino_backbone, EMBED_DIM, N_CLASSES, HEAD_HIDDEN, HEAD_DROPOUT,
    lora_active=USE_LORA,
).to(device)

# The head always trains. With LoRA the adapter params on the later blocks
# train too (lower LR); the frozen base stays in eval() so its LayerNorm
# stats + base dropout are identical at train, eval and export time.
if USE_LORA:
    _lora_params = [p for p in model.backbone.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(
        [{'params': model.head.parameters(), 'lr': HEAD_LR},
         {'params': _lora_params, 'lr': LORA_LR}], weight_decay=1e-4)
    print(f'optimiser: head@{HEAD_LR} + {len(_lora_params)} LoRA tensors@{LORA_LR}')
else:
    opt = torch.optim.AdamW(model.head.parameters(), lr=HEAD_LR, weight_decay=1e-4)
sched = _make_sched(opt, HEAD_EPOCHS, len(train_dl))

best_f1, best_ep, best_state, plateau = -1.0, 0, None, 0
best_sel = -1.0   # best precision-aware selection score (>= best_f1 by design)
for ep in range(HEAD_EPOCHS):
    model.train()
    model.backbone.eval()   # frozen backbone never leaves eval mode
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device)
        with amp_ctx():
            logits = model(x)
            loss = loss_fn(logits, y)
        if not torch.isfinite(loss):
            print(f'WARN: non-finite loss at head ep{ep+1}; skipping batch'); continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=5.0)
        opt.step(); sched.step()
    m = step_eval(model, f'head ep{ep+1}')
    ind = m['indomain']
    f1 = ind['disease_macro_f1']
    # Precision-aware selection score (see SELECT_WEAKCLASS_PRECISION_* above).
    _mean_weak_f05 = _weakclass_fbeta(ind.get('per_class_precision', {}),
                                      ind.get('per_class_recall', {}))
    sel = f1 + SELECT_WEAKCLASS_PRECISION_WEIGHT * _mean_weak_f05
    if sel > best_sel:
        best_sel, best_f1, best_ep, plateau = sel, f1, ep + 1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        log_run({'phase': 'head_select', 'tag': f'ep{ep+1}', 'sel_score': sel,
                 'indomain_disease_f1': f1, 'mean_weak_f05': _mean_weak_f05})
    else:
        plateau += 1
        if plateau >= HEAD_PATIENCE:
            print(f'head early stop @ ep{ep+1}  (best ep={best_ep} f1={best_f1:.3f} '
                  f'sel={best_sel:.3f})')
            break

if best_state is None:
    raise RuntimeError('head training produced no checkpoint — check the loop.')
model.load_state_dict(best_state)
print(f'restored best head  best_indomain_disease_f1={best_f1:.3f} (ep{best_ep})')

Path('models').mkdir(exist_ok=True)
torch.save(model.state_dict(), 'models/classifier.pth')
log_run({'phase': 'head_done', 'architecture': ARCHITECTURE,
         'best_indomain_f1': best_f1, 'best_ep': best_ep})


## §5 — Finalize teacher; Aqua: distill → MobileNetV3-Small TFLite student

For **gamma/frontier** this just freezes the §4 frozen-DINOv2 + MLP head for the §6 ONNX export. For **aqua** (`MODEL_FAMILY='aqua'`) the §4 model is the *teacher* and we knowledge-distill it into a MobileNetV3-Small student exported to TFLite (dynamic-range INT8, float I/O) for fully offline on-device inference. KD is warranted here because the target is a hard on-device size/latency budget — a tiny student is the deliverable, not a regression.

In [ ]:
# §5 — Finalize teacher; for Aqua: distill into a MobileNetV3-Small TFLite student
#
# Gamma / Frontier: the production model IS the §4 frozen-DINOv2 + MLP head, so
# this cell only freezes it for the §6 ONNX export.
#
# Aqua (FAMILY_CONFIG['distill']): the §4 model becomes the TEACHER and we
# knowledge-distill it into a small MobileNetV3-Small student that exports to
# TFLite for fully offline on-device inference. KD is justified HERE (unlike the
# removed DINOv2 KD): the target is a hard on-device latency/size budget, so a
# weaker-but-tiny student is the deliverable, not a regression.

model.eval()
for p in model.parameters():
    p.requires_grad_(False)
if device == 'cuda':
    torch.cuda.empty_cache()
print(f'Teacher (DINOv2 + head) ready  best_indomain_disease_f1={best_f1:.3f} (ep{best_ep})')

if FAMILY_CONFIG.get('distill'):
    # Aqua distill + TFLite export is FAIL-SOFT: a flaky on-device export
    # toolchain (onnx2tf / tensorflow) must NOT cost the expensive §4
    # teacher or block the §6 gamma server bundle below. On failure we log
    # + continue; the gamma bundle still ships and §5 can be re-run alone.
    try:
        import torchvision
        from PIL import Image as _PILImage
        AQUA_DIR = Path('models/aqua'); AQUA_DIR.mkdir(parents=True, exist_ok=True)

        # ── Hyperparameters ────────────────────────────────────────────
        STUDENT_IMG = 224        # MobileNetV3 native res (teacher still sees IMG)
        KD_EPOCHS   = 30
        KD_T        = 3.0        # softmax temperature for the soft-target KL term
        KD_ALPHA    = 0.7        # weight on the KD (soft) term vs hard CE
        KD_LR       = 1e-3
        KD_WD       = 1e-4
        IMAGENET_MEAN = [0.485, 0.456, 0.406]
        IMAGENET_STD  = [0.229, 0.224, 0.225]

        # ── Student: ImageNet-pretrained MobileNetV3-Small, head -> N_CLASSES ──
        student = torchvision.models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        _in = student.classifier[-1].in_features
        student.classifier[-1] = nn.Linear(_in, N_CLASSES)
        student = student.to(device)
        _n_params = sum(p.numel() for p in student.parameters())
        print(f'Aqua student = MobileNetV3-Small  params={_n_params/1e6:.2f}M  classes={N_CLASSES}')

        # ── Paired dataset: ONE augmented PIL view -> (student tensor, teacher
        #    tensor, label). Both views share the same geometry so the teacher's
        #    soft target corresponds to exactly what the student sees; only the
        #    resolution differs (student @224, teacher @IMG). ──────────────────
        _geo_aug = T.Compose([
            T.RandomResizedCrop(256, scale=(0.6, 1.0), ratio=(0.85, 1.18)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(0.2, 0.2, 0.2, 0.02),
        ])
        def _to_norm(img, size):
            img = T.functional.center_crop(T.functional.resize(img, size), size)
            return T.functional.normalize(T.functional.to_tensor(img), IMAGENET_MEAN, IMAGENET_STD)

        class _KDPairDataset(torch.utils.data.Dataset):
            def __init__(self, samples, train=True):
                self.samples = samples; self.train = train
            def __len__(self):
                return len(self.samples)
            def __getitem__(self, i):
                path, y = self.samples[i]
                img = _PILImage.open(path).convert('RGB')
                img = _geo_aug(img) if self.train else \
                    T.functional.center_crop(T.functional.resize(img, 256), 256)
                return _to_norm(img, STUDENT_IMG), _to_norm(img, IMG), y

        _kd_src = ImageFolder(str(TRAIN_ROOT))  # samples + class order only
        assert _kd_src.classes == CLASS_NAMES, (_kd_src.classes, CLASS_NAMES)
        # Mirror §4's class-imbalance handling for the student. §4's DINOv2 teacher
        # is trained with a class-balanced/source-weighted sampler + logit-adjusted
        # loss; a plain-shuffle student would see the thin classes
        # (gray_mold_botrytis, the physiological pair) in only ~a few % of batches
        # and REGRESS their recall vs the teacher. The class-balanced sampler
        # indexes _kd_src.samples — the same list (same order) _KDPairDataset is
        # built from — so its indices line up. With balanced batches the hard term
        # stays plain CE: a single rebalance (logit-adjustment on top would
        # double-correct). The KD soft term transfers the teacher's (source-weighted,
        # houseplant-domain) distribution on top of that.
        kd_sampler = make_class_balanced_sampler(_kd_src, power=0.5)
        kd_loader = DataLoader(_KDPairDataset(_kd_src.samples, train=True),
                               batch_size=BATCH, sampler=kd_sampler, num_workers=2,
                               pin_memory=(device == 'cuda'), drop_last=True)
        # Remap VAL_INDOMAIN's ImageFolder label ids into the student's class order
        # (CLASS_NAMES). VAL_INDOMAIN may not hold every class as a subdir, which
        # would otherwise shift the ImageFolder ids and silently corrupt the macro-F1
        # early-stop signal (the §4/§6 eval loaders already guard this via class_to_idx).
        _kd_val_if = ImageFolder(str(VAL_INDOMAIN))
        _kd_val_remap = {i: CLASS_NAMES.index(c) for i, c in enumerate(_kd_val_if.classes)}
        _kd_val_samples = [(p, _kd_val_remap[y]) for p, y in _kd_val_if.samples]
        kd_val = DataLoader(_KDPairDataset(_kd_val_samples, train=False),
                            batch_size=BATCH, shuffle=False, num_workers=2)

        opt = torch.optim.AdamW(student.parameters(), lr=KD_LR, weight_decay=KD_WD)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=KD_EPOCHS)
        ce = nn.CrossEntropyLoss()

        @torch.no_grad()
        def _student_macro_f1(loader):
            # Macro-F1 (not top-1) so early stopping tracks the THIN classes the
            # student is most at risk of dropping — top-1 on an imbalanced val set
            # is dominated by the majority classes. Mirrors §4's disease-F1 gate.
            student.eval(); ys, ps = [], []
            for sx, _, y in loader:
                ps.append(student(sx.to(device)).argmax(1).cpu()); ys.append(y)
            y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()
            return float(f1_score(y_true, y_pred, average='macro', zero_division=0))

        best_student_f1, best_student_state = -1.0, None
        for ep in range(KD_EPOCHS):
            student.train(); run = 0.0
            for sx, tx, y in kd_loader:
                sx, tx, y = sx.to(device), tx.to(device), y.to(device)
                with torch.no_grad():
                    t_logits = model(tx)                      # DINOv2 teacher view
                s_logits = student(sx)
                soft = F.kl_div(F.log_softmax(s_logits / KD_T, dim=1),
                                F.softmax(t_logits / KD_T, dim=1),
                                reduction='batchmean') * (KD_T * KD_T)
                hard = ce(s_logits, y)
                loss = KD_ALPHA * soft + (1.0 - KD_ALPHA) * hard
                opt.zero_grad(); loss.backward(); opt.step()
                run += loss.item()
            sched.step()
            f1 = _student_macro_f1(kd_val)
            if f1 > best_student_f1:
                best_student_f1 = f1
                best_student_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
            print(f'  KD ep{ep+1:02d}/{KD_EPOCHS}  loss={run/len(kd_loader):.3f}  val_macroF1={f1:.3f}')
        if best_student_state is not None:
            student.load_state_dict(best_student_state)
        student.eval()
        print(f'Aqua student trained  best_val_macroF1={best_student_f1:.3f}')

        # ── Export → TFLite ────────────────────────────────────────────
        # Path: torch -> ONNX -> (onnx2tf) TF SavedModel(NHWC) -> TFLite. We ship
        # dynamic-range INT8 (INT8 weights, FLOAT32 in/out) — ~4x smaller than FP32
        # and the float interface keeps the on-device preprocessing simple (feed a
        # normalised float NHWC tensor). Full int8-in/out is available by setting a
        # representative_dataset + inference_input/output_type=int8 below.
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               'tensorflow-cpu>=2.18', 'onnx2tf>=1.26', 'onnx-graphsurgeon',
                               'sng4onnx', 'onnxsim', 'psutil', 'ai-edge-litert'])
        _onnx_path = str(AQUA_DIR / 'aqua_student.onnx')
        _dummy = torch.randn(1, 3, STUDENT_IMG, STUDENT_IMG, device=device)
        torch.onnx.export(student, _dummy, _onnx_path, opset_version=17,
                          input_names=['input'], output_names=['logits'],
                          dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}})

        import onnx2tf
        _saved = str(AQUA_DIR / 'aqua_saved_model')
        onnx2tf.convert(input_onnx_file_path=_onnx_path, output_folder_path=_saved,
                        non_verbose=True, copy_onnx_input_output_names_to_tflite=True)

        import tensorflow as tf
        conv = tf.lite.TFLiteConverter.from_saved_model(_saved)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range INT8 weights
        tflite_bytes = conv.convert()
        _tflite_path = AQUA_DIR / 'aqua_classifier.tflite'
        _tflite_path.write_bytes(tflite_bytes)

        # classes.json mirrors the server bundle so the app reads the same metadata
        # (class order, input size, normalisation, family tag).
        (AQUA_DIR / 'classes.json').write_text(json.dumps({
            'model_version': MODEL_VERSION,
            'model_family': 'aqua',
            'architecture': 'mobilenetv3_small',
            'class_names': CLASS_NAMES,
            'training': {'img_size': STUDENT_IMG, 'distilled_from': FAMILY_CONFIG['hub_id']},
            'inference': {'layout': 'NHWC', 'normalize_mean': IMAGENET_MEAN,
                          'normalize_std': IMAGENET_STD, 'tta': 'none'},
        }, indent=2), encoding='utf-8')
        print(f'Aqua TFLite written: {_tflite_path}  ({_tflite_path.stat().st_size/1e6:.2f} MB)  '
              f'-> drop {_tflite_path.name} + classes.json into server static/aqua/; '
              f'the app downloads + caches it from GET /api/aqua/model (no APK bundling).')
    except Exception as _aqua_err:
        import traceback
        print(f'Aqua distillation/TFLite export FAILED ({type(_aqua_err).__name__}: {_aqua_err}); the §6 gamma bundle is unaffected.')
        traceback.print_exc()


## §6 — Production export: ONNX → dynamic INT8 + temperature + conformal

### Export & quantisation order of operations

1. Export FP32 ONNX from the trained **frozen-DINOv2 + head** model
   (backbone+head in one graph). CPU export keeps any CUDA-only ops out
   of the graph; `quant_pre_process` (symbolic-shape inference) runs with
   a copy-through fallback if it can't resolve the ViT graph. An
   **ONNX↔torch parity assert** (max abs logit diff < 1e-2) fails loudly
   if the position-embedding interpolation / opset broke the export
   before it can poison calibration.
2. **Dynamic INT8** quantisation (`quantize_dynamic`, `QInt8` weights).
   ORT's official guidance is dynamic quant for transformer/ViT models:
   activation ranges are computed at run time (no calibration set), and
   the weight-heavy MatMul/Gemm ops — most of a ViT — are quantised for
   ~2–3× CPU speedup at ~0 accuracy loss. This sidesteps the
   HardSwish/SE static-PTQ traps that crippled MobileNetV3.
3. Drift gate on val_realworld disease macro-F1:
   * `drift ≤ 0.02` → ship INT8.
   * `0.02 < drift ≤ 0.05` → ship INT8 with WARN.
   * `drift > 0.05` → **honest FP32 fallback**: write `classifier_fp32.onnx`,
     delete `classifier_int8.onnx`, record `int8.shipped = False`.
   The int8-vs-fp32 ordering is additionally reported with a **paired
   bootstrap** CI (report Stage 0) — a 0.03–0.04 gap at n≈450 is within
   sampling noise, so `paired_bootstrap.significant` is the honest test,
   not the raw F1 difference.
4. Fit temperature on the SHIPPED artefact's logits (INT8 if it passed
   the drift gate, FP32 otherwise — never fit T on a collapsed graph).
5. Fit conformal RAPS on `conformal_cal` softmaxed through the fitted T.
   Use ~80/20 fit/audit when `n_cal ≥ 200`; otherwise fit on the full
   split and report `audit_n = 0`.
6. Calibrate the MaxLogit OOD floor on `val_realworld` (plant vs NAP);
   the gate is disabled (`max_logit_floor=null`) when separation is the
   wrong sign.
7. Final eval on every val slice via the shipped ORT session.
8. **Ship gate** runs BEFORE any declarative artefact is persisted. Every
   calibration step above (F1, T, conformal, MaxLogit) is computed
   through the same `inference.tta` mode, so enabling `USE_TTA_HFLIP`
   never decalibrates the shipped model.


In [ ]:
# §6 — DINOv2 export: ONNX FP32 → dynamic INT8 + temperature + conformal RAPS + ship gate
import datetime
import shutil as _sh
import onnx  # noqa: F401  (import validates the onnx runtime dep is present)
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
except ImportError as e:
    raise RuntimeError(
        'onnxruntime.quantization.shape_inference.quant_pre_process is required. '
        'Upgrade onnxruntime to >=1.17 and re-run §1 install.'
    ) from e

Path('models').mkdir(exist_ok=True)

# Which family this §6 ONNX bundle (models/) is labelled as. On an AQUA run the
# §4 DINOv2 model IS gamma (aqua's teacher backbone = dinov2_vits14 @224,
# identical to gamma), so we export it as the GAMMA bundle — i.e. ONE
# MODEL_FAMILY='aqua' run yields BOTH the gamma server bundle (models/, below)
# AND the aqua TFLite student (models/aqua/, from §5). A 'gamma' run is gamma
# only; a 'frontier' run is the ViT-B/14 backbone, exported as 'frontier'.
_EXPORT_FAMILY = 'gamma' if FAMILY_CONFIG.get('distill') else MODEL_FAMILY
if FAMILY_CONFIG.get('distill'):
    print(f"aqua run: §6 exports the DINOv2 teacher as the '{_EXPORT_FAMILY}' bundle "
          "(models/); the shipped aqua model is models/aqua/aqua_classifier.tflite (§5). "
          "-> gamma + aqua from one run.")

# ── Test-time augmentation (report Stage 3) ─────────────────────
# Horizontal-flip TTA (2×) is "nearly free accuracy" given the CPU latency
# headroom. It is OFF by default: when ON, EVERYTHING below — the ship-gate
# F1, temperature, conformal q_hat and the MaxLogit floor — is computed
# through the SAME flip-averaging, and ``inference.tta='hflip'`` is written
# to classes.json so the server (`classifier.py`) reproduces the identical
# averaging. We average LOGITS (not softmax probs): that keeps temperature
# scaling, MaxLogit and conformal all operating on the same quantity they
# were fit on, so enabling TTA never silently decalibrates the model.
USE_TTA_HFLIP = True
TTA_MODE = 'hflip' if USE_TTA_HFLIP else 'none'

# INT8-vs-FP32 ship decision (report P1: ship INT8 for Gamma). Gamma has
# CPU latency headroom, but INT8 is ~4x smaller (=22.5 vs 87 MB) and faster
# on CPU, and it is the natural base for the Aqua tier -- so we now DEFAULT
# TO INT8 and only fall back to FP32 when INT8 is actually worse. "Actually
# worse" is decided below by the hard drift gate (drift > FAIL_DRIFT) AND the
# paired bootstrap (INT8 significantly below FP32, CI excludes 0) -- never by
# a point-estimate inversion at this sample size. Set PREFER_FP32=True to
# force FP32 regardless (a tier that wants max accuracy and has the budget).
PREFER_FP32 = False


def _ort_session(path):
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=['CPUExecutionProvider'])


def _ort_logits(sess, x_np):
    in_name = sess.get_inputs()[0].name
    logits = sess.run(None, {in_name: x_np})[0]
    if USE_TTA_HFLIP:
        flipped = np.ascontiguousarray(x_np[..., ::-1])
        logits = (logits + sess.run(None, {in_name: flipped})[0]) * 0.5
    return logits


def _ort_forward(sess, x_np):
    """Like _ort_logits but also returns the CLS embedding (2nd output)
    TTA-averaged the same way; embedding is None on a legacy 1-output graph."""
    in_name = sess.get_inputs()[0].name
    has_emb = len(sess.get_outputs()) >= 2
    outs = sess.run(None, {in_name: x_np})
    logits, emb = outs[0], (outs[1] if has_emb else None)
    if USE_TTA_HFLIP:
        fouts = sess.run(None, {in_name: np.ascontiguousarray(x_np[..., ::-1])})
        logits = (logits + fouts[0]) * 0.5
        if has_emb:
            emb = (emb + fouts[1]) * 0.5
    return logits, emb


disease_idx = [i for i, n in enumerate(CLASS_NAMES) if n not in ('healthy', 'not_a_plant')]


def _disease_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, labels=disease_idx, average='macro', zero_division=0))



def _predict_onnx(path, dl):
    """Return (y_true, y_pred) integer arrays — needed for the paired
    bootstrap so the int8-vs-fp32 comparison is on identical indices."""
    sess = _ort_session(path)
    y_true, y_pred = [], []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        y_pred.extend(logits.argmax(1).tolist())
        y_true.extend(y.tolist())
    return np.asarray(y_true), np.asarray(y_pred)


def _paired_bootstrap_f1(y_true, y_pred_a, y_pred_b, *, n_boot=5000, ci=0.95, seed=SEED):
    """Paired bootstrap for (A − B) disease macro-F1 on a shared test set
    (report Stage 0). Resample one index vector per iteration, apply to
    BOTH models, summarise the difference distribution. ``significant`` is
    True only when the CI of the difference excludes zero — the honest test
    the historical ``f1_int8=0.73 > f1_fp32=0.69`` inversion never passed.
    """
    y_true = np.asarray(y_true); a = np.asarray(y_pred_a); b = np.asarray(y_pred_b)
    n = len(y_true)
    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot, dtype=np.float64)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        diffs[i] = _disease_f1(yt, a[idx]) - _disease_f1(yt, b[idx])
    lo = float(np.percentile(diffs, (1 - ci) / 2 * 100))
    hi = float(np.percentile(diffs, (1 + ci) / 2 * 100))
    return {
        'metric': 'disease_macro_f1', 'compared': 'int8_minus_fp32',
        'diff_point': float(_disease_f1(y_true, a) - _disease_f1(y_true, b)),
        'diff_ci_low': lo, 'diff_ci_high': hi, 'ci_level': ci,
        'p_int8_gt_fp32': float(np.mean(diffs > 0.0)),
        'significant': bool(lo > 0.0 or hi < 0.0),
        'n': int(n), 'n_boot': int(n_boot),
    }


def _per_class_recall(y_true, y_pred):
    """Recall per class index; nan when a class has no support (so an
    absent class is never silently scored 0/1)."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    out = np.full(len(CLASS_NAMES), np.nan, dtype=np.float64)
    for i in range(len(CLASS_NAMES)):
        sup = int(np.sum(yt == i))
        if sup:
            out[i] = float(np.sum((yt == i) & (yp == i)) / sup)
    return out


def _per_class_recall_ci(y_true, y_pred, *, n_boot=2000, ci=0.95, seed=SEED):
    """Percentile-bootstrap recall CI per class on the shipped model
    (report Stage 0: report per-class recall CIs, not just point
    estimates — a ~30-50 sample class has a recall CI of +/-0.10+, so the
    weak-class numbers the report targets must ship with their CI)."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    rng = np.random.default_rng(seed)
    boot = np.full((n_boot, len(CLASS_NAMES)), np.nan, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = _per_class_recall(yt[idx], yp[idx])
    point = _per_class_recall(yt, yp)
    out = {}
    for i, c in enumerate(CLASS_NAMES):
        col = boot[:, i]; valid = col[~np.isnan(col)]
        if valid.size == 0:
            out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                      'ci_low': None, 'ci_high': None,
                      'support': int(np.sum(yt == i)), 'n_boot_valid': 0}
            continue
        lo = float(np.percentile(valid, (1 - ci) / 2 * 100))
        hi = float(np.percentile(valid, (1 + ci) / 2 * 100))
        out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                  'ci_low': lo, 'ci_high': hi, 'ci_half_width': (hi - lo) / 2,
                  'support': int(np.sum(yt == i)), 'n_boot_valid': int(valid.size)}
    return {'metric': 'per_class_recall', 'ci_level': ci, 'n': int(n),
            'n_boot': int(n_boot), 'classes': out}


def _per_class_precision(y_true, y_pred):
    """Precision per class index; nan when a class is never PREDICTED (so an
    unpredicted class is never silently scored 0/1). The physiological classes
    are precision-bound (they over-trigger), so precision ships beside recall."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    out = np.full(len(CLASS_NAMES), np.nan, dtype=np.float64)
    for i in range(len(CLASS_NAMES)):
        pred = int(np.sum(yp == i))
        if pred:
            out[i] = float(np.sum((yt == i) & (yp == i)) / pred)
    return out


def _per_class_precision_ci(y_true, y_pred, *, n_boot=2000, ci=0.95, seed=SEED):
    """Percentile-bootstrap precision CI per class on the shipped model — the
    precision companion to _per_class_recall_ci. 'support' is true-class support
    (comparable to the recall CI); 'n_predicted' is precision's denominator. A
    replicate that never predicts a class yields nan and is dropped from its
    percentile computation."""
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    rng = np.random.default_rng(seed)
    boot = np.full((n_boot, len(CLASS_NAMES)), np.nan, dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[b] = _per_class_precision(yt[idx], yp[idx])
    point = _per_class_precision(yt, yp)
    out = {}
    for i, c in enumerate(CLASS_NAMES):
        col = boot[:, i]; valid = col[~np.isnan(col)]
        base = {'support': int(np.sum(yt == i)), 'n_predicted': int(np.sum(yp == i))}
        if valid.size == 0:
            out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                      'ci_low': None, 'ci_high': None, 'n_boot_valid': 0, **base}
            continue
        lo = float(np.percentile(valid, (1 - ci) / 2 * 100))
        hi = float(np.percentile(valid, (1 + ci) / 2 * 100))
        out[c] = {'point': None if np.isnan(point[i]) else float(point[i]),
                  'ci_low': lo, 'ci_high': hi, 'ci_half_width': (hi - lo) / 2,
                  'n_boot_valid': int(valid.size), **base}
    return {'metric': 'per_class_precision', 'ci_level': ci, 'n': int(n),
            'n_boot': int(n_boot), 'classes': out}


# ── Export the frozen-DINOv2 + head model to ONNX ───────────────
# Move to CPU for a deployment-faithful FP32 export (no CUDA-only ops in
# the graph). DINOv2 interpolates its position embedding to the 224-px
# input; opset 17 covers the LayerNorm / SDPA / grid-sample ops the ViT
# emits. The parity assert below fails loudly if the export diverged.
# Merge LoRA adapters into the base Linears so the export is a single,
# self-contained graph (validated: ONNX<->torch parity ~1e-6 after merge).
if USE_LORA:
    model.backbone = model.backbone.merge_and_unload()
    model.lora_active = False
    print('LoRA adapters merged into the backbone for export.')
# Export with a SECOND output -- the DINOv2 CLS embedding -- so the server
# computes an embedding-space k-NN OOD score (ood.py) without a second model.
# output[0] stays the logits, so every existing run(...)[0] call site (eval,
# INT8 drift, temperature, conformal) is unaffected; a legacy bundle without
# this output simply has no embedding to score and OOD falls back to MaxLogit.
class _ExportWithEmbedding(nn.Module):
    def __init__(self, clf):
        super().__init__()
        self.clf = clf
    def forward(self, x):
        feats = self.clf.backbone(x)          # (N, embed_dim) CLS embedding
        return self.clf.head(feats), feats

export_model = model.to('cpu').eval()
export_wrapped = _ExportWithEmbedding(export_model).eval()
raw_path  = 'models/classifier_raw.onnx'
fp32_onnx = 'models/classifier.onnx'
int8_onnx = 'models/classifier_int8.onnx'
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(
    export_wrapped, dummy, raw_path,
    opset_version=17, input_names=['input'],
    output_names=['logits', 'embedding'],
    dynamic_axes={'input': {0: 'N'}, 'logits': {0: 'N'}, 'embedding': {0: 'N'}},
    dynamo=False,
)
try:
    quant_pre_process(raw_path, fp32_onnx, skip_symbolic_shape=False)
except Exception as exc:  # noqa: BLE001 — fall back to the un-preprocessed graph
    print(f'WARN: quant_pre_process failed ({type(exc).__name__}: {exc}); '
          f'using the raw export as the FP32 graph.')
    _sh.copyfile(raw_path, fp32_onnx)

# Export parity check — catches a broken ViT export (bad interpolation /
# opset) before it silently poisons every downstream calibration step.
with torch.no_grad():
    torch_logits = export_wrapped(dummy)[0].numpy()
_p_sess = _ort_session(fp32_onnx)
onnx_logits = _p_sess.run(None, {_p_sess.get_inputs()[0].name: dummy.numpy().astype(np.float32)})[0]
_parity = float(np.abs(torch_logits - onnx_logits).max())
print(f'ONNX↔torch parity: max_abs_logit_diff={_parity:.3e}')
if _parity > 1e-2:
    raise RuntimeError(
        f'ONNX export diverged from torch (max_abs_logit_diff={_parity:.3e}). '
        f'Investigate the DINOv2 position-embedding interpolation / opset before shipping.'
    )

# ── Dynamic INT8 quantisation ───────────────────────────────────
# ONNX Runtime's official guidance: dynamic quantization for transformer /
# ViT models, static for CNNs. Dynamic quant computes activation ranges at
# run time (no calibration set) and quantises the weight-heavy MatMul/Gemm
# ops — which is most of a ViT — for ~2–3× CPU speedup at ~0 accuracy loss,
# avoiding the HardSwish/SE static-PTQ traps that crippled MobileNetV3.
quantize_dynamic(fp32_onnx, int8_onnx, weight_type=QuantType.QInt8)
print(f'dynamic INT8 quantised: {Path(int8_onnx).stat().st_size / 1e6:.1f} MB '
      f'(fp32 {Path(fp32_onnx).stat().st_size / 1e6:.1f} MB)')
Path(raw_path).unlink(missing_ok=True)

# ── Drift gate (val_realworld) + paired bootstrap ───────────────
yt_rw, yp_fp32 = _predict_onnx(fp32_onnx, val_realworld_dl)
_,     yp_int8 = _predict_onnx(int8_onnx, val_realworld_dl)
f1_fp32 = _disease_f1(yt_rw, yp_fp32)
f1_int8 = _disease_f1(yt_rw, yp_int8)
drift = f1_fp32 - f1_int8
paired_int8_vs_fp32 = _paired_bootstrap_f1(yt_rw, yp_int8, yp_fp32)
print(f'val_realworld disease F1  fp32={f1_fp32:.4f}  int8={f1_int8:.4f}  drift={drift:+.4f}')
print(f'paired bootstrap (int8−fp32): diff={paired_int8_vs_fp32["diff_point"]:+.4f}  '
      f'95% CI=[{paired_int8_vs_fp32["diff_ci_low"]:+.4f}, {paired_int8_vs_fp32["diff_ci_high"]:+.4f}]  '
      f'significant={paired_int8_vs_fp32["significant"]}')

WARN_DRIFT, FAIL_DRIFT = 0.02, 0.05
canonical_fp32_fallback = Path('models/classifier_fp32.onnx')
# Ship FP32 when explicitly preferred, OR when INT8 fails the hard drift
# gate, OR when the paired bootstrap says INT8 is SIGNIFICANTLY worse than
# FP32 (CI excludes 0 and the point difference is negative). The bootstrap
# term is the honest backstop: a -0.01 drift the bootstrap cannot separate
# from noise is NOT a reason to forfeit INT8's 4x size + latency win.
_int8_significantly_worse = bool(
    paired_int8_vs_fp32['significant'] and paired_int8_vs_fp32['diff_point'] < 0.0
)
_ship_fp32 = PREFER_FP32 or (drift > FAIL_DRIFT) or _int8_significantly_worse
if _ship_fp32:
    why = (f'INT8 drift {drift:+.4f} > FAIL_DRIFT={FAIL_DRIFT}' if drift > FAIL_DRIFT
           else 'INT8 significantly worse than FP32 (paired bootstrap CI excludes 0)'
           if _int8_significantly_worse
           else f'PREFER_FP32=True (INT8 drift {drift:+.4f}; CPU latency headroom)')
    print(f'Shipping FP32 as classifier.onnx. Reason: {why}. '
          f'classifier_int8.onnx will NOT be emitted.')
    Path(int8_onnx).unlink(missing_ok=True)
    _sh.copyfile(fp32_onnx, canonical_fp32_fallback)
    int8_shipped, shipped_artifact = False, f'classifier.onnx (FP32; {why})'
    eval_sess_path = fp32_onnx
else:
    if drift > WARN_DRIFT:
        print(f'WARN: INT8 drift {drift:+.4f} > WARN_DRIFT={WARN_DRIFT}; shipping anyway.')
    canonical_fp32_fallback.unlink(missing_ok=True)
    int8_shipped, shipped_artifact = True, 'classifier_int8.onnx'
    eval_sess_path = int8_onnx

eval_sess = _ort_session(eval_sess_path)
QUANT_FIT_SOURCE = 'int8' if int8_shipped else 'fp32_fallback'

# Per-class recall bootstrap CI on the SHIPPED artefact's val_realworld
# predictions (report Stage 0). This is where the weak classes the report
# targets (nutrient_deficiency, leaf_scorch_*, gray_mold_*) live; their
# point recall is only trustworthy alongside a CI.
shipped_rw_pred = yp_int8 if int8_shipped else yp_fp32
recall_ci_realworld = _per_class_recall_ci(yt_rw, shipped_rw_pred)
print('per-class recall (val_realworld, shipped):')
for _c, _d in recall_ci_realworld['classes'].items():
    if _d.get('ci_low') is None:
        print(f'  {_c:28s} n={_d["support"]:>3}  (no bootstrap support)')
    else:
        print(f'  {_c:28s} n={_d["support"]:>3}  recall={_d["point"]:.3f} '
              f'95% CI=[{_d["ci_low"]:.3f}, {_d["ci_high"]:.3f}]')

# Per-class PRECISION bootstrap CI on the same shipped val_realworld
# predictions. The weak physiological classes are precision-bound, so their
# precision (and its CI) is the metric that actually tracks the over-trigger.
precision_ci_realworld = _per_class_precision_ci(yt_rw, shipped_rw_pred)
print('per-class precision (val_realworld, shipped):')
for _c, _d in precision_ci_realworld['classes'].items():
    if _d.get('ci_low') is None:
        print(f'  {_c:28s} pred={_d["n_predicted"]:>3}  (never predicted / no support)')
    else:
        print(f'  {_c:28s} pred={_d["n_predicted"]:>3}  precision={_d["point"]:.3f} '
              f'95% CI=[{_d["ci_low"]:.3f}, {_d["ci_high"]:.3f}]')

# Conformal split — used only for RAPS quantile fitting downstream.
cal_dataset, cal_dl = make_eval_loader(
    CONFORMAL_CAL, IMG, class_to_idx=TRAIN_CLASS_TO_IDX,
)


def _collect_logits(sess, dl):
    all_logits, all_y = [], []
    for x, y in dl:
        all_logits.append(_ort_logits(sess, x.numpy().astype(np.float32)))
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)


# ── Temperature scaling on the shipped artefact's logits ────────
logits_temp_np, y_temp_np = _collect_logits(eval_sess, temp_cal_dl)
logits_temp = torch.from_numpy(logits_temp_np)
y_temp = torch.from_numpy(y_temp_np).long()

log_T = torch.tensor(0.0, requires_grad=True)
opt_T = torch.optim.LBFGS([log_T], lr=0.05, max_iter=50)
def closure():
    opt_T.zero_grad()
    T_now = log_T.exp().clamp(0.05, 10.0)
    loss = F.cross_entropy(logits_temp / T_now, y_temp)
    loss.backward(); return loss
opt_T.step(closure)
T_val = float(log_T.detach().exp().clamp(0.05, 10.0))
temperature_payload = {
    'temperature': T_val,
    'source': QUANT_FIT_SOURCE,
    'tta': TTA_MODE,
    '_note': f'Fitted with LBFGS on temperature_cal NLL using {QUANT_FIT_SOURCE} ONNX '
             f'logits (tta={TTA_MODE}).',
    'fit_sources': ['plantdoc', 'inat', 'coco', 'inat_animalia', 'openimages_indoor',
                    'coco_hard_partial_plant', 'inat_hard_greentexture'],
}
print(f'fitted T={T_val:.3f}  source={QUANT_FIT_SOURCE}  '
      f'temp_cal MaxLogit mean={logits_temp.max(1).values.numpy().mean():.2f}')

# ── Conformal RAPS on conformal_cal probabilities (after temperature) ──
# RAPS non-conformity score — single source of truth in conformal.py, the
# same module the server runtime loads, so the notebook calibration and the
# served prediction sets can never silently drift apart. Fall back to an
# inline copy when the notebook runs somewhere conformal.py isn't importable
# (e.g. a bare Colab kernel). Signature matches conformal.raps_score
# (probs_sorted_desc, lambda_reg, k_reg) so every call site is unchanged.
# Put the repo dir on sys.path so the clone's conformal.py wins over the
# inline fallback when the notebook runs from a checkout.
import os as _os, sys as _sys
for _cp in ('.', _os.getcwd(), '/content/plantdiseases/server', '/content/server'):
    if _os.path.isfile(_os.path.join(_cp, 'conformal.py')) and _cp not in _sys.path:
        _sys.path.insert(0, _cp)
try:
    from conformal import raps_score as _raps_score  # type: ignore[import-not-found]
    print('conformal: using raps_score from conformal.py (single source of truth)')
except Exception as _e:  # noqa: BLE001 — keep the notebook self-contained off-repo
    print(f'conformal: conformal.py not importable ({_e!r}); using inline RAPS copy')
    def _raps_score(p_sorted, lam, k_reg):
        cs = np.cumsum(p_sorted)
        return cs + lam * np.maximum(0.0, np.arange(1, len(p_sorted)+1, dtype=np.float64) - k_reg)

def _eval_raps_set(probs, labels, q_hat, lambda_reg, k_reg):
    n = probs.shape[0]
    if n == 0:
        return float('nan'), float('nan')
    covered, set_sizes = 0, []
    for i in range(n):
        order = np.argsort(-probs[i])
        s = _raps_score(probs[i, order], lambda_reg, k_reg)
        above = np.where(s >= q_hat)[0]
        size = max(int(above[0]) + 1 if above.size else len(s), 1)
        set_sizes.append(size)
        covered += int(labels[i] in order[:size])
    return covered / n, float(np.mean(set_sizes))

def calibrate_q_hat(cal_probs, cal_labels, *, alpha=0.1, lambda_reg=0.01, k_reg=1,
                    audit_probs=None, audit_labels=None):
    n = cal_probs.shape[0]
    if n == 0:
        raise RuntimeError('Empty conformal calibration split.')
    nonconf = np.empty(n)
    for i in range(n):
        order = np.argsort(-cal_probs[i])
        s = _raps_score(cal_probs[i, order], lambda_reg, k_reg)
        nonconf[i] = s[int(np.where(order == cal_labels[i])[0][0])]
    q = float(np.quantile(nonconf, min(1.0, np.ceil((n+1) * (1-alpha)) / n), method='higher'))
    in_cov, in_size = _eval_raps_set(cal_probs, cal_labels, q, lambda_reg, k_reg)
    out_cov = out_size = float('nan'); audit_n = 0
    if audit_probs is not None and audit_probs.shape[0] > 0:
        audit_n = int(audit_probs.shape[0])
        out_cov, out_size = _eval_raps_set(audit_probs, audit_labels, q, lambda_reg, k_reg)
    primary_cov  = out_cov  if audit_n > 0 else in_cov
    primary_size = out_size if audit_n > 0 else in_size
    return {'method': 'raps', 'alpha': alpha, 'q_hat': q,
            'lambda_reg': lambda_reg, 'k_reg': k_reg,
            'calibration_n': int(n),
            'empirical_coverage': primary_cov,
            'empirical_coverage_kind': 'out_of_sample' if audit_n > 0 else 'in_sample',
            'mean_set_size': primary_size,
            'audit_n': audit_n,
            'in_sample_coverage': in_cov,
            'in_sample_mean_set_size': in_size}

logits_cal_np, y_cal_np = _collect_logits(eval_sess, cal_dl)
probs_cal = torch.softmax(torch.from_numpy(logits_cal_np) / T_val, dim=1).numpy()
n_cal = int(y_cal_np.shape[0])
order_cal = np.random.RandomState(SEED + 7).permutation(n_cal)
# ── Conformal hyper-parameter search (report P1: fix over-coverage) ──
# The old call hardcoded alpha=0.10, lambda_reg=0.01, k_reg=1 and shipped
# whatever coverage fell out -- on this very-confident classifier that was
# 99.4% (vs the 90% target) with a 2.56/9 mean set size, i.e. sets too big
# to be the honest "either of these two" hint they exist to be. RAPS only
# *guarantees* coverage >= 1 - alpha; on a sharp model the realised q_hat
# over-shoots, so we grid-search (alpha, lambda_reg, k_reg) and keep the
# config with the SMALLEST out-of-sample (audit) mean set size whose audit
# coverage still clears the product target (CONF_TARGET_COVERAGE) -- exactly
# the smallest set that still covers the truth ~90% of the time. The
# advertised coverage_target = 1 - chosen alpha then honestly reflects what
# was calibrated instead of a nominal 90% the sets silently overshoot.
CONF_TARGET_COVERAGE = 0.90
# Grid widened after a run landed at coverage 0.973 / set 1.93 (vs the 0.989 /
# 2.55 baseline): the classifier is confident enough that the previous grid's
# most-aggressive point still over-covered, so add higher alpha + a stronger
# RAPS size penalty (lambda) to let the search shrink sets further while the
# selection rule still requires audit coverage >= the 0.90 product target.
CONF_ALPHA_GRID  = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]
CONF_LAMBDA_GRID = [0.0, 0.01, 0.05, 0.10, 0.20, 0.30, 0.50]
CONF_KREG_GRID   = [1, 2]

def _search_conformal(fit_p, fit_y, audit_p, audit_y):
    trials = []
    for a in CONF_ALPHA_GRID:
        for lam in CONF_LAMBDA_GRID:
            for kr in CONF_KREG_GRID:
                trials.append(calibrate_q_hat(
                    fit_p, fit_y, alpha=a, lambda_reg=lam, k_reg=kr,
                    audit_probs=audit_p, audit_labels=audit_y))
    cov  = lambda c: c['empirical_coverage']
    size = lambda c: c['mean_set_size']
    # Smallest set whose (audit, else in-sample) coverage still clears the
    # product target; if the grid over-shrank everything, keep the most
    # covering trial rather than ship an under-covering set.
    covering = [c for c in trials if cov(c) is not None and cov(c) >= CONF_TARGET_COVERAGE]
    best = (min(covering, key=lambda c: (size(c), abs(cov(c) - CONF_TARGET_COVERAGE)))
            if covering else max(trials, key=cov))
    best['target_coverage'] = CONF_TARGET_COVERAGE
    best['grid'] = {'alpha': CONF_ALPHA_GRID, 'lambda_reg': CONF_LAMBDA_GRID,
                    'k_reg': CONF_KREG_GRID, 'n_trials': len(trials)}
    # Carry the pre-tune baseline (the old fixed config) so the over-coverage
    # fix is auditable before/after in the model card.
    base = next((c for c in trials if c['alpha'] == 0.10
                 and c['lambda_reg'] == 0.01 and c['k_reg'] == 1), None)
    if base is not None:
        best['baseline_alpha010_lambda001_k1'] = {
            'empirical_coverage': base['empirical_coverage'],
            'mean_set_size': base['mean_set_size']}
    return best

if n_cal >= 200:
    fit_n = int(n_cal * 0.80)
    fit_idx, audit_idx = order_cal[:fit_n], order_cal[fit_n:]
    cfg = _search_conformal(probs_cal[fit_idx], y_cal_np[fit_idx],
                            probs_cal[audit_idx], y_cal_np[audit_idx])
else:
    print(f'conformal: n_cal={n_cal} < 200 — grid-searching on the full split '
          f'with in-sample coverage (audit_n=0; coverage is optimistic)')
    cfg = _search_conformal(probs_cal, y_cal_np, None, None)
print(f"conformal: selected alpha={cfg['alpha']} lambda_reg={cfg['lambda_reg']} "
      f"k_reg={cfg['k_reg']} -> coverage={cfg['empirical_coverage']:.3f} "
      f"set_size={cfg['mean_set_size']:.2f} (kind={cfg['empirical_coverage_kind']})")
cfg['coverage_scope'] = (f'all {len(CLASS_NAMES)} classes (plantdoc, inat, coco, '
                          'inat_animalia, openimages_indoor); '
                          'not_a_plant is additionally guarded by the '
                          'Stage 1 detector + embedding k-NN OOD gate.')
cfg['cal_sources'] = ['plantdoc', 'inat', 'coco', 'inat_animalia',
                       'openimages_indoor', 'coco_hard_partial_plant',
                       'inat_hard_greentexture']
cfg['fit_source'] = QUANT_FIT_SOURCE
cfg['tta'] = TTA_MODE
print('conformal:', cfg)

# ── MaxLogit OOD floor on the shipped session ───────────────────
TARGET_PLANT_FPR = 0.05
NAP_CLASS_IDX = (CLASS_NAMES.index('not_a_plant')
                 if 'not_a_plant' in CLASS_NAMES else None)

def _maxlogit_distribution(sess, dl, *, only_class=None, exclude_class=None):
    out = []
    for x, y in dl:
        logits = _ort_logits(sess, x.numpy().astype(np.float32))
        for i in range(logits.shape[0]):
            yi = int(y[i])
            if only_class is not None and yi != only_class:
                continue
            if exclude_class is not None and yi == exclude_class:
                continue
            out.append(float(logits[i].max()))
    return np.array(out, dtype=np.float64)

plant_ml = _maxlogit_distribution(eval_sess, val_realworld_dl, exclude_class=NAP_CLASS_IDX)
nap_ml = (_maxlogit_distribution(eval_sess, val_realworld_dl, only_class=NAP_CLASS_IDX)
          if NAP_CLASS_IDX is not None else np.array([]))

if len(plant_ml) >= 30:
    ML_FLOOR = float(np.percentile(plant_ml, TARGET_PLANT_FPR * 100))
else:
    print(f'WARN: only {len(plant_ml)} plant MaxLogit samples — falling back to 4.0 default')
    ML_FLOOR = 4.0

ML_SEPARATION = (float(np.percentile(plant_ml, 50) - np.percentile(nap_ml, 50))
                  if len(plant_ml) and len(nap_ml) else None)
if ML_SEPARATION is not None and ML_SEPARATION < 0:
    print(f'WARN: NAP MaxLogit median above plant median (sep={ML_SEPARATION:+.3f}) '
          f'— disabling the MaxLogit gate (max_logit_floor=null).')
    ML_FLOOR_PERSISTED = None
else:
    ML_FLOOR_PERSISTED = ML_FLOOR

ML_CAL_META = {
    'target_plant_fpr': TARGET_PLANT_FPR,
    'plant_n': int(len(plant_ml)),
    'plant_p05': float(np.percentile(plant_ml, 5)) if len(plant_ml) else None,
    'plant_p50': float(np.percentile(plant_ml, 50)) if len(plant_ml) else None,
    'plant_p95': float(np.percentile(plant_ml, 95)) if len(plant_ml) else None,
    'nap_n': int(len(nap_ml)),
    'nap_p05': float(np.percentile(nap_ml, 5)) if len(nap_ml) else None,
    'nap_p50': float(np.percentile(nap_ml, 50)) if len(nap_ml) else None,
    'nap_p95': float(np.percentile(nap_ml, 95)) if len(nap_ml) else None,
    'separation_p50': ML_SEPARATION,
    'fit_source': QUANT_FIT_SOURCE,
}
print(f'MaxLogit OOD floor calibrated: '
      f'{"off" if ML_FLOOR_PERSISTED is None else f"{ML_FLOOR_PERSISTED:.3f}"}  '
      f'(plant n={len(plant_ml)} p50={ML_CAL_META["plant_p50"]}, '
      f'nap n={len(nap_ml)} p50={ML_CAL_META["nap_p50"]})')

# ── Embedding-space OOD (k-NN) calibration (report P1) ──────────────
# MaxLogit went dead once the NAP head grew over-confident. Calibrate a
# live, logit-independent OOD signal: a deep k-NN cosine distance in the
# DINOv2 CLS embedding space (Sun et al. 2022). Build an in-distribution
# memory bank of PLANT embeddings from conformal_cal, then set the OOD
# threshold on a DISJOINT split (temperature_cal) at TARGET_PLANT_FPR, so
# the bank and the threshold are never fit on the same samples. Both use the
# SHIPPED (possibly INT8) session so the bank/threshold match what serves.
OOD_KNN_K = 5
OOD_BANK_CAP = 4000

def _collect_embeddings(sess, dl):
    embs, ys = [], []
    for x, y in dl:
        _lg, e = _ort_forward(sess, x.numpy().astype(np.float32))
        if e is None:
            return None, None
        embs.append(e); ys.append(y.numpy())
    return np.concatenate(embs), np.concatenate(ys)

def _l2n(a):
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)

def _knn_scores(bank_norm, q, k):
    k = max(1, min(k, bank_norm.shape[0]))
    sims = _l2n(q) @ bank_norm.T
    topk = np.partition(sims, -k, axis=1)[:, -k:] if k < sims.shape[1] else sims
    return 1.0 - topk.mean(axis=1)

EMBEDDING_OOD, ood_bank_ship = None, None
_emb_bank, _y_bank = _collect_embeddings(eval_sess, cal_dl)        # conformal_cal
_emb_thr,  _y_thr  = _collect_embeddings(eval_sess, temp_cal_dl)   # temperature_cal
if (_emb_bank is not None and _emb_thr is not None
        and NAP_CLASS_IDX is not None):
    _bank_plant = _emb_bank[_y_bank != NAP_CLASS_IDX]
    _thr_plant  = _emb_thr[_y_thr != NAP_CLASS_IDX]
    _thr_nap    = _emb_thr[_y_thr == NAP_CLASS_IDX]
    if _bank_plant.shape[0] >= 10 and _thr_plant.shape[0] >= 10:
        if _bank_plant.shape[0] > OOD_BANK_CAP:
            _sel = np.random.RandomState(SEED + 11).choice(
                _bank_plant.shape[0], OOD_BANK_CAP, replace=False)
            _bank_plant = _bank_plant[_sel]
        _bn = _l2n(_bank_plant)
        _ps = _knn_scores(_bn, _thr_plant, OOD_KNN_K)
        # Threshold at the (1 - FPR) quantile of in-dist plant scores: only
        # TARGET_PLANT_FPR of real plants exceed it (false-OOD rate).
        _thr = float(np.quantile(_ps, 1.0 - TARGET_PLANT_FPR))
        _ns = _knn_scores(_bn, _thr_nap, OOD_KNN_K) if len(_thr_nap) else np.array([])
        _nap_recall = float(np.mean(_ns > _thr)) if len(_ns) else None
        ood_bank_ship = _bank_plant
        EMBEDDING_OOD = {
            'method': 'knn', 'metric': 'cosine', 'k': OOD_KNN_K,
            'threshold': _thr, 'embed_dim': int(_bank_plant.shape[1]),
            'bank_file': 'ood_embedding.npz',
            '_calibration': {
                'target_plant_fpr': TARGET_PLANT_FPR,
                'bank_n': int(_bank_plant.shape[0]),
                'thr_plant_n': int(len(_ps)), 'thr_nap_n': int(len(_ns)),
                'nap_recall_at_threshold': _nap_recall,
                'plant_score_p50': float(np.percentile(_ps, 50)),
                'plant_score_p95': float(np.percentile(_ps, 95)),
                'nap_score_p50': float(np.percentile(_ns, 50)) if len(_ns) else None,
                'fit_source': QUANT_FIT_SOURCE,
            },
        }
        print(f'Embedding OOD calibrated: threshold={_thr:.4f} '
              f'bank_n={_bank_plant.shape[0]} '
              f'nap_recall@thr={_nap_recall} (plant FPR={TARGET_PLANT_FPR})')
if EMBEDDING_OOD is None:
    print('Embedding OOD: graph has no embedding output or too few '
          'plant/NAP samples — not shipping a bank (OOD falls back to MaxLogit).')

# ── Final metrics across all val slices via the shipped ORT session ──
class _OnnxModelAdapter:
    """Thin wrapper so evaluate() can call model(x) on an ORT session."""
    def __init__(self, sess): self.sess = sess
    def eval(self): return self
    def __call__(self, x):
        out = _ort_logits(self.sess, x.detach().cpu().numpy().astype(np.float32))
        return torch.from_numpy(out).to(x.device)

eval_adapter = _OnnxModelAdapter(eval_sess)
ONNX_EVAL_DEVICE = 'cpu'
final_metrics = {
    'indomain':  evaluate(eval_adapter, val_indomain_dl,  CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_indomain'),
    'realworld': evaluate(eval_adapter, val_realworld_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_realworld'),
}
if val_studio_dl is not None:
    final_metrics['studio'] = evaluate(eval_adapter, val_studio_dl, CLASS_NAMES, ONNX_EVAL_DEVICE, 'final val_studio')

# ── Compose payloads (NOT yet written — the ship gate runs first) ──
classes_payload = {
    'model_version': MODEL_VERSION,
    'model_family': _EXPORT_FAMILY,
    'architecture': ARCHITECTURE,
    'class_names': CLASS_NAMES,
    'detector_classes': ['plant_leaf'],
    'training': {
        'img_size': IMG,
        'architecture': ARCHITECTURE,
        # Eval/calibration transform: Resize(int(img_size*eval_resize_ratio))
        # -> CenterCrop(img_size). Serving reads this so train/serve crop
        # parity holds (conformal q_hat + temperature were fit on it).
        'eval_resize_ratio': EVAL_RESIZE_RATIO,
        'batch_size': BATCH,
        'loss': 'logit_adjustment_ce',
        'logit_adjustment_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'prior_source_weights': dict(PRIOR_SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'lr_schedule': 'cosine + warmup',
        'precision': 'bf16-mixed' if device == 'cuda' else 'fp32',
        'primary_metric': 'val_realworld_disease_macro_f1',
        'primary_val_set': 'data_v2/classification/val/realworld',
        'isolated_val_slice': 'data_v2/classification/val/plantvillage_studio',
    },
    'backbone': {
        'name': _HUB_ID,
        'source': 'facebookresearch/dinov2 (torch.hub)',
        'frozen': True,
        'embed_dim': EMBED_DIM,
        'head': 'LayerNorm -> Linear -> GELU -> Dropout -> Linear (on CLS embedding)',
        'head_hidden': HEAD_HIDDEN,
        'head_dropout': HEAD_DROPOUT,
        '_note': 'Frozen DINOv2 ViT-S/14 feature extractor + trained MLP head. '
                 'No knowledge distillation (see §5).',
    },
    'inference': {
        'tta': TTA_MODE,
        '_tta_note': 'hflip averages original+mirror LOGITS; temperature / '
                     'conformal / MaxLogit above are all fit with the same mode.',
    },
    'ood': {
        'score': 'embedding_knn' if EMBEDDING_OOD is not None else 'max_logit',
        'max_logit_floor': ML_FLOOR_PERSISTED,
        'embedding': EMBEDDING_OOD,
        '_score_note': ('Primary: embedding-space k-NN distance on the DINOv2 '
                        'CLS vector (Sun 2022), the live replacement for the '
                        'now-dead MaxLogit floor. The server prefers the '
                        'embedding gate when a bank is shipped, else MaxLogit.'),
        '_calibration': ML_CAL_META,
    },
}

model_card = {
    'version': MODEL_VERSION,
    'model_family': _EXPORT_FAMILY,
    'architecture': ARCHITECTURE,
    'trained_at': datetime.datetime.now(datetime.timezone.utc).replace(microsecond=0).isoformat(),
    'classes': CLASS_NAMES,
    'img_size': IMG,
    'temperature': T_val,
    'conformal': cfg,
    'train_counts':           {c: count_images(TRAIN_ROOT  / c) for c in CLASS_NAMES},
    'val_realworld_counts':   {c: count_images(VAL_PRIMARY / c) for c in CLASS_NAMES},
    'val_indomain_counts':    {c: count_images(VAL_INDOMAIN/ c) for c in CLASS_NAMES},
    'temperature_cal_counts': {c: count_images(TEMP_CAL    / c) for c in CLASS_NAMES},
    'conformal_cal_counts':   {c: count_images(CONFORMAL_CAL/ c) for c in CLASS_NAMES},
    'val_studio_counts':      {c: count_images(VAL_STUDIO  / c) for c in CLASS_NAMES} if val_studio_dl is not None else None,
    'hyperparams': {
        'backbone': _HUB_ID,
        'backbone_frozen': True,
        'embed_dim': EMBED_DIM,
        'head_hidden': HEAD_HIDDEN,
        'head_dropout': HEAD_DROPOUT,
        'head_epochs': HEAD_EPOCHS,
        'head_lr': HEAD_LR,
        'la_tau': LA_TAU,
        'label_smoothing': LABEL_SMOOTHING,
        'select_weakclass_precision_weight': SELECT_WEAKCLASS_PRECISION_WEIGHT,
        'use_weighted_sampler': bool(USE_WEIGHTED_SAMPLER),
        'use_source_weighted_sampler': bool(USE_SOURCE_WEIGHTED_SAMPLER),
        'source_weights': dict(SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'prior_source_weights': dict(PRIOR_SOURCE_WEIGHTS) if USE_SOURCE_WEIGHTED_SAMPLER else None,
        'batch': BATCH, 'img_size': IMG,
        'lora': ({'enabled': True, 'rank': LORA_RANK, 'alpha': LORA_ALPHA,
                  'last_k_blocks': LORA_LAST_K_BLOCKS, 'lr': LORA_LR}
                 if USE_LORA else {'enabled': False}),
    },
    'int8': {
        '_note': ('f1_int8 / f1_fp32 are measured by raw argmax(logits) on '
                  'val_realworld — no temperature scaling, no weak-healthy '
                  'reroute, no very-low-confidence demotion. The runtime '
                  'pipeline applies all three on top; production disease '
                  'macro-F1 is therefore at least as good as f1_int8, never '
                  'less. The ship gate uses f1_int8 as a conservative floor. '
                  'The paired_bootstrap entry is the honest int8-vs-fp32 test: '
                  'treat the ordering as real only when significant=true.'),
        'drift': drift, 'f1_fp32': f1_fp32, 'f1_int8': f1_int8,
        'warn_drift': WARN_DRIFT, 'fail_drift': FAIL_DRIFT,
        'quant_mode': 'dynamic',
        'weight_type': 'QInt8',
        'paired_bootstrap': paired_int8_vs_fp32,
        'tta': TTA_MODE,
        'shipped': int8_shipped,
        'shipped_artifact': shipped_artifact,
    },
    'per_class_recall_ci_realworld': recall_ci_realworld,
    'per_class_precision_ci_realworld': precision_ci_realworld,
    'head': {'best_indomain_disease_f1': best_f1, 'best_epoch': best_ep},
    'reporting_backend': ('onnxruntime_int8_cpu' if int8_shipped
                          else 'onnxruntime_fp32_cpu'),
    'reproducibility': {
        'seed': SEED,
        'torch': torch.__version__,
        'onnxruntime': ort.__version__,
        'cudnn_deterministic': bool(getattr(torch.backends.cudnn, 'deterministic', False)),
        'cudnn_benchmark':     bool(getattr(torch.backends.cudnn, 'benchmark', True)),
    },
}

# ── Ship gate — runs BEFORE we persist anything declarative ─────
MIN_DISEASE_F1, MAX_HEALTHY_BIAS, MIN_HEALTHY_RECALL = 0.70, 1.50, 0.70
rw = final_metrics['realworld']
disease_classes = [c for c in CLASS_NAMES if c not in ('healthy', 'not_a_plant')]
zero_recall = [c for c in disease_classes if rw['per_class_recall'].get(c, 0.0) <= 0.0]
healthy_recall = float(rw['per_class_recall'].get('healthy', 0.0))
failures = []
if rw['disease_macro_f1'] < MIN_DISEASE_F1:
    failures.append(f'disease_macro_f1={rw["disease_macro_f1"]:.3f} < {MIN_DISEASE_F1}')
if rw['healthy_bias'] >= MAX_HEALTHY_BIAS:
    failures.append(f'healthy_bias={rw["healthy_bias"]:.2f} >= {MAX_HEALTHY_BIAS}')
if healthy_recall < MIN_HEALTHY_RECALL:
    failures.append(f'recall[healthy]={healthy_recall:.3f} < {MIN_HEALTHY_RECALL} (false-positive disease risk)')
if zero_recall:
    failures.append(f'zero recall classes: {zero_recall}')
if failures:
    raise RuntimeError('SHIP GATE FAILED: ' + '; '.join(failures))
print('SHIP GATE PASSED')

# Non-blocking weak-class precision audit (v2.5.1). The physiological classes
# over-trigger; a sharp argmax-precision drop is worth a loud line. NOT a hard
# gate — precision on a ~25-45 sample slice has a wide CI, so failing the run on
# it would be noise-driven; the runtime (temperature + low-confidence demotion +
# conformal sets) absorbs much of the raw-argmax over-trigger.
WEAK_PRECISION_WARN = 0.50
for _c in ('leaf_scorch_underwatering', 'nutrient_deficiency'):
    _pd = precision_ci_realworld['classes'].get(_c) or {}
    _pt = _pd.get('point')
    if _pt is not None and _pt < WEAK_PRECISION_WARN:
        print(f'WARN: weak-class precision[{_c}]={_pt:.3f} < {WEAK_PRECISION_WARN} '
              f'(over-triggering; n_pred={_pd.get("n_predicted")}, '
              f'CI=[{_pd.get("ci_low")}, {_pd.get("ci_high")}]) — non-blocking; '
              f'runtime demotion/conformal mitigate.')

# ── Persist the declarative artefacts now that the gate is green ──
Path('models/classes.json').write_text(json.dumps(classes_payload, indent=2))
Path('models/temperature.json').write_text(json.dumps(temperature_payload, indent=2))
Path('models/conformal.json').write_text(json.dumps(cfg, indent=2))
Path('models/training_metrics.json').write_text(json.dumps(final_metrics, indent=2))
Path('models/model_card.json').write_text(json.dumps(model_card, indent=2))
if EMBEDDING_OOD is not None and ood_bank_ship is not None:
    np.savez_compressed('models/ood_embedding.npz',
                        bank=ood_bank_ship.astype(np.float16))
    print(f'ood_embedding.npz: bank shape={ood_bank_ship.shape}')
print(f'classes.json: {len(CLASS_NAMES)} classes  architecture={ARCHITECTURE}')
print('Persisted: classes.json, temperature.json, conformal.json, '
      'training_metrics.json, model_card.json')

# Detector artefact contract.
missing = [s for s in ('detector.pt', 'detector.onnx')
           if not Path(f'models/{s}').exists()]
if missing:
    print(f'WARN: {missing} not in models/ — drop the shipped '
          f'server/models/detector.{{pt,onnx}} pair into ./models/ '
          f'before assembling the deploy bundle, or re-run §3 with '
          f'a YOLO data.yaml under data_v2/.')

log_run({'phase': 'export_done', 'architecture': ARCHITECTURE,
         'f1_fp32': f1_fp32, 'f1_int8': f1_int8, 'T': T_val,
         'q_hat': cfg['q_hat'], 'empirical_coverage': cfg['empirical_coverage'],
         'paired_significant': paired_int8_vs_fp32['significant'],
         'int8_shipped': int8_shipped, 'shipped_artifact': shipped_artifact})

# ── Production-faithful eval: detector -> crop -> classify (diagnostic) ──
# The metrics above score the WHOLE val image, but production crops the leaf
# via the Stage-1 detector first (pipeline.py). This optional pass reports
# the detector-crop macro-F1 + per-class recall so the headline numbers can
# be read against what users actually get (the real-photo test showed a
# single diseased leaf gets diluted in a whole-frame). FULLY GUARDED: the
# model + artefacts are already shipped above, so any failure here only
# skips the diagnostic and never affects the released model.
RUN_DETECTOR_CROP_EVAL = True
if RUN_DETECTOR_CROP_EVAL:
    try:
        import gc as _gc
        from ultralytics import YOLO as _YOLO
        from PIL import Image as _PImg
        # Reclaim VRAM before the two-stage eval. After export the torch DINOv2
        # model is no longer needed here — classification runs through the
        # shipped ONNX session (eval_sess) — so park the training graph on CPU
        # and free the cache. Without this, YOLO + fragmentation OOMs on a 40GB
        # A100 (the eval is a diagnostic, so it was silently skipped before).
        try:
            model.to('cpu')
            dino_backbone.to('cpu')
        except Exception:
            pass
        _gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        _dp = Path('models/detector.pt')
        if not _dp.exists():
            raise FileNotFoundError('models/detector.pt not present')
        _det = _YOLO(str(_dp))
        _tx = T.Compose([T.Resize(int(IMG * EVAL_RESIZE_RATIO)), T.CenterCrop(IMG), T.ToTensor(), NORM])
        _tt = getattr(val_realworld_ds, 'target_transform', None)
        _samples = list(val_realworld_ds.samples)
        # Subsample purely to bound the diagnostic's runtime (it is NOT the ship
        # gate); 0 = use all. Batched either way.
        DETECTOR_CROP_EVAL_CAP = 0
        if DETECTOR_CROP_EVAL_CAP and len(_samples) > DETECTOR_CROP_EVAL_CAP:
            _sel = random.Random(SEED).sample(range(len(_samples)), DETECTOR_CROP_EVAL_CAP)
            _samples = [_samples[i] for i in _sel]
        _paths = [p for p, _ in _samples]
        _yt = np.array([int(_tt(l) if _tt is not None else l) for _, l in _samples])
        # Stage 1 — BATCHED detection (stream + batch amortises the per-image
        # NMS/Python overhead that made this loop the slow tail of §6).
        _crops, _wholes = [], []
        for _res, _p in zip(_det.predict(_paths, verbose=False, conf=0.25,
                                         stream=True, batch=16, device='cpu'), _paths):
            _pil = _PImg.open(_p).convert('RGB')
            _bx = _res.boxes
            if _bx is not None and len(_bx) > 0:
                _x1, _y1, _x2, _y2 = (int(v) for v in _bx[int(_bx.conf.argmax())].xyxy[0].tolist())
                _crop = _pil.crop((_x1, _y1, _x2, _y2)) if (_x2 > _x1 and _y2 > _y1) else _pil
            else:
                _crop = _pil   # no leaf box -> classify the whole frame
            _crops.append(_tx(_crop).numpy())
            _wholes.append(_tx(_pil).numpy())
        # Stage 2 — BATCHED classification through the shipped session (one ORT
        # call per 128, vs one per image before; identical math, ~hundreds x fewer calls).
        def _batched_logits(arrs, bs=128):
            return np.concatenate([_ort_logits(eval_sess, np.stack(arrs[i:i + bs]).astype(np.float32))
                                   for i in range(0, len(arrs), bs)])
        _lc = _batched_logits(_crops)
        _lw = _batched_logits(_wholes)
        def _softmax_rows(z):
            e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
        _yp = _lc.argmax(1)
        _yp_fold = (_softmax_rows(_lc) + _softmax_rows(_lw)).argmax(1)
        _f1c = _disease_f1(_yt, _yp)
        _f1f = _disease_f1(_yt, _yp_fold)
        _whole = final_metrics['realworld']['disease_macro_f1']
        print(f'[detector-crop val_realworld] crop_only={_f1c:.3f} '
              f'crop+whole_fold={_f1f:.3f} whole_image={_whole:.3f}  '
              f'(fold recovers {_f1f-_f1c:+.3f} vs crop-only; '
              f'crop-only delta vs whole={_f1c-_whole:+.3f})')
        for _ci, _c in enumerate(CLASS_NAMES):
            _sup = int((_yt == _ci).sum())
            if _sup:
                _rec = float(((_yp == _ci) & (_yt == _ci)).sum() / _sup)
                print(f'    recall[{_c:25s}] = {_rec:.3f}  (n={_sup})')
        log_run({'phase': 'detector_crop_eval', 'disease_macro_f1': _f1c,
                 'fold_disease_macro_f1': _f1f, 'whole_image_f1': _whole})
    except Exception as _e:  # noqa: BLE001 — diagnostic only, model already shipped
        print(f'detector-crop eval skipped: {type(_e).__name__}: {_e}')


In [ ]:
# §6b — Bundle the models/ dir into a single archive for manual deploy.
# The server does NOT auto-download models; copy the extracted artefacts
# into its models/ dir before starting. Uncompressed .tar of every
# artefact §6 wrote
# (classifier*.onnx, detector.{pt,onnx}, classes.json, temperature.json,
# conformal.json, model_card.json, ...), members at the archive root so the
# server's flatten step drops them straight into its models/ dir.
import tarfile as _tarfile
BUNDLE_PATH = Path('models') / 'upgrmodels.tar'
# Deploy artefacts only. The runtime is ONNX-only, so the torch checkpoints
# (classifier.pth ~86 MB, detector.pt ~6 MB) are NOT bundled -- the server
# cannot load them and they only bloat the archive (~92 MB). ``.npz`` is
# REQUIRED: it carries the embedding-OOD bank that classes.json ->
# ood.embedding references; omitting it silently disables the embedding OOD
# gate at serve time (the bug that shipped in v2.3.1).
_BUNDLE_SUFFIXES = {'.onnx', '.json', '.jsonl', '.npz'}
_added = []
with _tarfile.open(BUNDLE_PATH, 'w') as _tar:
    for _p in sorted(Path('models').iterdir()):
        if _p.is_file() and _p.name != BUNDLE_PATH.name and _p.suffix in _BUNDLE_SUFFIXES:
            _tar.add(_p, arcname=_p.name)
            _added.append(_p.name)
print(f'wrote {BUNDLE_PATH}  ({BUNDLE_PATH.stat().st_size / 1e6:.1f} MB)')
print(f'  members: {_added}')
# Guard: a declared embedding-OOD bank must actually be in the archive.
try:
    _cj = json.loads((Path('models') / 'classes.json').read_text())
    _bank = (((_cj.get('ood') or {}).get('embedding') or {}) or {}).get('bank_file')
    if _bank and _bank not in _added:
        print(f'  WARNING: classes.json references ood bank {_bank!r} but it is NOT '
              f'in the bundle -> embedding OOD will be DISABLED on deploy. '
              f'Ensure {_bank} exists in models/ and its suffix is in _BUNDLE_SUFFIXES.')
except Exception:
    pass
print('\nShip checklist (manual deploy — the server does NOT auto-download):')
print(f'  1. Copy the artefacts in models/ (or extract {BUNDLE_PATH.name}) into')
print( '     the server\'s models/ directory before starting it.')
print( '  2. Restart the server — boot is fail-fast if an artefact is missing.')


## §7 — Evaluation contract

| Slice | Role | Pass criterion |
|---|---|---|
| `val_indomain` | training-loop metric, drives early stopping | tracked; best head checkpoint restored |
| `val_realworld` | **primary — ship/no-ship gate** | disease macro-F1 ≥ **0.70** (must not regress below the live model); no disease class with recall = 0; healthy-bias < 1.5; healthy recall ≥ 0.70 |
| `val_studio` | PlantVillage transfer / background-bias monitor | tracked, never blocking |
| `temperature_cal` | temperature scaling | held out |
| `conformal_cal` | RAPS calibration | held out; 80/20 fit/audit when n ≥ 200, else fit-only |

Persisted artefacts (all written to `models/` in the notebook CWD, **only after the ship gate passes**):

* `classifier.pth` — full `DINOv2Classifier` state_dict (frozen backbone + trained head). Dev fallback only; production serves the ONNX graph.
* `classifier.onnx` — FP32 backbone+head export (also serves as fallback when INT8 fails the drift gate).
* `classifier_int8.onnx` — dynamic INT8. **Only present when INT8 drift ≤ 0.05.**
* `classifier_fp32.onnx` — copy of the FP32 graph. **Only present when INT8 drift > 0.05.** The explicit "INT8 export failed" signal.
* `classes.json` — the contract the server reads at startup (`model_version`, `architecture`, class names, `training.img_size`, `inference.tta`, OOD config).
* `temperature.json` — fitted T plus `source` ∈ {`int8`, `fp32_fallback`} and `tta`.
* `conformal.json` — RAPS calibration; `fit_source` matches `temperature.json.source`.
* `detector.{pt,onnx}` — Stage 1 weights (re-trained from §3 if a YOLO `data.yaml` is supplied; the shipped pair otherwise).
* `dinov2_probe.json` — diagnostic linear-probe curve (cached-feature implementation).
* `training_metrics.json` — structured per-class recall + confusion matrix on every val slice.
* `model_card.json` — provenance: train/val counts per class, hyperparams, INT8 drift, **paired-bootstrap int8-vs-fp32 CI**, **per-class recall bootstrap CIs on val_realworld** (`per_class_recall_ci_realworld` — the weak-class numbers the report tracks, shipped with their CI), best head epoch, reproducibility flags, `int8.shipped` flag and `shipped_artifact` label.
* `run_log.jsonl` — one JSON-line per epoch / phase, useful for post-mortem.

For an out-of-notebook A/B (e.g. a Stage 2 weak-class run vs the current model on a shared, cleaned test set), `tools/eval_bootstrap.py` runs the same paired bootstrap on **both** disease macro-F1 and per-class recall — the honest gate the report asks for before trusting any improvement.
